In [1]:
"""
================================================================================
MODULE 1 — Dataset Preparation, Feature Engineering, Model Training and
            Performance Evaluation
Dataset: PAMAP2 Physical Activity Monitoring
Part of: "Toward Deployment-Aware Human Activity Recognition: A Multi-Dataset
          Evaluation of Classical Machine Learning Models Using Robustness,
          Statistical Validation, and TinyML Metrics."

SELF-CONTAINED SINGLE-FILE IMPLEMENTATION
------------------------------------------
Paste this entire script into one Jupyter cell (or run as `python module1_pamap2.py`).
It will, in order:
  1. Automatically download the official PAMAP2 dataset from UCI if not
     already present locally.
  2. Clean the raw data and handle missing values.
  3. Perform sliding-window segmentation (5.12 s windows, 50% overlap).
  4. Extract statistical (time + frequency domain + cross-axis correlation)
     features — the fixed feature set reused across the whole framework.
  5. Standardize features and tune/train 7 classical models via GridSearchCV:
     Logistic Regression, Decision Tree, Random Forest, KNN, Gaussian Naive
     Bayes, Linear SVM, XGBoost.
  6. Evaluate every model: accuracy, precision, recall, F1, confusion matrix,
     classification report, Stratified K-Fold Cross-Validation.
  7. Save trained models, comparison tables, 300 dpi figures, CSV results,
     and a markdown report — all inside ./pamap2_training/.

Explicitly OUT of scope for this module (implemented separately downstream):
  TinyML profiling, robustness/perturbation testing, cross-dataset
  statistical significance testing (Friedman/Nemenyi, etc.).

If automatic download fails (e.g. no outbound network access), manually
download the dataset from:
  https://archive.ics.uci.edu/dataset/231/pamap2+physical+activity+monitoring
and place PAMAP2_Dataset.zip at ./pamap2_training/data/raw/PAMAP2_Dataset.zip,
then re-run this script — it will detect and extract the local file.
================================================================================
"""

import os
import sys
import json
import time
import zipfile
import urllib.request
import urllib.error

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC

try:
    from xgboost import XGBClassifier
    _XGBOOST_AVAILABLE = True
except ImportError:
    _XGBOOST_AVAILABLE = False

sns.set_theme(style="whitegrid", context="talk")


# ==============================================================================
# SECTION 1: CONFIGURATION
# (the only section that would need to change for a future dataset)
# ==============================================================================

PROJECT_ROOT = os.path.join(os.getcwd(), "pamap2_training")

RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
INTERIM_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "interim")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

for d in [RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR,
          TABLES_DIR, FIGURES_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

PAMAP2_ZIP_NAME = "PAMAP2_Dataset.zip"
PAMAP2_ZIP_PATH = os.path.join(RAW_DATA_DIR, PAMAP2_ZIP_NAME)
PAMAP2_EXTRACT_DIR = os.path.join(RAW_DATA_DIR, "PAMAP2_Dataset")
PAMAP2_PROTOCOL_DIR = os.path.join(PAMAP2_EXTRACT_DIR, "Protocol")

PAMAP2_DOWNLOAD_URLS = [
    "https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00231/PAMAP2_Dataset.zip",
]

RANDOM_STATE = 42

# Official PAMAP2 layout: 54 columns per row.
IMU_SENSOR_BLOCK = [
    "temperature",
    "acc16_x", "acc16_y", "acc16_z",
    "acc6_x", "acc6_y", "acc6_z",
    "gyro_x", "gyro_y", "gyro_z",
    "mag_x", "mag_y", "mag_z",
    "orient_1", "orient_2", "orient_3", "orient_4",  # invalid, dropped
]

COLUMN_NAMES = (
    ["timestamp", "activity_id", "heart_rate"]
    + [f"hand_{c}" for c in IMU_SENSOR_BLOCK]
    + [f"chest_{c}" for c in IMU_SENSOR_BLOCK]
    + [f"ankle_{c}" for c in IMU_SENSOR_BLOCK]
)
assert len(COLUMN_NAMES) == 54, "PAMAP2 schema must have 54 columns"

SENSOR_LOCATIONS = ["hand", "chest", "ankle"]
SENSOR_MODALITIES = {
    "acc16": ["acc16_x", "acc16_y", "acc16_z"],
    "gyro": ["gyro_x", "gyro_y", "gyro_z"],
    "mag": ["mag_x", "mag_y", "mag_z"],
}

SIGNAL_CHANNELS = ["heart_rate"]
for loc in SENSOR_LOCATIONS:
    for modality_axes in SENSOR_MODALITIES.values():
        for axis in modality_axes:
            SIGNAL_CHANNELS.append(f"{loc}_{axis}")

# 12-activity protocol (standard subset used across the PAMAP2 HAR literature)
ACTIVITY_MAP = {
    1: "lying", 2: "sitting", 3: "standing", 4: "walking", 5: "running",
    6: "cycling", 7: "nordic_walking", 12: "ascending_stairs",
    13: "descending_stairs", 16: "vacuum_cleaning", 17: "ironing", 24: "rope_jumping",
}
VALID_ACTIVITY_IDS = sorted(ACTIVITY_MAP.keys())
SUBJECT_IDS = list(range(101, 110))  # subject101 .. subject109

SAMPLING_RATE_HZ = 100
WINDOW_SIZE_SAMPLES = 512          # 5.12 s @ 100 Hz
WINDOW_OVERLAP = 0.5               # 50%
WINDOW_STEP_SAMPLES = int(WINDOW_SIZE_SAMPLES * (1 - WINDOW_OVERLAP))

MAX_INTERP_GAP = 100  # samples (~1 s)

N_SPLITS_KFOLD = 5
CV_INNER_FOLDS = 3

MODEL_REGISTRY = {
    "LogisticRegression": {"param_grid": {
        "clf__C": [0.01, 0.1, 1, 10], "clf__solver": ["lbfgs"], "clf__max_iter": [2000]}},
    "DecisionTree": {"param_grid": {
        "clf__max_depth": [5, 10, 20, None], "clf__min_samples_split": [2, 5, 10]}},
    "RandomForest": {"param_grid": {
        "clf__n_estimators": [100, 200], "clf__max_depth": [10, 20, None],
        "clf__min_samples_split": [2, 5]}},
    "KNN": {"param_grid": {
        "clf__n_neighbors": [3, 5, 7, 9], "clf__weights": ["uniform", "distance"]}},
    "GaussianNB": {"param_grid": {"clf__var_smoothing": [1e-9, 1e-8, 1e-7]}},
    "LinearSVM": {"param_grid": {"clf__C": [0.01, 0.1, 1, 10]}},
    "XGBoost": {"param_grid": {
        "clf__n_estimators": [100, 200], "clf__max_depth": [3, 6],
        "clf__learning_rate": [0.05, 0.1]}},
}
MODEL_DISPLAY_ORDER = list(MODEL_REGISTRY.keys())
FIGURE_DPI = 300
NON_FEATURE_COLS = ["window_id", "subject_id", "activity_id", "activity_label"]


# ==============================================================================
# SECTION 2: AUTOMATIC DATA ACQUISITION
# ==============================================================================

class DatasetDownloadError(RuntimeError):
    pass


def _download_with_progress_urllib(url: str, dest_path: str, timeout: int = 30) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        total = int(response.headers.get("Content-Length", 0))
        downloaded = 0
        chunk_size = 1024 * 1024
        with open(dest_path, "wb") as f:
            while True:
                chunk = response.read(chunk_size)
                if not chunk:
                    break
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f"\r  Downloading PAMAP2 (urllib): {pct:5.1f}% "
                          f"({downloaded / 1e6:.1f} MB / {total / 1e6:.1f} MB)", end="", flush=True)
        print()


def _download_with_progress_requests(url: str, dest_path: str, timeout: int = 120) -> None:
    import requests  # raises ImportError if unavailable, handled by caller
    with requests.get(url, stream=True, timeout=timeout) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("Content-Length", 0))
        downloaded = 0
        with open(dest_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f"\r  Downloading PAMAP2 (requests): {pct:5.1f}% "
                          f"({downloaded / 1e6:.1f} MB / {total / 1e6:.1f} MB)", end="", flush=True)
                else:
                    print(f"\r  Downloading PAMAP2 (requests): {downloaded / 1e6:.1f} MB",
                          end="", flush=True)
        print()


def is_dataset_available() -> bool:
    if not os.path.isdir(PAMAP2_PROTOCOL_DIR):
        return False
    expected = {f"subject{sid}.dat" for sid in SUBJECT_IDS}
    present = set(os.listdir(PAMAP2_PROTOCOL_DIR))
    return expected.issubset(present)


def download_pamap2(force: bool = False) -> str:
    os.makedirs(RAW_DATA_DIR, exist_ok=True)

    if not force and is_dataset_available():
        print(f"[download] PAMAP2 already present at {PAMAP2_PROTOCOL_DIR} — skipping download.")
        return PAMAP2_PROTOCOL_DIR

    # Check for an already-downloaded zip BEFORE attempting any download.
    # A manually-placed file, or one left over from a previous run, must be
    # validated and extracted directly rather than silently re-downloaded
    # and overwritten.
    if not force and os.path.exists(PAMAP2_ZIP_PATH):
        if zipfile.is_zipfile(PAMAP2_ZIP_PATH):
            print(f"[download] Found existing zip at {PAMAP2_ZIP_PATH} "
                  f"({os.path.getsize(PAMAP2_ZIP_PATH) / 1e6:.1f} MB) — "
                  "skipping download, extracting directly.")
        else:
            print(f"[download] Existing zip at {PAMAP2_ZIP_PATH} is invalid "
                  "— removing and re-downloading.")
            os.remove(PAMAP2_ZIP_PATH)

    MIN_EXPECTED_ZIP_BYTES = 600_000_000  # official archive is ~688 MB

    if not os.path.exists(PAMAP2_ZIP_PATH):
        last_error = None
        downloaded_ok = False

        # Check once whether `requests` is available; prefer it, since it is
        # the more reliable method on networks where urllib's connection
        # handling gets silently corrupted by a proxy/firewall/AV filter.
        try:
            import requests  # noqa: F401
            requests_available = True
        except ImportError:
            requests_available = False

        download_methods = (
            [("requests", _download_with_progress_requests), ("urllib", _download_with_progress_urllib)]
            if requests_available else
            [("urllib", _download_with_progress_urllib)]
        )
        if not requests_available:
            print("[download] NOTE: 'requests' package not found — using urllib only. "
                  "If downloads keep failing/corrupting, run `pip install requests` "
                  "and re-run this cell, as urllib is known to be unreliable on some networks.")

        for url in PAMAP2_DOWNLOAD_URLS:
            for method_name, download_fn in download_methods:
                try:
                    print(f"[download] Attempting download ({method_name}) from: {url}")
                    download_fn(url, PAMAP2_ZIP_PATH)
                    size = os.path.getsize(PAMAP2_ZIP_PATH) if os.path.exists(PAMAP2_ZIP_PATH) else 0
                    if size >= MIN_EXPECTED_ZIP_BYTES and zipfile.is_zipfile(PAMAP2_ZIP_PATH):
                        with zipfile.ZipFile(PAMAP2_ZIP_PATH) as zf_check:
                            if zf_check.testzip() is None:
                                print(f"[download] SUCCESS via {method_name}: "
                                      f"{size / 1e6:.1f} MB, verified intact.")
                                downloaded_ok = True
                                break
                    print(f"[download] {method_name} download incomplete/corrupted "
                          f"(size={size / 1e6:.1f} MB) — trying next method.")
                    if os.path.exists(PAMAP2_ZIP_PATH):
                        os.remove(PAMAP2_ZIP_PATH)
                except Exception as e:
                    print(f"[download] {method_name} failed: {e}")
                    last_error = e
                    if os.path.exists(PAMAP2_ZIP_PATH):
                        os.remove(PAMAP2_ZIP_PATH)
            if downloaded_ok:
                break

        if not downloaded_ok:
            raise DatasetDownloadError(
                "Automatic download failed or repeatedly produced a corrupted/incomplete "
                "file for all known mirrors and methods. This is most commonly caused by "
                "a network proxy, firewall, or antivirus web-filter interfering with large "
                "binary downloads.\n"
                "To proceed manually:\n"
                "  1. Download from: "
                "https://archive.ics.uci.edu/dataset/231/pamap2+physical+activity+monitoring\n"
                f"  2. Place the zip file at: {PAMAP2_ZIP_PATH}\n"
                "  3. Re-run this script (it will validate and extract the local zip).\n"
                f"Last underlying error: {last_error}"
            )

    print("[download] Verifying archive integrity before extraction...")
    with zipfile.ZipFile(PAMAP2_ZIP_PATH, "r") as zf_check:
        bad_member = zf_check.testzip()
        if bad_member is not None:
            os.remove(PAMAP2_ZIP_PATH)
            raise DatasetDownloadError(
                f"Downloaded zip failed integrity check (corrupted member: {bad_member}). "
                "The corrupted file has been removed. Please re-run this script to retry, "
                "or download the archive manually (see instructions above)."
            )

    print("[download] Extracting archive...")
    with zipfile.ZipFile(PAMAP2_ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DATA_DIR)

    if not is_dataset_available():
        for root, dirs, files in os.walk(RAW_DATA_DIR):
            if "Protocol" in dirs:
                nested_root = root
                if nested_root != PAMAP2_EXTRACT_DIR and not os.path.isdir(PAMAP2_EXTRACT_DIR):
                    os.rename(nested_root, PAMAP2_EXTRACT_DIR)
                break

    if not is_dataset_available():
        raise DatasetDownloadError(
            "Archive was downloaded and extracted, but the expected "
            f"Protocol/subjectXXX.dat files were not found under {PAMAP2_EXTRACT_DIR}. "
            "Please verify the archive contents."
        )

    print(f"[download] PAMAP2 ready at {PAMAP2_PROTOCOL_DIR}")
    return PAMAP2_PROTOCOL_DIR


# ==============================================================================
# SECTION 3: PREPROCESSING (cleaning + missing-value handling)
# ==============================================================================

def _load_single_subject(subject_id: int, protocol_dir: str) -> pd.DataFrame:
    path = os.path.join(protocol_dir, f"subject{subject_id}.dat")
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMN_NAMES)
    df["subject_id"] = subject_id
    return df


def load_raw_pamap2(protocol_dir: str) -> pd.DataFrame:
    frames = []
    for sid in SUBJECT_IDS:
        path = os.path.join(protocol_dir, f"subject{sid}.dat")
        if not os.path.exists(path):
            print(f"[preprocessing] WARNING: missing file for subject{sid}, skipping.")
            continue
        frames.append(_load_single_subject(sid, protocol_dir))
    if not frames:
        raise FileNotFoundError(f"No subject .dat files found in {protocol_dir}.")
    raw = pd.concat(frames, axis=0, ignore_index=True)
    print(f"[preprocessing] Loaded raw data: {raw.shape[0]} rows, "
          f"{raw['subject_id'].nunique()} subjects.")
    return raw


def drop_invalid_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_drop = [c for c in df.columns if c.endswith(
        ("orient_1", "orient_2", "orient_3", "orient_4"))]
    return df.drop(columns=cols_to_drop)


def filter_activities(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df[df["activity_id"].isin(VALID_ACTIVITY_IDS)].copy()
    df["activity_label"] = df["activity_id"].map(ACTIVITY_MAP)
    print(f"[preprocessing] Filtered to {len(VALID_ACTIVITY_IDS)} protocol "
          f"activities: {before} -> {len(df)} rows.")
    return df


def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    signal_cols = [c for c in SIGNAL_CHANNELS if c in df.columns]
    n_before = df[signal_cols].isna().sum().sum()

    interpolated_groups = []
    for subject_id, g in df.groupby("subject_id", sort=False):
        g = g.copy()
        g[signal_cols] = g[signal_cols].interpolate(
            method="linear", limit=MAX_INTERP_GAP, limit_direction="both"
        )
        interpolated_groups.append(g)
    df = pd.concat(interpolated_groups, axis=0, ignore_index=True)

    n_after_interp = df[signal_cols].isna().sum().sum()
    rows_before_dropna = len(df)
    df = df.dropna(subset=signal_cols).reset_index(drop=True)
    rows_after_dropna = len(df)

    print(f"[preprocessing] Missing values: {int(n_before)} -> {int(n_after_interp)} after interpolation.")
    print(f"[preprocessing] Dropped {rows_before_dropna - rows_after_dropna} rows "
          f"with unresolved NaNs ({rows_after_dropna} rows remain).")
    return df


def run_preprocessing(protocol_dir: str) -> pd.DataFrame:
    df = load_raw_pamap2(protocol_dir)
    df = drop_invalid_columns(df)
    df = filter_activities(df)
    df = handle_missing_values(df)
    out_path = os.path.join(INTERIM_DATA_DIR, "pamap2_clean.pkl")
    df.to_pickle(out_path)
    print(f"[preprocessing] Saved cleaned dataset to {out_path}")
    return df


# ==============================================================================
# SECTION 4: SLIDING-WINDOW SEGMENTATION
# ==============================================================================

def _contiguous_activity_blocks(df: pd.DataFrame):
    df = df.sort_values(["subject_id", "timestamp"]).reset_index(drop=True)
    change_point = (
        (df["subject_id"] != df["subject_id"].shift())
        | (df["activity_id"] != df["activity_id"].shift())
    ).cumsum()
    for _, block in df.groupby(change_point):
        yield block["subject_id"].iloc[0], block["activity_id"].iloc[0], block


def segment_windows(df: pd.DataFrame):
    signal_cols = [c for c in SIGNAL_CHANNELS if c in df.columns]
    window_size = WINDOW_SIZE_SAMPLES
    step = WINDOW_STEP_SAMPLES

    window_records = []
    window_id = 0
    n_blocks_used = 0

    for subject_id, activity_id, block in _contiguous_activity_blocks(df):
        n = len(block)
        if n < window_size:
            continue
        n_blocks_used += 1
        values = block[signal_cols].to_numpy()
        for start in range(0, n - window_size + 1, step):
            end = start + window_size
            window_records.append({
                "window_id": window_id,
                "subject_id": subject_id,
                "activity_id": activity_id,
                "activity_label": ACTIVITY_MAP[activity_id],
                "data": values[start:end, :],
            })
            window_id += 1

    print(f"[segmentation] Built {window_id} windows from {n_blocks_used} "
          f"contiguous activity blocks (window={window_size} samples, step={step} samples, "
          f"{WINDOW_OVERLAP*100:.0f}% overlap).")

    meta = pd.DataFrame([
        {k: r[k] for k in ("window_id", "subject_id", "activity_id", "activity_label")}
        for r in window_records
    ])
    windows_array = (np.stack([r["data"] for r in window_records], axis=0)
                      if window_records else np.empty((0, window_size, len(signal_cols))))
    return meta, windows_array, signal_cols


def run_segmentation(df: pd.DataFrame):
    meta, windows_array, signal_cols = segment_windows(df)
    meta_path = os.path.join(INTERIM_DATA_DIR, "windows_meta.pkl")
    array_path = os.path.join(INTERIM_DATA_DIR, "windows_array.npy")
    meta.to_pickle(meta_path)
    np.save(array_path, windows_array)
    print(f"[segmentation] Saved window metadata to {meta_path}")
    print(f"[segmentation] Saved window array {windows_array.shape} to {array_path}")
    return meta, windows_array, signal_cols


# ==============================================================================
# SECTION 5: FEATURE ENGINEERING
# (fixed feature set reused across the whole multi-dataset framework)
# ==============================================================================

def _time_domain_features(x: np.ndarray) -> dict:
    mean = np.mean(x)
    std = np.std(x)
    minimum = np.min(x)
    maximum = np.max(x)
    median = np.median(x)
    q75, q25 = np.percentile(x, [75, 25])
    rms = np.sqrt(np.mean(np.square(x)))
    zero_crossings = np.sum(np.diff(np.sign(x - mean)) != 0)
    mean_abs_change = np.mean(np.abs(np.diff(x)))
    return {
        "mean": mean, "std": std, "variance": np.var(x),
        "min": minimum, "max": maximum, "range": maximum - minimum,
        "median": median, "iqr": q75 - q25, "rms": rms,
        "skewness": skew(x, bias=False) if std > 1e-12 else 0.0,
        "kurtosis": kurtosis(x, bias=False) if std > 1e-12 else 0.0,
        "zero_crossing_rate": zero_crossings / len(x),
        "mean_abs_change": mean_abs_change,
    }


def _frequency_domain_features(x: np.ndarray, fs: int) -> dict:
    n = len(x)
    x_centered = x - np.mean(x)
    spectrum = np.abs(rfft(x_centered))
    freqs = rfftfreq(n, d=1.0 / fs)

    if len(spectrum) > 1:
        dominant_idx = np.argmax(spectrum[1:]) + 1
        dominant_freq = freqs[dominant_idx]
    else:
        dominant_freq = 0.0

    energy = np.sum(spectrum ** 2) / n

    psd = spectrum ** 2
    psd_sum = np.sum(psd)
    if psd_sum > 1e-12:
        p = psd / psd_sum
        p = p[p > 1e-12]
        spectral_entropy = -np.sum(p * np.log2(p)) / np.log2(len(p)) if len(p) > 1 else 0.0
    else:
        spectral_entropy = 0.0

    return {"dominant_freq": dominant_freq, "spectral_energy": energy,
            "spectral_entropy": spectral_entropy}


def _correlation_features(window_df_values: dict) -> dict:
    feats = {}
    for loc in SENSOR_LOCATIONS:
        for modality, axes in SENSOR_MODALITIES.items():
            cols = [f"{loc}_{a}" for a in axes]
            if not all(c in window_df_values for c in cols):
                continue
            x, y, z = (window_df_values[c] for c in cols)
            pairs = {"xy": (x, y), "yz": (y, z), "xz": (x, z)}
            for pair_name, (a, b) in pairs.items():
                if np.std(a) < 1e-12 or np.std(b) < 1e-12:
                    corr = 0.0
                else:
                    corr = np.corrcoef(a, b)[0, 1]
                feats[f"{loc}_{modality}_corr_{pair_name}"] = corr
    return feats


def extract_features_for_window(window: np.ndarray, signal_cols: list, fs: int) -> dict:
    feats = {}
    channel_values = {}
    for i, ch in enumerate(signal_cols):
        x = window[:, i]
        channel_values[ch] = x
        td = _time_domain_features(x)
        fd = _frequency_domain_features(x, fs)
        for k, v in td.items():
            feats[f"{ch}__{k}"] = v
        for k, v in fd.items():
            feats[f"{ch}__{k}"] = v
    feats.update(_correlation_features(channel_values))
    return feats


def extract_feature_matrix(windows_array: np.ndarray, signal_cols: list,
                            fs: int = SAMPLING_RATE_HZ) -> pd.DataFrame:
    n_windows = windows_array.shape[0]
    rows = []
    for i in range(n_windows):
        rows.append(extract_features_for_window(windows_array[i], signal_cols, fs))
        if (i + 1) % 1000 == 0 or (i + 1) == n_windows:
            print(f"\r[features] Extracted {i + 1}/{n_windows} windows", end="")
    print()
    feature_df = pd.DataFrame(rows)
    print(f"[features] Feature matrix shape: {feature_df.shape}")
    return feature_df


def run_feature_extraction(meta: pd.DataFrame, windows_array: np.ndarray, signal_cols: list):
    feature_df = extract_feature_matrix(windows_array, signal_cols)
    full_df = pd.concat([meta.reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)
    out_path = os.path.join(PROCESSED_DATA_DIR, "pamap2_features.pkl")
    full_df.to_pickle(out_path)
    print(f"[features] Saved feature dataset to {out_path}")
    return full_df


# ==============================================================================
# SECTION 6: MODEL TRAINING (GridSearchCV) + EVALUATION
# ==============================================================================

def _build_estimator(model_name: str):
    if model_name == "LogisticRegression":
        return LogisticRegression(random_state=RANDOM_STATE)
    if model_name == "DecisionTree":
        return DecisionTreeClassifier(random_state=RANDOM_STATE)
    if model_name == "RandomForest":
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    if model_name == "KNN":
        return KNeighborsClassifier(n_jobs=-1)
    if model_name == "GaussianNB":
        return GaussianNB()
    if model_name == "LinearSVM":
        return LinearSVC(random_state=RANDOM_STATE, max_iter=5000, dual="auto")
    if model_name == "XGBoost":
        if not _XGBOOST_AVAILABLE:
            raise ImportError(
                "xgboost is not installed in this environment. "
                "Install it with `pip install xgboost` to train this model."
            )
        return XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", n_jobs=-1)
    raise ValueError(f"Unknown model: {model_name}")


def load_feature_dataset() -> pd.DataFrame:
    path = os.path.join(PROCESSED_DATA_DIR, "pamap2_features.pkl")
    df = pd.read_pickle(path)
    print(f"[train_evaluate] Loaded feature dataset: {df.shape}")
    return df


def prepare_X_y(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    X = df[feature_cols].to_numpy()
    y_raw = df["activity_label"].to_numpy()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)
    return X, y, feature_cols, encoder


def plot_confusion_matrix(cm: np.ndarray, class_names: list, model_name: str, out_path: str):
    fig, ax = plt.subplots(figsize=(9, 7))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                cbar_kws={"label": "Proportion"})
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"Confusion Matrix (row-normalized) — {model_name}\nPAMAP2, held-out test set")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_model_comparison(results_df: pd.DataFrame, out_path: str):
    metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    plot_df = results_df.melt(id_vars="model", value_vars=metrics,
                               var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(data=plot_df, x="model", y="score", hue="metric", ax=ax)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Score")
    ax.set_xlabel("Model")
    ax.set_title("Model Comparison — PAMAP2 Held-Out Test Set")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_cv_boxplot(cv_scores_dict: dict, out_path: str):
    records = []
    for model_name, scores in cv_scores_dict.items():
        for s in scores:
            records.append({"model": model_name, "cv_accuracy": s})
    plot_df = pd.DataFrame(records)
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.boxplot(data=plot_df, x="model", y="cv_accuracy", ax=ax)
    sns.stripplot(data=plot_df, x="model", y="cv_accuracy", ax=ax, color="black", alpha=0.5, size=4)
    ax.set_ylabel(f"Accuracy ({N_SPLITS_KFOLD}-Fold Stratified CV)")
    ax.set_xlabel("Model")
    ax.set_title(f"Stratified {N_SPLITS_KFOLD}-Fold Cross-Validation Accuracy — PAMAP2")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def train_and_evaluate_all_models(X, y, feature_cols, encoder):
    class_names = list(encoder.classes_)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    print(f"[train_evaluate] Train/test split: {X_train.shape[0]} / {X_test.shape[0]} windows")

    inner_cv = StratifiedKFold(n_splits=CV_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    outer_cv = StratifiedKFold(n_splits=N_SPLITS_KFOLD, shuffle=True, random_state=RANDOM_STATE)

    all_results = []
    cv_scores_dict = {}

    for model_name in MODEL_DISPLAY_ORDER:
        print(f"\n[train_evaluate] ===== {model_name} =====")
        t0 = time.time()

        try:
            base_estimator = _build_estimator(model_name)
        except ImportError as e:
            print(f"[train_evaluate] SKIPPING {model_name}: {e}")
            continue

        pipeline = Pipeline([("scaler", StandardScaler()), ("clf", base_estimator)])
        param_grid = MODEL_REGISTRY[model_name]["param_grid"]

        grid = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring="f1_macro",
                             n_jobs=-1, refit=True)
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        train_time = time.time() - t0

        y_pred = best_pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
        rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
        prec_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        cm = confusion_matrix(y_test, y_pred)
        report_dict = classification_report(y_test, y_pred, target_names=class_names,
                                             output_dict=True, zero_division=0)
        report_str = classification_report(y_test, y_pred, target_names=class_names, zero_division=0)

        print(f"[train_evaluate] Best params: {grid.best_params_}")
        print(f"[train_evaluate] Test accuracy={acc:.4f}  f1_macro={f1_macro:.4f}  "
              f"(train+tune time: {train_time:.1f}s)")

        cv_pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", _build_estimator(model_name).set_params(
                **{k.replace("clf__", ""): v for k, v in grid.best_params_.items()}
            )),
        ])
        cv_scores = cross_val_score(cv_pipeline, X, y, cv=outer_cv, scoring="accuracy", n_jobs=-1)
        cv_scores_dict[model_name] = cv_scores
        print(f"[train_evaluate] {N_SPLITS_KFOLD}-fold CV accuracy: "
              f"{cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

        model_path = os.path.join(MODELS_DIR, f"{model_name}.joblib")
        joblib.dump(best_pipeline, model_path)

        cm_fig_path = os.path.join(FIGURES_DIR, f"confusion_matrix_{model_name}.png")
        plot_confusion_matrix(cm, class_names, model_name, cm_fig_path)

        cr_csv_path = os.path.join(TABLES_DIR, f"classification_report_{model_name}.csv")
        pd.DataFrame(report_dict).transpose().to_csv(cr_csv_path)

        cr_txt_path = os.path.join(TABLES_DIR, f"classification_report_{model_name}.txt")
        with open(cr_txt_path, "w") as f:
            f.write(f"Model: {model_name}\nBest params: {grid.best_params_}\n\n")
            f.write(report_str)

        cm_csv_path = os.path.join(TABLES_DIR, f"confusion_matrix_{model_name}.csv")
        pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_csv_path)

        all_results.append({
            "model": model_name,
            "best_params": json.dumps(grid.best_params_),
            "accuracy": acc, "precision_macro": prec_macro, "recall_macro": rec_macro,
            "f1_macro": f1_macro, "precision_weighted": prec_weighted,
            "recall_weighted": rec_weighted, "f1_weighted": f1_weighted,
            "cv_accuracy_mean": cv_scores.mean(), "cv_accuracy_std": cv_scores.std(),
            "train_tune_time_sec": train_time,
        })

    results_df = pd.DataFrame(all_results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
    return results_df, cv_scores_dict, class_names


def run_training_and_evaluation():
    df = load_feature_dataset()
    X, y, feature_cols, encoder = prepare_X_y(df)

    results_df, cv_scores_dict, class_names = train_and_evaluate_all_models(X, y, feature_cols, encoder)

    comparison_path = os.path.join(TABLES_DIR, "model_comparison.csv")
    results_df.to_csv(comparison_path, index=False)
    print(f"\n[train_evaluate] Saved model comparison table to {comparison_path}")

    cv_scores_table = pd.DataFrame({m: s for m, s in cv_scores_dict.items()})
    cv_scores_table.to_csv(os.path.join(TABLES_DIR, "cv_scores_per_fold.csv"), index=False)

    plot_model_comparison(results_df, os.path.join(FIGURES_DIR, "model_comparison.png"))
    plot_cv_boxplot(cv_scores_dict, os.path.join(FIGURES_DIR, "cv_accuracy_boxplot.png"))

    joblib.dump(encoder, os.path.join(MODELS_DIR, "label_encoder.joblib"))
    with open(os.path.join(PROCESSED_DATA_DIR, "feature_columns.json"), "w") as f:
        json.dump(feature_cols, f)

    return results_df, cv_scores_dict


# ==============================================================================
# SECTION 7: REPORT GENERATION
# ==============================================================================

def generate_report(results_df: pd.DataFrame, n_windows: int, n_features: int, n_subjects: int) -> str:
    best_row = results_df.iloc[0]
    lines = []
    lines.append("# Module 1 Report — Dataset Preparation, Feature Engineering, "
                  "Model Training and Performance Evaluation")
    lines.append("## Dataset: PAMAP2 Physical Activity Monitoring\n")

    lines.append("## 1. Dataset Summary")
    lines.append(f"- Subjects used: {n_subjects}")
    lines.append(f"- Activity classes: {len(ACTIVITY_MAP)} ({', '.join(ACTIVITY_MAP.values())})")
    lines.append(f"- Sampling rate: {SAMPLING_RATE_HZ} Hz")
    lines.append(f"- Window size: {WINDOW_SIZE_SAMPLES} samples "
                 f"({WINDOW_SIZE_SAMPLES / SAMPLING_RATE_HZ:.2f} s)")
    lines.append(f"- Window overlap: {WINDOW_OVERLAP * 100:.0f}%")
    lines.append(f"- Total windows generated: {n_windows}")
    lines.append(f"- Total features extracted per window: {n_features}")
    lines.append("")

    lines.append("## 2. Preprocessing Summary")
    lines.append("- Orientation columns removed (documented as invalid in the official PAMAP2 readme).")
    lines.append("- Restricted to the standard 12-activity protocol subset.")
    lines.append("- Missing values (heart-rate sub-sampling gaps and occasional IMU dropouts) handled "
                 f"via per-subject linear interpolation (max gap: {MAX_INTERP_GAP} samples); "
                 "unresolved NaNs dropped.")
    lines.append("- Sliding-window segmentation performed per-subject, per-contiguous-activity-block "
                 "to guarantee 100% window label purity.")
    lines.append("")

    lines.append("## 3. Feature Engineering Summary")
    lines.append("Per signal channel, per window:")
    lines.append("- **Time domain:** mean, std, variance, min, max, range, median, IQR, RMS, "
                 "skewness, kurtosis, zero-crossing rate, mean absolute change.")
    lines.append("- **Frequency domain:** dominant frequency, spectral energy, spectral entropy.")
    lines.append("- **Cross-axis correlation:** x-y, y-z, x-z pairwise correlation per "
                 "accelerometer/gyroscope/magnetometer group at hand/chest/ankle.")
    lines.append("- Features standardized (zero mean, unit variance), fit on the training split only.")
    lines.append("")

    lines.append("## 4. Model Training Protocol")
    lines.append("- 7 classical models trained: Logistic Regression, Decision Tree, Random Forest, "
                 "K-Nearest Neighbors, Gaussian Naive Bayes, Linear SVM, XGBoost.")
    lines.append(f"- Hyperparameters tuned via `GridSearchCV` (inner {CV_INNER_FOLDS}-fold "
                 "Stratified CV, scoring = macro F1).")
    lines.append(f"- Held-out evaluation: stratified 80/20 train/test split (random_state={RANDOM_STATE}).")
    lines.append(f"- Additional outer {N_SPLITS_KFOLD}-fold Stratified Cross-Validation performed on "
                 "the full dataset with tuned hyperparameters to assess variance in accuracy.")
    lines.append("")

    lines.append("## 5. Results — Held-Out Test Set")
    lines.append(results_df[[
        "model", "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "cv_accuracy_mean", "cv_accuracy_std"
    ]].round(4).to_markdown(index=False))
    lines.append("")
    lines.append(f"**Best model (by macro F1): {best_row['model']}** — "
                 f"accuracy = {best_row['accuracy']:.4f}, F1 (macro) = {best_row['f1_macro']:.4f}, "
                 f"{N_SPLITS_KFOLD}-fold CV accuracy = "
                 f"{best_row['cv_accuracy_mean']:.4f} ± {best_row['cv_accuracy_std']:.4f}")
    lines.append("")

    lines.append("## 6. Artifacts Produced")
    lines.append("- `models/*.joblib` — trained pipelines (scaler + tuned classifier) per model")
    lines.append("- `results/tables/model_comparison.csv` — cross-model metric comparison")
    lines.append("- `results/tables/classification_report_<model>.csv/.txt` — per-model classification reports")
    lines.append("- `results/tables/confusion_matrix_<model>.csv` — raw confusion matrices")
    lines.append("- `results/tables/cv_scores_per_fold.csv` — per-fold CV accuracy for all models")
    lines.append("- `results/figures/confusion_matrix_<model>.png` — 300 dpi confusion matrix heatmaps")
    lines.append("- `results/figures/model_comparison.png` — 300 dpi cross-model bar chart")
    lines.append("- `results/figures/cv_accuracy_boxplot.png` — 300 dpi CV accuracy distribution")
    lines.append("")

    lines.append("## 7. Scope Boundaries of Module 1")
    lines.append("This module deliberately excludes TinyML profiling, robustness/perturbation testing, "
                 "and cross-model statistical significance testing (Friedman/Nemenyi, etc.). Those are "
                 "implemented in dedicated downstream modules operating on the artifacts produced here.")

    report_text = "\n".join(lines)
    report_path = os.path.join(REPORTS_DIR, "module1_report.md")
    with open(report_path, "w") as f:
        f.write(report_text)
    print(f"[report] Saved report to {report_path}")
    return report_path


# ==============================================================================
# SECTION 8: MAIN ORCHESTRATION
# ==============================================================================

def main():
    print("=" * 50)
    print("MODULE 1 START — PAMAP2")
    print("=" * 50)

    try:
        protocol_dir = download_pamap2()
    except DatasetDownloadError as e:
        print(f"\n[main] Automatic download unavailable:\n{e}\n")
        print("[main] Aborting Module 1 — real PAMAP2 data is required to produce publishable results.")
        return

    clean_df = run_preprocessing(protocol_dir)
    meta, windows_array, signal_cols = run_segmentation(clean_df)
    feature_df = run_feature_extraction(meta, windows_array, signal_cols)
    results_df, cv_scores_dict = run_training_and_evaluation()

    n_features = feature_df.shape[1] - 4
    generate_report(results_df, n_windows=len(feature_df), n_features=n_features,
                     n_subjects=clean_df["subject_id"].nunique())

    print("\n" + "=" * 50)
    print("MODULE 1 COMPLETE")
    print("=" * 50)


if __name__ == "__main__":
    main()

MODULE 1 START — PAMAP2
[download] PAMAP2 already present at C:\Users\Rohan Jaiswal\pamap2_training\data\raw\PAMAP2_Dataset\Protocol — skipping download.
[preprocessing] Loaded raw data: 2872533 rows, 9 subjects.
[preprocessing] Filtered to 12 protocol activities: 2872533 -> 1942872 rows.
[preprocessing] Missing values: 1963923 -> 0 after interpolation.
[preprocessing] Dropped 0 rows with unresolved NaNs (1942872 rows remain).
[preprocessing] Saved cleaned dataset to C:\Users\Rohan Jaiswal\pamap2_training\data\interim\pamap2_clean.pkl
[segmentation] Built 7435 windows from 106 contiguous activity blocks (window=512 samples, step=256 samples, 50% overlap).
[segmentation] Saved window metadata to C:\Users\Rohan Jaiswal\pamap2_training\data\interim\windows_meta.pkl
[segmentation] Saved window array (7435, 512, 28) to C:\Users\Rohan Jaiswal\pamap2_training\data\interim\windows_array.npy
[features] Extracted 7435/7435 windows
[features] Feature matrix shape: (7435, 475)
[features] Saved fea

In [3]:
"""
================================================================================
MODULE 2 — TinyML-Oriented Deployment Analysis
Dataset: PAMAP2 Physical Activity Monitoring
Part of: "Toward Deployment-Aware Human Activity Recognition: A Multi-Dataset
          Evaluation of Classical Machine Learning Models Using Robustness,
          Statistical Validation, and TinyML Metrics."

SELF-CONTAINED SINGLE-FILE IMPLEMENTATION
------------------------------------------
Paste this entire script into one Jupyter cell (or run as
`python module2_pamap2_tinyml.py`), in the SAME working directory that
contains the `pamap2_training/` folder produced by Module 1.

STRICT CONTRACT WITH MODULE 1
------------------------------
  - Does NOT retrain any model.
  - Does NOT re-run preprocessing, segmentation, or feature extraction.
  - Only READS from ./pamap2_training/ (models, comparison table, and the
    already-processed feature matrix — used solely to source realistic,
    correctly-shaped input vectors for latency benchmarking, not to
    reprocess raw data).
  - Never writes to or modifies anything under ./pamap2_training/.

For each of the 7 models trained in Module 1, this module computes:
  - Model Size (KB)                 -- measured directly from the saved .joblib file
  - Prediction Latency (ms)         -- measured, single-sample inference, on the
                                        already-trained pipeline
  - Training Time (s)               -- carried over from Module 1's results
                                        (never re-measured, never re-run)
  - Estimated RAM Usage (KB)        -- proxy estimate (documented formula/assumptions)
  - Estimated Flash Usage (KB)      -- proxy estimate (documented formula/assumptions)
  - Deployment Friendliness Score   -- composite 0-100 score, relative ranking
  - Deployment Tier                 -- absolute-threshold classification

All outputs are saved inside ./pamap2_tinyml/.

NOTE ON "ESTIMATED" METRICS
----------------------------
RAM and Flash usage are PROXY estimates (Tier-1 TinyML metrics), not measured
on physical embedded hardware. They use documented, transparent formulas
based on the measured on-disk model size and input feature-vector size. A
Tier-2 embedded/MCU benchmarking pass (e.g. via emlearn/m2cgen export and
on-device timing) is explicitly out of scope here and is left as a future
extension — see the constants and docstrings below for the exact assumptions
used, so results stay reproducible and auditable.
================================================================================
"""

import os
import sys
import json
import time
import warnings

import numpy as np
import pandas as pd
import joblib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")


# ==============================================================================
# SECTION 1: CONFIGURATION
# ==============================================================================

# ---- Module 1 outputs (READ-ONLY — never written to) -----------------------
MODULE1_ROOT = os.path.join(os.getcwd(), "pamap2_training")
MODULE1_MODELS_DIR = os.path.join(MODULE1_ROOT, "models")
MODULE1_TABLES_DIR = os.path.join(MODULE1_ROOT, "results", "tables")
MODULE1_PROCESSED_DIR = os.path.join(MODULE1_ROOT, "data", "processed")
MODULE1_COMPARISON_CSV = os.path.join(MODULE1_TABLES_DIR, "model_comparison.csv")
MODULE1_FEATURES_PKL = os.path.join(MODULE1_PROCESSED_DIR, "pamap2_features.pkl")
MODULE1_FEATURE_COLUMNS_JSON = os.path.join(MODULE1_PROCESSED_DIR, "feature_columns.json")

# ---- Module 2 outputs -------------------------------------------------------
PROJECT_ROOT = os.path.join(os.getcwd(), "pamap2_tinyml")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

for d in [TABLES_DIR, FIGURES_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

NON_FEATURE_COLS = ["window_id", "subject_id", "activity_id", "activity_label"]

MODEL_DISPLAY_ORDER = [
    "LogisticRegression", "DecisionTree", "RandomForest", "KNN",
    "GaussianNB", "LinearSVM", "XGBoost",
]

FIGURE_DPI = 300
RANDOM_STATE = 42

# ---- Latency benchmarking parameters ----------------------------------------
LATENCY_N_SAMPLES = 200        # number of single-instance predictions timed per model
LATENCY_N_WARMUP = 10          # warm-up predictions (excluded from timing)

# ---- Proxy RAM / Flash estimation formulas (documented assumptions) --------
# RAM usage on an embedded target must hold: (a) the deserialized/loaded model
# parameters, (b) the input feature buffer, and (c) a small runtime/stack
# overhead for the inference call itself.
RAM_MODEL_MULTIPLIER = 1.20     # loaded/deserialized objects typically cost
                                # ~20% more RAM than their serialized (on-disk)
                                # byte size, due to Python/joblib object overhead
RAM_RUNTIME_OVERHEAD_KB = 2.0   # fixed stack/runtime overhead assumption
BYTES_PER_FEATURE = 4           # float32 input buffer, one value per feature

# Flash usage holds the serialized model parameters (read-only) plus a fixed
# code-size overhead for the inference engine itself (e.g. a lightweight
# C runtime such as emlearn/m2cgen, assumed constant across models here).
FLASH_MODEL_MULTIPLIER = 1.00   # serialized model stored close to its on-disk size
FLASH_CODE_OVERHEAD_KB = 20.0   # fixed inference-engine code footprint assumption

# ---- Deployment tier thresholds (absolute, documented) ----------------------
# Loosely modeled on common microcontroller classes:
#   Tier A ~ Cortex-M0/M0+-class (very constrained: e.g. 64 KB RAM / 256 KB flash)
#   Tier B ~ Cortex-M4/M7-class  (moderate: e.g. 512 KB RAM / 2 MB flash)
#   Tier C ~ exceeds typical MCU budgets -> gateway/edge-server class hardware only
TIER_A_RAM_KB, TIER_A_FLASH_KB, TIER_A_LATENCY_MS = 64, 256, 10
TIER_B_RAM_KB, TIER_B_FLASH_KB, TIER_B_LATENCY_MS = 512, 2048, 100

TIER_A_LABEL = "Tier A — MCU-Ready (e.g. Cortex-M0/M0+ class)"
TIER_B_LABEL = "Tier B — Edge-Capable (e.g. Cortex-M4/M7 class)"
TIER_C_LABEL = "Tier C — Gateway / Edge-Server Only"

# ---- Deployment Friendliness Score weights (documented, equal-weighted) ----
SCORE_WEIGHTS = {
    "model_size_kb": 0.25,
    "latency_ms_mean": 0.25,
    "ram_est_kb": 0.25,
    "flash_est_kb": 0.25,
}


# ==============================================================================
# SECTION 2: LOAD MODULE 1 OUTPUTS (READ-ONLY)
# ==============================================================================

def verify_module1_outputs_exist():
    missing = []
    if not os.path.isdir(MODULE1_MODELS_DIR):
        missing.append(MODULE1_MODELS_DIR)
    if not os.path.exists(MODULE1_COMPARISON_CSV):
        missing.append(MODULE1_COMPARISON_CSV)
    if not os.path.exists(MODULE1_FEATURES_PKL):
        missing.append(MODULE1_FEATURES_PKL)
    if missing:
        raise FileNotFoundError(
            "Module 1 outputs not found. Module 2 requires Module 1 to have "
            "been run first in this same working directory. Missing:\n  "
            + "\n  ".join(missing)
        )
    print("[module2] Verified Module 1 outputs are present. Proceeding read-only.")


def load_module1_comparison_table() -> pd.DataFrame:
    df = pd.read_csv(MODULE1_COMPARISON_CSV)
    print(f"[module2] Loaded Module 1 comparison table: {df.shape[0]} models.")
    return df


def load_module1_sample_inputs(n_samples: int = LATENCY_N_SAMPLES + LATENCY_N_WARMUP):
    """
    Loads the ALREADY-PROCESSED feature matrix saved by Module 1 (never
    recomputed) purely to source realistic, correctly-shaped input rows for
    latency benchmarking. This is reading a saved artifact, not preprocessing.
    """
    df = pd.read_pickle(MODULE1_FEATURES_PKL)
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    X = df[feature_cols].to_numpy()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    rng = np.random.default_rng(RANDOM_STATE)
    n_available = X.shape[0]
    n_take = min(n_samples, n_available)
    idx = rng.choice(n_available, size=n_take, replace=False)
    X_sample = X[idx]
    print(f"[module2] Loaded {X_sample.shape[0]} sample rows "
          f"({X_sample.shape[1]} features) from Module 1's processed feature matrix "
          "for latency benchmarking.")
    return X_sample, len(feature_cols)


def load_trained_models(model_names):
    """Loads each model's saved Pipeline (scaler + tuned classifier) from disk.
    Skips gracefully (with a clear warning) if a model file is missing or
    cannot be unpickled (e.g. its library isn't installed in this environment)
    — never retrains, never substitutes."""
    loaded = {}
    for name in model_names:
        path = os.path.join(MODULE1_MODELS_DIR, f"{name}.joblib")
        if not os.path.exists(path):
            print(f"[module2] SKIPPING {name}: no saved model file at {path} "
                  "(likely not trained/saved in Module 1).")
            continue
        try:
            pipeline = joblib.load(path)
        except Exception as e:
            print(f"[module2] SKIPPING {name}: failed to load saved model ({e}).")
            continue
        loaded[name] = {"pipeline": pipeline, "path": path}
    print(f"[module2] Successfully loaded {len(loaded)} / {len(model_names)} trained models.")
    return loaded


# ==============================================================================
# SECTION 3: METRIC COMPUTATION (per model, identical methodology for all)
# ==============================================================================

def measure_model_size_kb(model_path: str) -> float:
    """Measured directly from the saved artifact on disk — not estimated."""
    return os.path.getsize(model_path) / 1024.0


def measure_prediction_latency_ms(pipeline, X_sample: np.ndarray) -> dict:
    """
    Single-instance inference latency, measured on the already-trained
    pipeline exactly as saved by Module 1. No retraining, no refitting.
    """
    n_warmup = min(LATENCY_N_WARMUP, len(X_sample))
    n_timed = min(LATENCY_N_SAMPLES, len(X_sample) - n_warmup)

    # Warm-up (JIT/caching effects, first-call overhead)
    for i in range(n_warmup):
        _ = pipeline.predict(X_sample[i:i + 1])

    timings_ms = []
    for i in range(n_warmup, n_warmup + n_timed):
        t0 = time.perf_counter()
        _ = pipeline.predict(X_sample[i:i + 1])
        t1 = time.perf_counter()
        timings_ms.append((t1 - t0) * 1000.0)

    timings_ms = np.array(timings_ms)
    return {
        "latency_ms_mean": float(timings_ms.mean()),
        "latency_ms_std": float(timings_ms.std()),
        "latency_ms_p95": float(np.percentile(timings_ms, 95)),
        "n_timed_samples": n_timed,
    }


def estimate_ram_kb(model_size_kb: float, n_features: int) -> float:
    """
    Proxy RAM estimate:
        RAM = (model_size_kb * RAM_MODEL_MULTIPLIER)     # loaded model params
              + (n_features * BYTES_PER_FEATURE / 1024)  # input feature buffer
              + RAM_RUNTIME_OVERHEAD_KB                  # fixed runtime overhead
    """
    input_buffer_kb = (n_features * BYTES_PER_FEATURE) / 1024.0
    return (model_size_kb * RAM_MODEL_MULTIPLIER) + input_buffer_kb + RAM_RUNTIME_OVERHEAD_KB


def estimate_flash_kb(model_size_kb: float) -> float:
    """
    Proxy Flash estimate:
        Flash = (model_size_kb * FLASH_MODEL_MULTIPLIER)  # serialized model
                + FLASH_CODE_OVERHEAD_KB                   # fixed inference-engine code
    """
    return (model_size_kb * FLASH_MODEL_MULTIPLIER) + FLASH_CODE_OVERHEAD_KB


def assign_deployment_tier(ram_kb: float, flash_kb: float, latency_ms: float) -> str:
    if ram_kb <= TIER_A_RAM_KB and flash_kb <= TIER_A_FLASH_KB and latency_ms <= TIER_A_LATENCY_MS:
        return TIER_A_LABEL
    if ram_kb <= TIER_B_RAM_KB and flash_kb <= TIER_B_FLASH_KB and latency_ms <= TIER_B_LATENCY_MS:
        return TIER_B_LABEL
    return TIER_C_LABEL


def compute_deployment_friendliness_scores(df: pd.DataFrame) -> pd.Series:
    """
    Composite 0-100 score, RELATIVE to the other models evaluated here (not
    an absolute deployability guarantee). For each cost metric (lower is
    better), normalize to [0, 1] via min-max across the evaluated models,
    average the four normalized costs with equal weights, then invert so
    higher = more deployment-friendly.
    """
    normalized_costs = pd.DataFrame(index=df.index)
    for col, weight in SCORE_WEIGHTS.items():
        values = df[col].astype(float)
        vmin, vmax = values.min(), values.max()
        if vmax > vmin:
            norm = (values - vmin) / (vmax - vmin)
        else:
            norm = pd.Series(0.0, index=values.index)
        normalized_costs[col] = norm * weight

    avg_cost = normalized_costs.sum(axis=1) / sum(SCORE_WEIGHTS.values())
    score = 100.0 * (1.0 - avg_cost)
    return score.round(2)


# ==============================================================================
# SECTION 4: FIGURES
# ==============================================================================

def plot_deployment_landscape(df: pd.DataFrame, out_path: str):
    fig, ax = plt.subplots(figsize=(11, 8))
    tier_colors = {TIER_A_LABEL: "#2ca02c", TIER_B_LABEL: "#ff7f0e", TIER_C_LABEL: "#d62728"}

    for tier, color in tier_colors.items():
        subset = df[df["deployment_tier"] == tier]
        if subset.empty:
            continue
        ax.scatter(subset["ram_est_kb"], subset["latency_ms_mean"],
                   s=subset["model_size_kb"] * 8 + 60, color=color, alpha=0.75,
                   edgecolor="black", linewidth=0.8, label=tier)

    for _, row in df.iterrows():
        ax.annotate(row["model"], (row["ram_est_kb"], row["latency_ms_mean"]),
                    textcoords="offset points", xytext=(6, 6), fontsize=10)

    ax.axvline(TIER_A_RAM_KB, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    ax.axvline(TIER_B_RAM_KB, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    ax.axhline(TIER_A_LATENCY_MS, color="gray", linestyle=":", linewidth=1, alpha=0.6)

    ax.set_xscale("log")
    ax.set_xlabel("Estimated RAM Usage (KB, log scale)")
    ax.set_ylabel("Prediction Latency (ms)")
    ax.set_title("Deployment Landscape — PAMAP2 Classical Models\n"
                 "(bubble size = model size on disk, KB)")
    ax.legend(title="Deployment Tier", loc="upper left", fontsize=9)
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_metric_bar(df: pd.DataFrame, metric_col: str, ylabel: str, title: str, out_path: str,
                     log_scale: bool = False):
    fig, ax = plt.subplots(figsize=(11, 7))
    plot_df = df.sort_values(metric_col, ascending=True)
    sns.barplot(data=plot_df, x="model", y=metric_col, hue="model", ax=ax,
                palette="viridis", legend=False)
    if log_scale:
        ax.set_yscale("log")
    ax.set_xlabel("Model")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


# ==============================================================================
# SECTION 5: REPORT GENERATION
# ==============================================================================

def generate_report(results_df: pd.DataFrame, ranking_df: pd.DataFrame,
                     excluded_models: list) -> str:
    best = ranking_df.iloc[0]
    lines = []
    lines.append("# Module 2 Report — TinyML-Oriented Deployment Analysis")
    lines.append("## Dataset: PAMAP2 Physical Activity Monitoring\n")

    lines.append("## 1. Scope and Constraints")
    lines.append("- This module performs **no retraining** and **no re-preprocessing**. "
                 "All models were loaded exactly as saved by Module 1 "
                 "(`pamap2_training/models/*.joblib`).")
    lines.append("- Training Time values are carried over unchanged from Module 1's "
                 "`model_comparison.csv` (`train_tune_time_sec`).")
    lines.append("- Input rows used for latency benchmarking were sourced from Module 1's "
                 "already-processed feature matrix (`data/processed/pamap2_features.pkl`), "
                 "read only — never recomputed.")
    if excluded_models:
        lines.append(f"- Excluded from this analysis (no saved model artifact found): "
                     f"{', '.join(excluded_models)}.")
    lines.append("")

    lines.append("## 2. Metric Definitions and Methodology")
    lines.append("| Metric | How it was obtained |")
    lines.append("|---|---|")
    lines.append("| Model Size (KB) | Measured directly: on-disk size of the saved `.joblib` file |")
    lines.append(f"| Prediction Latency (ms) | Measured: mean single-instance `predict()` time over "
                 f"{LATENCY_N_SAMPLES} timed calls (after {LATENCY_N_WARMUP} warm-up calls) |")
    lines.append("| Training Time (s) | Carried over from Module 1 (`train_tune_time_sec`); not re-measured |")
    lines.append(f"| Estimated RAM Usage (KB) | Proxy estimate: `model_size_kb × {RAM_MODEL_MULTIPLIER}` "
                 f"+ input feature buffer (`n_features × {BYTES_PER_FEATURE} bytes`) "
                 f"+ `{RAM_RUNTIME_OVERHEAD_KB} KB` fixed runtime overhead |")
    lines.append(f"| Estimated Flash Usage (KB) | Proxy estimate: `model_size_kb × {FLASH_MODEL_MULTIPLIER}` "
                 f"+ `{FLASH_CODE_OVERHEAD_KB} KB` fixed inference-engine code overhead |")
    lines.append("| Deployment Friendliness Score (0-100) | Composite, equal-weighted (25% each) "
                 "min-max-normalized inverse cost across Model Size, Latency, Estimated RAM, "
                 "Estimated Flash — **relative to the 7 models evaluated here**, not an absolute "
                 "deployability guarantee |")
    lines.append("| Deployment Tier | Absolute-threshold classification against typical MCU-class "
                 "budgets (see thresholds below) |")
    lines.append("")

    lines.append("### Deployment Tier Thresholds")
    lines.append(f"- **{TIER_A_LABEL}**: RAM ≤ {TIER_A_RAM_KB} KB, Flash ≤ {TIER_A_FLASH_KB} KB, "
                 f"Latency ≤ {TIER_A_LATENCY_MS} ms")
    lines.append(f"- **{TIER_B_LABEL}**: RAM ≤ {TIER_B_RAM_KB} KB, Flash ≤ {TIER_B_FLASH_KB} KB, "
                 f"Latency ≤ {TIER_B_LATENCY_MS} ms")
    lines.append(f"- **{TIER_C_LABEL}**: exceeds the above budgets")
    lines.append("")
    lines.append("> **Important caveat:** RAM/Flash figures are Tier-1 proxy estimates based on "
                 "documented, transparent formulas — not measurements from physical embedded "
                 "hardware. A Tier-2 embedded benchmarking pass (e.g. exporting via emlearn/m2cgen "
                 "and timing on an actual MCU) is left as a documented future extension of this "
                 "framework and is out of scope for Module 2.")
    lines.append("")

    lines.append("## 3. TinyML Comparison Table")
    display_cols = ["model", "model_size_kb", "latency_ms_mean", "training_time_sec",
                    "ram_est_kb", "flash_est_kb", "deployment_score", "deployment_tier"]
    lines.append(results_df[display_cols].round(3).to_markdown(index=False))
    lines.append("")

    lines.append("## 4. Deployment Ranking (by Deployment Friendliness Score)")
    rank_cols = ["rank", "model", "deployment_score", "deployment_tier"]
    lines.append(ranking_df[rank_cols].to_markdown(index=False))
    lines.append("")
    lines.append(f"**Most deployment-friendly model: {best['model']}** "
                 f"(score = {best['deployment_score']:.2f}, tier = {best['deployment_tier']})")
    lines.append("")

    lines.append("## 5. Artifacts Produced")
    lines.append("- `results/tables/tinyml_comparison.csv` — full per-model metric table")
    lines.append("- `results/tables/deployment_ranking.csv` — models ranked by Deployment Friendliness Score")
    lines.append("- `results/tables/comparison_results.csv` — same metric table (canonical export name)")
    lines.append("- `results/figures/deployment_landscape.png` — RAM vs. latency scatter, "
                 "bubble size = model size, colored by tier (300 dpi)")
    lines.append("- `results/figures/model_size_comparison.png`")
    lines.append("- `results/figures/latency_comparison.png`")
    lines.append("- `results/figures/training_time_comparison.png`")
    lines.append("- `results/figures/ram_usage_comparison.png`")
    lines.append("- `results/figures/flash_usage_comparison.png`")
    lines.append("- `results/figures/deployment_score_comparison.png`")
    lines.append("(all figures saved at 300 dpi)")
    lines.append("")

    lines.append("## 6. Scope Boundaries of Module 2")
    lines.append("This module deliberately excludes robustness/perturbation testing and "
                 "cross-model statistical significance testing — those remain separate, "
                 "dedicated modules. This module also does not perform Tier-2 embedded/MCU "
                 "benchmarking; see the caveat in Section 2.")

    report_text = "\n".join(lines)
    report_path = os.path.join(REPORTS_DIR, "report.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)
    print(f"[report] Saved report to {report_path}")
    return report_path


# ==============================================================================
# SECTION 6: MAIN ORCHESTRATION
# ==============================================================================

def main():
    print("=" * 50)
    print("MODULE 2 START — TinyML-Oriented Deployment Analysis (PAMAP2)")
    print("=" * 50)

    verify_module1_outputs_exist()
    comparison_df = load_module1_comparison_table()
    X_sample, n_features = load_module1_sample_inputs()

    loaded_models = load_trained_models(MODEL_DISPLAY_ORDER)
    excluded_models = [m for m in MODEL_DISPLAY_ORDER if m not in loaded_models]

    records = []
    for model_name, info in loaded_models.items():
        print(f"\n[module2] ===== Profiling {model_name} =====")
        pipeline = info["pipeline"]
        model_path = info["path"]

        model_size_kb = measure_model_size_kb(model_path)
        latency_stats = measure_prediction_latency_ms(pipeline, X_sample)

        train_row = comparison_df[comparison_df["model"] == model_name]
        training_time_sec = (
            float(train_row["train_tune_time_sec"].iloc[0])
            if not train_row.empty and "train_tune_time_sec" in train_row.columns
            else np.nan
        )

        ram_est_kb = estimate_ram_kb(model_size_kb, n_features)
        flash_est_kb = estimate_flash_kb(model_size_kb)

        print(f"[module2] Model size:       {model_size_kb:.2f} KB")
        print(f"[module2] Latency (mean):   {latency_stats['latency_ms_mean']:.4f} ms")
        print(f"[module2] Training time:    {training_time_sec:.2f} s (from Module 1)")
        print(f"[module2] Estimated RAM:    {ram_est_kb:.2f} KB")
        print(f"[module2] Estimated Flash:  {flash_est_kb:.2f} KB")

        records.append({
            "model": model_name,
            "model_size_kb": model_size_kb,
            "latency_ms_mean": latency_stats["latency_ms_mean"],
            "latency_ms_std": latency_stats["latency_ms_std"],
            "latency_ms_p95": latency_stats["latency_ms_p95"],
            "training_time_sec": training_time_sec,
            "ram_est_kb": ram_est_kb,
            "flash_est_kb": flash_est_kb,
        })

    results_df = pd.DataFrame(records)

    # Deployment Friendliness Score (relative, min-max normalized inverse cost)
    results_df["deployment_score"] = compute_deployment_friendliness_scores(results_df)

    # Deployment Tier (absolute thresholds)
    results_df["deployment_tier"] = results_df.apply(
        lambda r: assign_deployment_tier(r["ram_est_kb"], r["flash_est_kb"], r["latency_ms_mean"]),
        axis=1
    )

    # Ranking table
    ranking_df = results_df.sort_values("deployment_score", ascending=False).reset_index(drop=True)
    ranking_df.insert(0, "rank", ranking_df.index + 1)

    # --- Save tables ---
    tinyml_comparison_path = os.path.join(TABLES_DIR, "tinyml_comparison.csv")
    results_df.to_csv(tinyml_comparison_path, index=False)
    print(f"\n[module2] Saved TinyML comparison table to {tinyml_comparison_path}")

    ranking_path = os.path.join(TABLES_DIR, "deployment_ranking.csv")
    ranking_df.to_csv(ranking_path, index=False)
    print(f"[module2] Saved deployment ranking table to {ranking_path}")

    comparison_results_path = os.path.join(TABLES_DIR, "comparison_results.csv")
    results_df.to_csv(comparison_results_path, index=False)
    print(f"[module2] Saved comparison_results.csv to {comparison_results_path}")

    # --- Figures ---
    plot_deployment_landscape(results_df, os.path.join(FIGURES_DIR, "deployment_landscape.png"))
    plot_metric_bar(results_df, "model_size_kb", "Model Size (KB)",
                    "Model Size Comparison — PAMAP2", os.path.join(FIGURES_DIR, "model_size_comparison.png"))
    plot_metric_bar(results_df, "latency_ms_mean", "Prediction Latency (ms)",
                    "Prediction Latency Comparison — PAMAP2",
                    os.path.join(FIGURES_DIR, "latency_comparison.png"), log_scale=True)
    plot_metric_bar(results_df, "training_time_sec", "Training Time (s)",
                    "Training Time Comparison — PAMAP2 (from Module 1)",
                    os.path.join(FIGURES_DIR, "training_time_comparison.png"))
    plot_metric_bar(results_df, "ram_est_kb", "Estimated RAM Usage (KB)",
                    "Estimated RAM Usage Comparison — PAMAP2",
                    os.path.join(FIGURES_DIR, "ram_usage_comparison.png"))
    plot_metric_bar(results_df, "flash_est_kb", "Estimated Flash Usage (KB)",
                    "Estimated Flash Usage Comparison — PAMAP2",
                    os.path.join(FIGURES_DIR, "flash_usage_comparison.png"))
    plot_metric_bar(ranking_df, "deployment_score", "Deployment Friendliness Score (0-100)",
                    "Deployment Friendliness Score — PAMAP2",
                    os.path.join(FIGURES_DIR, "deployment_score_comparison.png"))
    print("[module2] Saved all figures at 300 dpi.")

    # --- Report ---
    generate_report(results_df, ranking_df, excluded_models)

    print("\n" + "=" * 50)
    print("MODULE 2 COMPLETE")
    print("=" * 50)


if __name__ == "__main__":
    main()

MODULE 2 START — TinyML-Oriented Deployment Analysis (PAMAP2)
[module2] Verified Module 1 outputs are present. Proceeding read-only.
[module2] Loaded Module 1 comparison table: 7 models.
[module2] Loaded 210 sample rows (475 features) from Module 1's processed feature matrix for latency benchmarking.
[module2] Successfully loaded 7 / 7 trained models.

[module2] ===== Profiling LogisticRegression =====
[module2] Model size:       57.12 KB
[module2] Latency (mean):   0.4083 ms
[module2] Training time:    19.50 s (from Module 1)
[module2] Estimated RAM:    72.40 KB
[module2] Estimated Flash:  77.12 KB

[module2] ===== Profiling DecisionTree =====
[module2] Model size:       48.03 KB
[module2] Latency (mean):   0.2662 ms
[module2] Training time:    24.33 s (from Module 1)
[module2] Estimated RAM:    61.50 KB
[module2] Estimated Flash:  68.03 KB

[module2] ===== Profiling RandomForest =====
[module2] Model size:       15908.00 KB
[module2] Latency (mean):   36.9590 ms
[module2] Training ti

In [4]:
"""
================================================================================
MODULE 3 — Robustness Analysis
Dataset: PAMAP2 Physical Activity Monitoring
Part of: "Toward Deployment-Aware Human Activity Recognition: A Multi-Dataset
          Evaluation of Classical Machine Learning Models Using Robustness,
          Statistical Validation, and TinyML Metrics."

SELF-CONTAINED SINGLE-FILE IMPLEMENTATION
------------------------------------------
Paste this entire script into one Jupyter cell (or run as
`python module3_pamap2_robustness.py`), in the SAME working directory that
contains the `pamap2_training/` folder produced by Module 1.

STRICT CONTRACT WITH MODULE 1 / MODULE 2
------------------------------------------
  - Does NOT retrain any model.
  - Does NOT re-run preprocessing, segmentation, or feature extraction.
  - Only READS from ./pamap2_training/ (trained pipelines + the
    already-processed feature matrix, used solely to reconstruct the exact
    same held-out test split Module 1 used, and to serve as the clean input
    that corruptions are then applied to prior to inference).
  - Never writes to or modifies anything under ./pamap2_training/ or
    ./pamap2_tinyml/.

METHODOLOGY (identical across every model, and reused for future datasets)
----------------------------------------------------------------------------
1. Reconstruct Module 1's exact held-out test split (same random_state,
   same test_size=0.2, same stratify=y) directly from the saved feature
   matrix. This is a deterministic re-split of already-processed data, NOT
   a re-preprocessing step — no raw signal is touched.
2. Compute each model's clean (uncorrupted) baseline performance on that
   test split.
3. Apply three corruption families, each at 4 increasing severity levels,
   to the RAW (pre-scaling) feature vectors, then run inference through the
   model's saved pipeline exactly as-is (the pipeline's own StandardScaler
   handles scaling internally, unmodified):
     - Gaussian Noise Injection   (sensor noise proxy)
     - Feature Removal            (missing/failed feature channels proxy)
     - Sensor Perturbation        (entire sensor-location dropout proxy)
4. For every (model, corruption type, severity level) compute Accuracy,
   Precision (macro), Recall (macro), F1 (macro).
5. Aggregate into Average Accuracy Retention, Average F1 Retention, an
   Overall Robustness Score (0-100), and a Robustness Ranking.

All outputs are saved inside ./pamap2_robustness/.
================================================================================
"""

import os
import sys
import warnings

import numpy as np
import pandas as pd
import joblib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")


# ==============================================================================
# SECTION 1: CONFIGURATION
# ==============================================================================

# ---- Module 1 outputs (READ-ONLY — never written to) -----------------------
MODULE1_ROOT = os.path.join(os.getcwd(), "pamap2_training")
MODULE1_MODELS_DIR = os.path.join(MODULE1_ROOT, "models")
MODULE1_PROCESSED_DIR = os.path.join(MODULE1_ROOT, "data", "processed")
MODULE1_FEATURES_PKL = os.path.join(MODULE1_PROCESSED_DIR, "pamap2_features.pkl")

# ---- Module 3 outputs -------------------------------------------------------
PROJECT_ROOT = os.path.join(os.getcwd(), "pamap2_robustness")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

for d in [TABLES_DIR, FIGURES_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

NON_FEATURE_COLS = ["window_id", "subject_id", "activity_id", "activity_label"]

MODEL_DISPLAY_ORDER = [
    "LogisticRegression", "DecisionTree", "RandomForest", "KNN",
    "GaussianNB", "LinearSVM", "XGBoost",
]

FIGURE_DPI = 300
RANDOM_STATE = 42          # identical to Module 1, to reproduce the same split
TEST_SIZE = 0.2            # identical to Module 1

# ---- Corruption severity levels (documented, identical methodology for
#      every model and reused verbatim for every future dataset) ------------

# 1) Gaussian Noise Injection: additive noise, sigma = fraction * per-feature
#    std (computed from the clean test set), simulating increasing sensor
#    measurement noise.
NOISE_LEVELS = [0.10, 0.25, 0.50, 1.00]

# 2) Feature Removal: fraction of feature columns randomly zeroed out
#    (fixed selection per level via a seeded RNG), simulating increasing
#    numbers of unavailable/failed feature channels.
FEATURE_REMOVAL_FRACTIONS = [0.10, 0.20, 0.30, 0.50]

# 3) Sensor Perturbation: entire sensor-location feature blocks zeroed out,
#    cumulatively, simulating progressive physical sensor-unit failure.
#    Order: ankle -> +chest -> +hand -> +heart_rate (complete sensor loss).
SENSOR_PERTURBATION_STAGES = [
    ["ankle"],
    ["ankle", "chest"],
    ["ankle", "chest", "hand"],
    ["ankle", "chest", "hand", "heart_rate"],
]

SEVERITY_LEVEL_LABELS = ["Level 1", "Level 2", "Level 3", "Level 4"]


# ==============================================================================
# SECTION 2: LOAD MODULE 1 OUTPUTS (READ-ONLY)
# ==============================================================================

def verify_module1_outputs_exist():
    missing = []
    if not os.path.isdir(MODULE1_MODELS_DIR):
        missing.append(MODULE1_MODELS_DIR)
    if not os.path.exists(MODULE1_FEATURES_PKL):
        missing.append(MODULE1_FEATURES_PKL)
    if missing:
        raise FileNotFoundError(
            "Module 1 outputs not found. Module 3 requires Module 1 to have "
            "been run first in this same working directory. Missing:\n  "
            + "\n  ".join(missing)
        )
    print("[module3] Verified Module 1 outputs are present. Proceeding read-only.")


def load_feature_dataset_and_reconstruct_split():
    """
    Loads Module 1's already-processed feature matrix (read-only) and
    deterministically reconstructs the exact same held-out test split
    Module 1 used (same random_state, test_size, stratify). This is NOT a
    re-preprocessing step: no raw signal, window, or feature-extraction code
    is touched — only a reproducible re-split of an already-saved artifact.
    """
    df = pd.read_pickle(MODULE1_FEATURES_PKL)
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    X = df[feature_cols].to_numpy()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y_raw = df["activity_label"].to_numpy()

    # Encode labels the same way Module 1 did (alphabetical LabelEncoder order)
    classes = sorted(pd.unique(y_raw))
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y = np.array([class_to_idx[c] for c in y_raw])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    print(f"[module3] Reconstructed Module 1's held-out test split: "
          f"{X_test.shape[0]} test windows, {X_test.shape[1]} features.")
    return X_test, y_test, feature_cols, classes


def load_trained_models(model_names):
    """Loads each model's saved Pipeline exactly as Module 1 saved it.
    Skips gracefully if a model file is missing or cannot be unpickled —
    never retrains, never substitutes."""
    loaded = {}
    for name in model_names:
        path = os.path.join(MODULE1_MODELS_DIR, f"{name}.joblib")
        if not os.path.exists(path):
            print(f"[module3] SKIPPING {name}: no saved model file at {path}.")
            continue
        try:
            pipeline = joblib.load(path)
        except Exception as e:
            print(f"[module3] SKIPPING {name}: failed to load saved model ({e}).")
            continue
        loaded[name] = pipeline
    print(f"[module3] Successfully loaded {len(loaded)} / {len(model_names)} trained models.")
    return loaded


def map_feature_columns_to_sensor_location(feature_cols):
    """Each feature column is named '<channel>__<stat>' or
    '<location>_<modality>_corr_<pair>'; the channel/column always begins
    with one of hand_/chest_/ankle_/heart_rate. Returns dict: location ->
    list of column indices."""
    location_indices = {"hand": [], "chest": [], "ankle": [], "heart_rate": []}
    for idx, col in enumerate(feature_cols):
        if col.startswith("hand_"):
            location_indices["hand"].append(idx)
        elif col.startswith("chest_"):
            location_indices["chest"].append(idx)
        elif col.startswith("ankle_"):
            location_indices["ankle"].append(idx)
        elif col.startswith("heart_rate"):
            location_indices["heart_rate"].append(idx)
    return location_indices


# ==============================================================================
# SECTION 3: CORRUPTION FUNCTIONS
# (identical methodology applied uniformly to every model)
# ==============================================================================

def apply_gaussian_noise(X: np.ndarray, feature_stds: np.ndarray, sigma_fraction: float,
                          seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=sigma_fraction * feature_stds, size=X.shape)
    return X + noise


def apply_feature_removal(X: np.ndarray, fraction: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n_features = X.shape[1]
    n_remove = int(round(fraction * n_features))
    cols_to_zero = rng.choice(n_features, size=n_remove, replace=False)
    X_corrupted = X.copy()
    X_corrupted[:, cols_to_zero] = 0.0
    return X_corrupted


def apply_sensor_perturbation(X: np.ndarray, locations_to_zero: list,
                               location_indices: dict) -> np.ndarray:
    X_corrupted = X.copy()
    for loc in locations_to_zero:
        idxs = location_indices.get(loc, [])
        if idxs:
            X_corrupted[:, idxs] = 0.0
    return X_corrupted


# ==============================================================================
# SECTION 4: METRIC COMPUTATION
# ==============================================================================

def compute_metrics(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def evaluate_model_robustness(model_name: str, pipeline, X_test: np.ndarray, y_test: np.ndarray,
                               feature_stds: np.ndarray, location_indices: dict) -> list:
    rows = []

    # --- Baseline (clean, uncorrupted) ---
    y_pred_clean = pipeline.predict(X_test)
    baseline_metrics = compute_metrics(y_test, y_pred_clean)
    rows.append({
        "model": model_name, "corruption_type": "Baseline", "level": 0,
        "level_label": "Baseline (clean)", "severity_value": 0.0,
        **baseline_metrics,
    })

    # --- 1) Gaussian Noise Injection ---
    for level_idx, sigma_fraction in enumerate(NOISE_LEVELS, start=1):
        seed = RANDOM_STATE + level_idx
        X_noisy = apply_gaussian_noise(X_test, feature_stds, sigma_fraction, seed)
        y_pred = pipeline.predict(X_noisy)
        metrics = compute_metrics(y_test, y_pred)
        rows.append({
            "model": model_name, "corruption_type": "Gaussian Noise Injection",
            "level": level_idx, "level_label": SEVERITY_LEVEL_LABELS[level_idx - 1],
            "severity_value": sigma_fraction, **metrics,
        })

    # --- 2) Feature Removal ---
    for level_idx, fraction in enumerate(FEATURE_REMOVAL_FRACTIONS, start=1):
        seed = RANDOM_STATE + 100 + level_idx
        X_removed = apply_feature_removal(X_test, fraction, seed)
        y_pred = pipeline.predict(X_removed)
        metrics = compute_metrics(y_test, y_pred)
        rows.append({
            "model": model_name, "corruption_type": "Feature Removal",
            "level": level_idx, "level_label": SEVERITY_LEVEL_LABELS[level_idx - 1],
            "severity_value": fraction, **metrics,
        })

    # --- 3) Sensor Perturbation ---
    for level_idx, locations in enumerate(SENSOR_PERTURBATION_STAGES, start=1):
        X_perturbed = apply_sensor_perturbation(X_test, locations, location_indices)
        y_pred = pipeline.predict(X_perturbed)
        metrics = compute_metrics(y_test, y_pred)
        rows.append({
            "model": model_name, "corruption_type": "Sensor Perturbation",
            "level": level_idx, "level_label": SEVERITY_LEVEL_LABELS[level_idx - 1],
            "severity_value": len(locations), **metrics,
        })

    return rows


# ==============================================================================
# SECTION 5: RETENTION + ROBUSTNESS SCORE AGGREGATION
# ==============================================================================

def compute_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    for model_name, group in results_df.groupby("model"):
        baseline = group[group["corruption_type"] == "Baseline"].iloc[0]
        corrupted = group[group["corruption_type"] != "Baseline"].copy()

        corrupted["accuracy_retention"] = corrupted["accuracy"] / max(baseline["accuracy"], 1e-9)
        corrupted["f1_retention"] = corrupted["f1_macro"] / max(baseline["f1_macro"], 1e-9)

        avg_acc_retention = corrupted["accuracy_retention"].mean()
        avg_f1_retention = corrupted["f1_retention"].mean()

        # Overall Robustness Score: mean of average accuracy/F1 retention,
        # expressed on a 0-100 scale, capped at 100 (retention cannot
        # meaningfully exceed the clean baseline in a robustness context).
        overall_score = 100.0 * min((avg_acc_retention + avg_f1_retention) / 2.0, 1.0)

        summary_rows.append({
            "model": model_name,
            "baseline_accuracy": baseline["accuracy"],
            "baseline_f1_macro": baseline["f1_macro"],
            "avg_accuracy_retention": avg_acc_retention,
            "avg_f1_retention": avg_f1_retention,
            "overall_robustness_score": round(overall_score, 2),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(
        "overall_robustness_score", ascending=False
    ).reset_index(drop=True)
    summary_df.insert(0, "rank", summary_df.index + 1)
    return summary_df


def add_retention_columns(results_df: pd.DataFrame) -> pd.DataFrame:
    """Adds per-row accuracy_retention / f1_retention columns (relative to
    that model's own clean baseline) to the full granular results table."""
    out = results_df.copy()
    out["accuracy_retention"] = np.nan
    out["f1_retention"] = np.nan
    for model_name, group in out.groupby("model"):
        baseline_acc = group.loc[group["corruption_type"] == "Baseline", "accuracy"].iloc[0]
        baseline_f1 = group.loc[group["corruption_type"] == "Baseline", "f1_macro"].iloc[0]
        mask = out["model"] == model_name
        out.loc[mask, "accuracy_retention"] = out.loc[mask, "accuracy"] / max(baseline_acc, 1e-9)
        out.loc[mask, "f1_retention"] = out.loc[mask, "f1_macro"] / max(baseline_f1, 1e-9)
    return out


# ==============================================================================
# SECTION 6: FIGURES
# ==============================================================================

def plot_robustness_curve(results_df: pd.DataFrame, corruption_type: str, metric_col: str,
                           ylabel: str, out_path: str):
    fig, ax = plt.subplots(figsize=(11, 7))
    subset = results_df[results_df["corruption_type"].isin(["Baseline", corruption_type])]

    for model_name, group in subset.groupby("model"):
        group = group.copy()
        # x = 0 for baseline, 1..4 for the corruption levels
        group["x"] = group.apply(
            lambda r: 0 if r["corruption_type"] == "Baseline" else r["level"], axis=1
        )
        group = group.sort_values("x")
        ax.plot(group["x"], group[metric_col], marker="o", label=model_name, linewidth=2)

    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_xticklabels(["Baseline"] + SEVERITY_LEVEL_LABELS)
    ax.set_xlabel("Corruption Severity")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Robustness Curve — {corruption_type} — PAMAP2")
    ax.legend(title="Model", fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_retention_comparison(summary_df: pd.DataFrame, metric_col: str, ylabel: str,
                               title: str, out_path: str):
    fig, ax = plt.subplots(figsize=(11, 7))
    plot_df = summary_df.sort_values(metric_col, ascending=False)
    sns.barplot(data=plot_df, x="model", y=metric_col, hue="model", ax=ax,
                palette="viridis", legend=False)
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1, alpha=0.6,
               label="No degradation (retention = 1.0)")
    ax.set_xlabel("Model")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_overall_ranking(summary_df: pd.DataFrame, out_path: str):
    fig, ax = plt.subplots(figsize=(11, 7))
    plot_df = summary_df.sort_values("overall_robustness_score", ascending=False)
    sns.barplot(data=plot_df, x="model", y="overall_robustness_score", hue="model", ax=ax,
                palette="mako", legend=False)
    ax.set_xlabel("Model")
    ax.set_ylabel("Overall Robustness Score (0-100)")
    ax.set_title("Overall Robustness Ranking — PAMAP2")
    ax.set_ylim(0, 100)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


# ==============================================================================
# SECTION 7: REPORT GENERATION
# ==============================================================================

def generate_report(results_df: pd.DataFrame, summary_df: pd.DataFrame,
                     excluded_models: list) -> str:
    best = summary_df.iloc[0]
    lines = []
    lines.append("# Module 3 Report — Robustness Analysis")
    lines.append("## Dataset: PAMAP2 Physical Activity Monitoring\n")

    lines.append("## 1. Scope and Constraints")
    lines.append("- This module performs **no retraining** and **no re-preprocessing**. "
                 "All models were loaded exactly as saved by Module 1 "
                 "(`pamap2_training/models/*.joblib`).")
    lines.append("- The held-out test split is Module 1's exact split, deterministically "
                 f"reconstructed from the saved feature matrix (`random_state={RANDOM_STATE}`, "
                 f"`test_size={TEST_SIZE}`, stratified) — not a new split, not a re-preprocessing step.")
    if excluded_models:
        lines.append(f"- Excluded from this analysis (no saved model artifact found): "
                     f"{', '.join(excluded_models)}.")
    lines.append("")

    lines.append("## 2. Robustness Test Methodology")
    lines.append("All corruptions are applied to the raw (pre-scaling) feature vectors; each "
                 "model's own saved pipeline (including its fitted `StandardScaler`) then "
                 "performs inference unmodified.\n")
    lines.append("**1) Gaussian Noise Injection** — additive noise per feature, "
                 "`sigma = severity_fraction × per-feature std` (std computed from the clean "
                 f"test set). Severity fractions: {NOISE_LEVELS}.\n")
    lines.append("**2) Feature Removal** — a randomly selected fraction of feature columns is "
                 "zeroed out (fixed, seeded selection per level), simulating unavailable/failed "
                 f"feature channels. Fractions removed: {FEATURE_REMOVAL_FRACTIONS}.\n")
    lines.append("**3) Sensor Perturbation** — all features originating from an entire sensor "
                 "location are zeroed out, cumulatively, simulating progressive physical sensor "
                 "failure. Stages: " +
                 " -> ".join([" + ".join(s) for s in SENSOR_PERTURBATION_STAGES]) + ".\n")
    lines.append("Each corruption family is tested at 4 increasing severity levels "
                 "(Level 1 = mildest, Level 4 = most severe), applied identically to every model.")
    lines.append("")

    lines.append("## 3. Retention and Scoring Definitions")
    lines.append("- **Accuracy Retention** (per level) = corrupted accuracy / clean baseline accuracy")
    lines.append("- **F1 Retention** (per level) = corrupted F1 (macro) / clean baseline F1 (macro)")
    lines.append("- **Average Accuracy / F1 Retention** = mean retention across all 12 corrupted "
                 "conditions (3 corruption types × 4 levels)")
    lines.append("- **Overall Robustness Score (0-100)** = "
                 "`100 × min((avg_accuracy_retention + avg_f1_retention) / 2, 1.0)`")
    lines.append("- **Robustness Ranking** = models sorted by Overall Robustness Score, descending")
    lines.append("")

    lines.append("## 4. Robustness Summary")
    display_cols = ["rank", "model", "baseline_accuracy", "baseline_f1_macro",
                    "avg_accuracy_retention", "avg_f1_retention", "overall_robustness_score"]
    lines.append(summary_df[display_cols].round(4).to_markdown(index=False))
    lines.append("")
    lines.append(f"**Most robust model: {best['model']}** "
                 f"(Overall Robustness Score = {best['overall_robustness_score']:.2f})")
    lines.append("")

    lines.append("## 5. Artifacts Produced")
    lines.append("- `results/tables/robustness_results.csv` — full granular results "
                 "(model × corruption type × level × metrics)")
    lines.append("- `results/tables/robustness_summary.csv` — per-model aggregated retention "
                 "and Overall Robustness Score / ranking")
    lines.append("- `results/figures/robustness_curve_gaussian_noise.png`")
    lines.append("- `results/figures/robustness_curve_feature_removal.png`")
    lines.append("- `results/figures/robustness_curve_sensor_perturbation.png`")
    lines.append("- `results/figures/accuracy_retention_comparison.png`")
    lines.append("- `results/figures/f1_retention_comparison.png`")
    lines.append("- `results/figures/overall_robustness_ranking.png`")
    lines.append("(all figures saved at 300 dpi)")
    lines.append("")

    lines.append("## 6. Scope Boundaries of Module 3")
    lines.append("This module deliberately excludes TinyML deployment profiling (Module 2) and "
                 "cross-model statistical significance testing (a separate, dedicated module). "
                 "Corruptions are applied at the engineered-feature level, not the raw sensor "
                 "signal level; this is consistent with Module 1's architecture, where classical "
                 "ML operates on extracted statistical features rather than raw time series.")

    report_text = "\n".join(lines)
    report_path = os.path.join(REPORTS_DIR, "report.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)
    print(f"[report] Saved report to {report_path}")
    return report_path


# ==============================================================================
# SECTION 8: MAIN ORCHESTRATION
# ==============================================================================

def main():
    print("=" * 50)
    print("MODULE 3 START — Robustness Analysis (PAMAP2)")
    print("=" * 50)

    verify_module1_outputs_exist()
    X_test, y_test, feature_cols, classes = load_feature_dataset_and_reconstruct_split()
    location_indices = map_feature_columns_to_sensor_location(feature_cols)
    feature_stds = X_test.std(axis=0)
    feature_stds[feature_stds < 1e-9] = 1e-9  # guard against zero-variance columns

    loaded_models = load_trained_models(MODEL_DISPLAY_ORDER)
    excluded_models = [m for m in MODEL_DISPLAY_ORDER if m not in loaded_models]

    all_rows = []
    for model_name, pipeline in loaded_models.items():
        print(f"\n[module3] ===== Robustness testing {model_name} =====")
        rows = evaluate_model_robustness(
            model_name, pipeline, X_test, y_test, feature_stds, location_indices
        )
        all_rows.extend(rows)
        baseline_acc = rows[0]["accuracy"]
        worst_acc = min(r["accuracy"] for r in rows[1:])
        print(f"[module3] Baseline accuracy: {baseline_acc:.4f} | "
              f"Worst-case corrupted accuracy: {worst_acc:.4f}")

    results_df = pd.DataFrame(all_rows)
    results_df = add_retention_columns(results_df)

    summary_df = compute_summary(results_df)

    # --- Save tables ---
    results_path = os.path.join(TABLES_DIR, "robustness_results.csv")
    results_df.to_csv(results_path, index=False)
    print(f"\n[module3] Saved full robustness results to {results_path}")

    summary_path = os.path.join(TABLES_DIR, "robustness_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"[module3] Saved robustness summary/ranking to {summary_path}")

    # --- Figures ---
    plot_robustness_curve(results_df, "Gaussian Noise Injection", "accuracy", "Accuracy",
                          os.path.join(FIGURES_DIR, "robustness_curve_gaussian_noise.png"))
    plot_robustness_curve(results_df, "Feature Removal", "accuracy", "Accuracy",
                          os.path.join(FIGURES_DIR, "robustness_curve_feature_removal.png"))
    plot_robustness_curve(results_df, "Sensor Perturbation", "accuracy", "Accuracy",
                          os.path.join(FIGURES_DIR, "robustness_curve_sensor_perturbation.png"))

    plot_retention_comparison(summary_df, "avg_accuracy_retention", "Average Accuracy Retention",
                              "Accuracy Retention Comparison — PAMAP2",
                              os.path.join(FIGURES_DIR, "accuracy_retention_comparison.png"))
    plot_retention_comparison(summary_df, "avg_f1_retention", "Average F1 Retention",
                              "F1 Retention Comparison — PAMAP2",
                              os.path.join(FIGURES_DIR, "f1_retention_comparison.png"))
    plot_overall_ranking(summary_df, os.path.join(FIGURES_DIR, "overall_robustness_ranking.png"))
    print("[module3] Saved all figures at 300 dpi.")

    # --- Report ---
    generate_report(results_df, summary_df, excluded_models)

    print("\n" + "=" * 50)
    print("MODULE 3 COMPLETE")
    print("=" * 50)


if __name__ == "__main__":
    main()

MODULE 3 START — Robustness Analysis (PAMAP2)
[module3] Verified Module 1 outputs are present. Proceeding read-only.
[module3] Reconstructed Module 1's held-out test split: 1487 test windows, 475 features.
[module3] Successfully loaded 7 / 7 trained models.

[module3] ===== Robustness testing LogisticRegression =====
[module3] Baseline accuracy: 0.9657 | Worst-case corrupted accuracy: 0.0995

[module3] ===== Robustness testing DecisionTree =====
[module3] Baseline accuracy: 0.9361 | Worst-case corrupted accuracy: 0.0955

[module3] ===== Robustness testing RandomForest =====
[module3] Baseline accuracy: 0.9711 | Worst-case corrupted accuracy: 0.0995

[module3] ===== Robustness testing KNN =====
[module3] Baseline accuracy: 0.9670 | Worst-case corrupted accuracy: 0.0982

[module3] ===== Robustness testing GaussianNB =====
[module3] Baseline accuracy: 0.9032 | Worst-case corrupted accuracy: 0.0874

[module3] ===== Robustness testing LinearSVM =====
[module3] Baseline accuracy: 0.9731 | Wo

In [5]:
"""
================================================================================
MODULE 4 — Statistical Validation
Dataset: PAMAP2 Physical Activity Monitoring
Part of: "Toward Deployment-Aware Human Activity Recognition: A Multi-Dataset
          Evaluation of Classical Machine Learning Models Using Robustness,
          Statistical Validation, and TinyML Metrics."

SELF-CONTAINED SINGLE-FILE IMPLEMENTATION
------------------------------------------
Paste this entire script into one Jupyter cell (or run as
`python module4_pamap2_statistics.py`), in the SAME working directory that
contains the `pamap2_training/` folder produced by Module 1.

STRICT CONTRACT WITH MODULE 1 / MODULE 2 / MODULE 3
------------------------------------------------------
  - Does NOT retrain any model.
  - Does NOT re-run preprocessing, segmentation, or feature extraction.
  - Does NOT re-run TinyML profiling (Module 2) or robustness testing (Module 3).
  - Only READS from ./pamap2_training/ (trained pipelines + the
    already-processed feature matrix, used solely to deterministically
    reconstruct Module 1's exact held-out test split and obtain each
    model's predictions on it).
  - Never writes to or modifies ./pamap2_training/, ./pamap2_tinyml/, or
    ./pamap2_robustness/.

METHODOLOGY (identical across every model, and reused for future datasets)
----------------------------------------------------------------------------
1. Reconstruct Module 1's exact held-out test split (same random_state,
   test_size=0.2, stratify=y) directly from the saved feature matrix —
   the same deterministic re-split used in Module 3, not a re-preprocessing
   step.
2. Obtain each model's predictions on that fixed test set (inference only,
   no retraining) and derive a per-sample binary correctness vector.
3. Wilson 95% Confidence Intervals on each model's accuracy.
4. Cochran's Q Test — omnibus test of whether accuracy differs across all
   models simultaneously (paired binary outcomes on the same test samples).
5. Pairwise McNemar Tests — every model pair, using the same fixed test set.
6. Holm-Bonferroni correction of the pairwise McNemar p-values to control
   the family-wise error rate across all pairwise comparisons.

All outputs are saved inside ./pamap2_statistics/.
================================================================================
"""

import os
import sys
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import joblib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats as scipy_stats
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")


# ==============================================================================
# SECTION 1: CONFIGURATION
# ==============================================================================

# ---- Module 1 outputs (READ-ONLY — never written to) -----------------------
MODULE1_ROOT = os.path.join(os.getcwd(), "pamap2_training")
MODULE1_MODELS_DIR = os.path.join(MODULE1_ROOT, "models")
MODULE1_PROCESSED_DIR = os.path.join(MODULE1_ROOT, "data", "processed")
MODULE1_FEATURES_PKL = os.path.join(MODULE1_PROCESSED_DIR, "pamap2_features.pkl")

# ---- Module 4 outputs -------------------------------------------------------
PROJECT_ROOT = os.path.join(os.getcwd(), "pamap2_statistics")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

for d in [TABLES_DIR, FIGURES_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

NON_FEATURE_COLS = ["window_id", "subject_id", "activity_id", "activity_label"]

MODEL_DISPLAY_ORDER = [
    "LogisticRegression", "DecisionTree", "RandomForest", "KNN",
    "GaussianNB", "LinearSVM", "XGBoost",
]

FIGURE_DPI = 300
RANDOM_STATE = 42          # identical to Module 1 / Module 3, to reproduce the same split
TEST_SIZE = 0.2            # identical to Module 1 / Module 3

ALPHA = 0.05                # significance level used throughout
Z_95 = 1.959963984540054    # z-score for a 95% Wilson confidence interval

# McNemar exact-vs-asymptotic decision threshold (standard recommendation:
# use the exact binomial test when the number of discordant pairs is small)
MCNEMAR_EXACT_THRESHOLD = 25


# ==============================================================================
# SECTION 2: LOAD MODULE 1 OUTPUTS (READ-ONLY) + RECONSTRUCT TEST SPLIT
# ==============================================================================

def verify_module1_outputs_exist():
    missing = []
    if not os.path.isdir(MODULE1_MODELS_DIR):
        missing.append(MODULE1_MODELS_DIR)
    if not os.path.exists(MODULE1_FEATURES_PKL):
        missing.append(MODULE1_FEATURES_PKL)
    if missing:
        raise FileNotFoundError(
            "Module 1 outputs not found. Module 4 requires Module 1 to have "
            "been run first in this same working directory. Missing:\n  "
            + "\n  ".join(missing)
        )
    print("[module4] Verified Module 1 outputs are present. Proceeding read-only.")


def load_feature_dataset_and_reconstruct_split():
    """
    Loads Module 1's already-processed feature matrix (read-only) and
    deterministically reconstructs the exact same held-out test split used
    by Module 1 and Module 3 (same random_state, test_size, stratify). This
    is NOT a re-preprocessing step — only a reproducible re-split of an
    already-saved artifact.
    """
    df = pd.read_pickle(MODULE1_FEATURES_PKL)
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    X = df[feature_cols].to_numpy()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y_raw = df["activity_label"].to_numpy()

    classes = sorted(pd.unique(y_raw))
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y = np.array([class_to_idx[c] for c in y_raw])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    print(f"[module4] Reconstructed Module 1's held-out test split: "
          f"{X_test.shape[0]} test windows, {X_test.shape[1]} features.")
    return X_test, y_test, classes


def load_trained_models(model_names):
    """Loads each model's saved Pipeline exactly as Module 1 saved it.
    Skips gracefully if a model file is missing or cannot be unpickled —
    never retrains, never substitutes."""
    loaded = {}
    for name in model_names:
        path = os.path.join(MODULE1_MODELS_DIR, f"{name}.joblib")
        if not os.path.exists(path):
            print(f"[module4] SKIPPING {name}: no saved model file at {path}.")
            continue
        try:
            pipeline = joblib.load(path)
        except Exception as e:
            print(f"[module4] SKIPPING {name}: failed to load saved model ({e}).")
            continue
        loaded[name] = pipeline
    print(f"[module4] Successfully loaded {len(loaded)} / {len(model_names)} trained models.")
    return loaded


def get_correctness_matrix(loaded_models: dict, X_test: np.ndarray, y_test: np.ndarray) -> pd.DataFrame:
    """
    Runs inference (no retraining) for every loaded model on the fixed test
    set and returns an (n_samples x n_models) DataFrame of 1/0 correctness
    indicators — the shared input to every statistical test in this module.
    """
    correctness = {}
    for model_name, pipeline in loaded_models.items():
        y_pred = pipeline.predict(X_test)
        correctness[model_name] = (y_pred == y_test).astype(int)
    return pd.DataFrame(correctness)


# ==============================================================================
# SECTION 3: WILSON 95% CONFIDENCE INTERVALS
# ==============================================================================

def wilson_confidence_interval(successes: int, n: int, z: float = Z_95) -> tuple:
    """Wilson score interval for a binomial proportion (accuracy)."""
    if n == 0:
        return 0.0, 0.0, 0.0
    phat = successes / n
    denom = 1 + (z ** 2) / n
    center = (phat + (z ** 2) / (2 * n)) / denom
    margin = (z * np.sqrt((phat * (1 - phat) / n) + (z ** 2) / (4 * n ** 2))) / denom
    lower = max(0.0, center - margin)
    upper = min(1.0, center + margin)
    return phat, lower, upper


def compute_wilson_table(correctness_df: pd.DataFrame) -> pd.DataFrame:
    n = len(correctness_df)
    rows = []
    for model_name in correctness_df.columns:
        successes = int(correctness_df[model_name].sum())
        phat, lower, upper = wilson_confidence_interval(successes, n)
        rows.append({
            "model": model_name,
            "n_test_samples": n,
            "n_correct": successes,
            "accuracy": phat,
            "wilson_ci_lower": lower,
            "wilson_ci_upper": upper,
            "ci_width": upper - lower,
        })
    return pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)


# ==============================================================================
# SECTION 4: COCHRAN'S Q TEST
# ==============================================================================

def cochrans_q_test(correctness_df: pd.DataFrame) -> dict:
    """
    Omnibus test for whether k related (paired) binary outcome sequences
    (one per model, on the same test samples) have equal success rates.

    Q = (k-1) * [k * sum_j(Gj^2) - (sum_j Gj)^2] / [k * L - sum_i(Li^2)]

    where Gj = column total (successes for model j), Li = row total
    (number of models correct for sample i), L = grand total of successes.
    Q ~ chi-square with (k-1) degrees of freedom under H0.
    """
    k = correctness_df.shape[1]
    N = correctness_df.shape[0]

    if k < 2:
        return {"k_models": k, "n_samples": N, "Q_statistic": np.nan,
                "df": np.nan, "p_value": np.nan,
                "conclusion": "Fewer than 2 models available — Cochran's Q not applicable."}

    col_totals = correctness_df.sum(axis=0).to_numpy()   # Gj, length k
    row_totals = correctness_df.sum(axis=1).to_numpy()   # Li, length N
    L = col_totals.sum()

    numerator = (k - 1) * (k * np.sum(col_totals ** 2) - L ** 2)
    denominator = k * L - np.sum(row_totals ** 2)

    if denominator == 0:
        # Degenerate case: every model got every sample right (or wrong) —
        # there is no variability to test.
        return {"k_models": k, "n_samples": N, "Q_statistic": 0.0,
                "df": k - 1, "p_value": 1.0,
                "conclusion": "Degenerate case (no variability across models/samples): "
                              "all models agree on every sample. No significant difference."}

    Q = numerator / denominator
    df = k - 1
    p_value = float(scipy_stats.chi2.sf(Q, df))
    conclusion = (
        f"Reject H0 at alpha={ALPHA}: model accuracies differ significantly (p={p_value:.4g})."
        if p_value < ALPHA else
        f"Fail to reject H0 at alpha={ALPHA}: no significant difference detected among "
        f"model accuracies overall (p={p_value:.4g})."
    )
    return {"k_models": k, "n_samples": N, "Q_statistic": float(Q), "df": df,
            "p_value": p_value, "conclusion": conclusion}


# ==============================================================================
# SECTION 5: PAIRWISE MCNEMAR TESTS
# ==============================================================================

def mcnemar_test(correct_a: np.ndarray, correct_b: np.ndarray) -> dict:
    """
    McNemar's test on the discordant pairs between two paired binary
    sequences (same test samples, two different models).
      n10 = A correct, B incorrect
      n01 = A incorrect, B correct
    Uses the exact binomial test when discordant pairs < MCNEMAR_EXACT_THRESHOLD
    (standard recommendation for small samples), otherwise the
    continuity-corrected chi-square approximation.
    """
    n10 = int(np.sum((correct_a == 1) & (correct_b == 0)))
    n01 = int(np.sum((correct_a == 0) & (correct_b == 1)))
    n_discordant = n10 + n01

    if n_discordant == 0:
        return {"n10": n10, "n01": n01, "n_discordant": 0, "statistic": 0.0,
                "p_value": 1.0, "method": "degenerate (no discordant pairs)"}

    if n_discordant < MCNEMAR_EXACT_THRESHOLD:
        k = min(n10, n01)
        p_value = float(scipy_stats.binomtest(k, n_discordant, 0.5).pvalue)
        return {"n10": n10, "n01": n01, "n_discordant": n_discordant,
                "statistic": float(k), "p_value": p_value, "method": "exact binomial"}
    else:
        statistic = ((abs(n10 - n01) - 1) ** 2) / n_discordant
        p_value = float(scipy_stats.chi2.sf(statistic, df=1))
        return {"n10": n10, "n01": n01, "n_discordant": n_discordant,
                "statistic": float(statistic), "p_value": p_value,
                "method": "continuity-corrected chi-square"}


def compute_pairwise_mcnemar(correctness_df: pd.DataFrame) -> pd.DataFrame:
    models = list(correctness_df.columns)
    rows = []
    for model_a, model_b in combinations(models, 2):
        result = mcnemar_test(
            correctness_df[model_a].to_numpy(), correctness_df[model_b].to_numpy()
        )
        rows.append({"model_a": model_a, "model_b": model_b, **result})
    return pd.DataFrame(rows)


def build_pvalue_matrix(pairwise_df: pd.DataFrame, models: list, value_col: str = "p_value") -> pd.DataFrame:
    matrix = pd.DataFrame(np.nan, index=models, columns=models)
    for _, row in pairwise_df.iterrows():
        matrix.loc[row["model_a"], row["model_b"]] = row[value_col]
        matrix.loc[row["model_b"], row["model_a"]] = row[value_col]
    for m in models:
        matrix.loc[m, m] = np.nan
    return matrix


# ==============================================================================
# SECTION 6: HOLM-BONFERRONI CORRECTION
# ==============================================================================

def holm_bonferroni_correction(p_values: list, alpha: float = ALPHA) -> list:
    """
    Holm's step-down procedure. Returns adjusted p-values in the ORIGINAL
    input order (enforcing monotonicity as required by the method).
    """
    m = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    adjusted = [0.0] * m
    running_max = 0.0
    for rank, (orig_idx, p) in enumerate(indexed):
        adj = (m - rank) * p
        running_max = max(running_max, adj)
        adjusted[orig_idx] = min(running_max, 1.0)
    return adjusted


def compute_holm_adjusted_table(pairwise_df: pd.DataFrame, alpha: float = ALPHA) -> pd.DataFrame:
    df = pairwise_df.copy()
    df["holm_adjusted_p_value"] = holm_bonferroni_correction(df["p_value"].tolist(), alpha)
    df["significant_after_holm"] = df["holm_adjusted_p_value"] < alpha
    return df.sort_values("p_value").reset_index(drop=True)


# ==============================================================================
# SECTION 7: STATISTICAL SUMMARY TABLE
# ==============================================================================

def compute_statistical_summary(wilson_df: pd.DataFrame, holm_df: pd.DataFrame,
                                 cochran_result: dict) -> pd.DataFrame:
    best_model = wilson_df.iloc[0]["model"]
    summary_rows = []
    for _, row in wilson_df.iterrows():
        model_name = row["model"]
        if model_name == best_model:
            differs_from_best = False
            holm_p_vs_best = np.nan
        else:
            match = holm_df[
                ((holm_df["model_a"] == model_name) & (holm_df["model_b"] == best_model)) |
                ((holm_df["model_a"] == best_model) & (holm_df["model_b"] == model_name))
            ]
            if match.empty:
                differs_from_best = False
                holm_p_vs_best = np.nan
            else:
                holm_p_vs_best = float(match.iloc[0]["holm_adjusted_p_value"])
                differs_from_best = bool(match.iloc[0]["significant_after_holm"])

        summary_rows.append({
            "model": model_name,
            "accuracy": row["accuracy"],
            "wilson_ci_lower": row["wilson_ci_lower"],
            "wilson_ci_upper": row["wilson_ci_upper"],
            "is_best_model": model_name == best_model,
            "holm_adjusted_p_vs_best": holm_p_vs_best,
            "significantly_worse_than_best": differs_from_best,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.attrs["cochran_q_p_value"] = cochran_result.get("p_value", np.nan)
    summary_df.attrs["cochran_q_conclusion"] = cochran_result.get("conclusion", "")
    return summary_df


# ==============================================================================
# SECTION 8: FIGURES
# ==============================================================================

def plot_wilson_forest(wilson_df: pd.DataFrame, out_path: str):
    plot_df = wilson_df.sort_values("accuracy", ascending=True).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.7 * len(plot_df) + 2)))
    y_pos = np.arange(len(plot_df))

    lower_err = plot_df["accuracy"] - plot_df["wilson_ci_lower"]
    upper_err = plot_df["wilson_ci_upper"] - plot_df["accuracy"]

    ax.errorbar(plot_df["accuracy"], y_pos, xerr=[lower_err, upper_err],
               fmt="o", color="#2c7fb8", ecolor="#2c7fb8", elinewidth=2,
               capsize=5, markersize=8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["model"])
    ax.set_xlabel("Accuracy (with Wilson 95% Confidence Interval)")
    ax.set_title("Wilson 95% Confidence Intervals — Model Accuracy — PAMAP2")
    ax.set_xlim(0, 1.05)
    ax.grid(True, axis="x", alpha=0.4)
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_pvalue_heatmap(matrix: pd.DataFrame, title: str, out_path: str, annotate_stars: bool = False):
    fig, ax = plt.subplots(figsize=(9, 7.5))
    mask = matrix.isna()
    sns.heatmap(matrix.astype(float), annot=True, fmt=".3f", cmap="viridis_r",
               vmin=0, vmax=1, mask=mask, ax=ax, cbar_kws={"label": "p-value"},
               linewidths=0.5, linecolor="white")
    ax.set_title(title)
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_significance_heatmap(pairwise_holm_df: pd.DataFrame, models: list, out_path: str):
    sig_matrix = pd.DataFrame(0, index=models, columns=models, dtype=float)
    for _, row in pairwise_holm_df.iterrows():
        val = 1.0 if row["significant_after_holm"] else 0.0
        sig_matrix.loc[row["model_a"], row["model_b"]] = val
        sig_matrix.loc[row["model_b"], row["model_a"]] = val
    for m in models:
        sig_matrix.loc[m, m] = np.nan

    fig, ax = plt.subplots(figsize=(9, 7.5))
    mask = sig_matrix.isna()
    cmap = sns.color_palette(["#d9d9d9", "#d62728"])
    sns.heatmap(sig_matrix, cmap=cmap, cbar=False, mask=mask, ax=ax,
               linewidths=0.5, linecolor="white", annot=sig_matrix.replace({0: "n.s.", 1: "sig."}),
               fmt="")
    ax.set_title(f"Pairwise Significance After Holm-Bonferroni Correction (alpha={ALPHA})\nPAMAP2")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


def plot_accuracy_with_significance(wilson_df: pd.DataFrame, summary_df: pd.DataFrame, out_path: str):
    plot_df = wilson_df.sort_values("accuracy", ascending=False).reset_index(drop=True)
    sig_lookup = summary_df.set_index("model")["significantly_worse_than_best"].to_dict()
    best_lookup = summary_df.set_index("model")["is_best_model"].to_dict()

    fig, ax = plt.subplots(figsize=(11, 7))
    colors = ["#2ca02c" if best_lookup.get(m, False) else
              ("#d62728" if sig_lookup.get(m, False) else "#1f77b4")
              for m in plot_df["model"]]

    lower_err = plot_df["accuracy"] - plot_df["wilson_ci_lower"]
    upper_err = plot_df["wilson_ci_upper"] - plot_df["accuracy"]
    bars = ax.bar(plot_df["model"], plot_df["accuracy"], color=colors,
                  yerr=[lower_err, upper_err], capsize=5, edgecolor="black", linewidth=0.6)

    for i, m in enumerate(plot_df["model"]):
        if sig_lookup.get(m, False):
            ax.text(i, plot_df["wilson_ci_upper"].iloc[i] + 0.02, "*",
                   ha="center", fontsize=18, fontweight="bold")

    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Accuracy (Wilson 95% CI)")
    ax.set_xlabel("Model")
    ax.set_title("Model Accuracy with Statistical Significance vs. Best Model — PAMAP2\n"
                 "(green = best model, * = significantly worse after Holm correction)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI)
    plt.close(fig)


# ==============================================================================
# SECTION 9: REPORT GENERATION
# ==============================================================================

def generate_report(wilson_df, cochran_result, pairwise_df, holm_df, summary_df,
                     excluded_models) -> str:
    lines = []
    lines.append("# Module 4 Report — Statistical Validation")
    lines.append("## Dataset: PAMAP2 Physical Activity Monitoring\n")

    lines.append("## 1. Scope and Constraints")
    lines.append("- This module performs **no retraining**, **no re-preprocessing**, and does "
                 "not re-run Module 2 (TinyML) or Module 3 (Robustness).")
    lines.append("- The held-out test split is Module 1's exact split, deterministically "
                 f"reconstructed from the saved feature matrix (`random_state={RANDOM_STATE}`, "
                 f"`test_size={TEST_SIZE}`, stratified) — identical to the split used in Module 3.")
    lines.append("- All statistical tests operate on this single, fixed test set so that every "
                 "model's predictions are directly comparable sample-for-sample.")
    if excluded_models:
        lines.append(f"- Excluded from this analysis (no saved model artifact found): "
                     f"{', '.join(excluded_models)}.")
    lines.append("")

    lines.append("## 2. Wilson 95% Confidence Intervals")
    lines.append(wilson_df[["model", "n_test_samples", "n_correct", "accuracy",
                            "wilson_ci_lower", "wilson_ci_upper"]].round(4).to_markdown(index=False))
    lines.append("")

    lines.append("## 3. Cochran's Q Test (Omnibus)")
    lines.append(f"- k (models compared): {cochran_result['k_models']}")
    lines.append(f"- N (test samples): {cochran_result['n_samples']}")
    lines.append(f"- Q statistic: {cochran_result['Q_statistic']}")
    lines.append(f"- Degrees of freedom: {cochran_result['df']}")
    lines.append(f"- p-value: {cochran_result['p_value']}")
    lines.append(f"- **Conclusion:** {cochran_result['conclusion']}")
    lines.append("")

    lines.append("## 4. Pairwise McNemar Tests")
    lines.append(f"Exact binomial test used when discordant pairs < {MCNEMAR_EXACT_THRESHOLD}; "
                 "continuity-corrected chi-square approximation otherwise.")
    lines.append("")
    lines.append(pairwise_df[["model_a", "model_b", "n10", "n01", "n_discordant",
                              "p_value", "method"]].round(4).to_markdown(index=False))
    lines.append("")

    lines.append("## 5. Holm-Bonferroni Corrected Significance")
    lines.append(holm_df[["model_a", "model_b", "p_value", "holm_adjusted_p_value",
                          "significant_after_holm"]].round(4).to_markdown(index=False))
    lines.append("")

    lines.append("## 6. Statistical Summary")
    lines.append(summary_df[["model", "accuracy", "wilson_ci_lower", "wilson_ci_upper",
                             "is_best_model", "holm_adjusted_p_vs_best",
                             "significantly_worse_than_best"]].round(4).to_markdown(index=False))
    lines.append("")

    lines.append("## 7. Artifacts Produced")
    lines.append("- `results/tables/wilson_confidence_intervals.csv`")
    lines.append("- `results/tables/cochran_q_results.csv`")
    lines.append("- `results/tables/mcnemar_pairwise_results.csv`")
    lines.append("- `results/tables/holm_adjusted_results.csv`")
    lines.append("- `results/tables/statistical_summary.csv`")
    lines.append("- `results/figures/wilson_ci_forest_plot.png`")
    lines.append("- `results/figures/mcnemar_pvalue_heatmap.png`")
    lines.append("- `results/figures/holm_significance_heatmap.png`")
    lines.append("- `results/figures/accuracy_with_significance.png`")
    lines.append("(all figures saved at 300 dpi)")
    lines.append("")

    lines.append("## 8. Scope Boundaries of Module 4")
    lines.append("This module deliberately excludes TinyML deployment profiling (Module 2) and "
                 "robustness/perturbation testing (Module 3). All comparisons are made on Module "
                 "1's original clean held-out test set only.")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)
    print(f"[report] Saved report to {report_path}")
    return report_path


# ==============================================================================
# SECTION 10: MAIN ORCHESTRATION
# ==============================================================================

def main():
    print("=" * 50)
    print("MODULE 4 START — Statistical Validation (PAMAP2)")
    print("=" * 50)

    verify_module1_outputs_exist()
    X_test, y_test, classes = load_feature_dataset_and_reconstruct_split()

    loaded_models = load_trained_models(MODEL_DISPLAY_ORDER)
    excluded_models = [m for m in MODEL_DISPLAY_ORDER if m not in loaded_models]

    if len(loaded_models) < 2:
        print("[module4] WARNING: fewer than 2 models available. Pairwise tests and "
              "Cochran's Q require at least 2 models; results will be degenerate/empty.")

    correctness_df = get_correctness_matrix(loaded_models, X_test, y_test)
    print(f"[module4] Built correctness matrix: {correctness_df.shape[0]} samples x "
          f"{correctness_df.shape[1]} models.")

    # --- 1) Wilson 95% CIs ---
    wilson_df = compute_wilson_table(correctness_df)
    wilson_path = os.path.join(TABLES_DIR, "wilson_confidence_intervals.csv")
    wilson_df.to_csv(wilson_path, index=False)
    print(f"\n[module4] Saved Wilson confidence intervals to {wilson_path}")
    print(wilson_df[["model", "accuracy", "wilson_ci_lower", "wilson_ci_upper"]].to_string(index=False))

    # --- 2) Cochran's Q ---
    cochran_result = cochrans_q_test(correctness_df)
    cochran_df = pd.DataFrame([cochran_result])
    cochran_path = os.path.join(TABLES_DIR, "cochran_q_results.csv")
    cochran_df.to_csv(cochran_path, index=False)
    print(f"\n[module4] Cochran's Q: {cochran_result}")
    print(f"[module4] Saved Cochran's Q results to {cochran_path}")

    # --- 3) Pairwise McNemar ---
    pairwise_df = compute_pairwise_mcnemar(correctness_df)
    mcnemar_path = os.path.join(TABLES_DIR, "mcnemar_pairwise_results.csv")
    pairwise_df.to_csv(mcnemar_path, index=False)
    print(f"\n[module4] Saved pairwise McNemar results to {mcnemar_path}")

    # --- 4) Holm-Bonferroni correction ---
    if not pairwise_df.empty:
        holm_df = compute_holm_adjusted_table(pairwise_df)
    else:
        holm_df = pairwise_df.copy()
        holm_df["holm_adjusted_p_value"] = []
        holm_df["significant_after_holm"] = []
    holm_path = os.path.join(TABLES_DIR, "holm_adjusted_results.csv")
    holm_df.to_csv(holm_path, index=False)
    print(f"[module4] Saved Holm-Bonferroni adjusted results to {holm_path}")

    # --- 5) Statistical summary ---
    summary_df = compute_statistical_summary(wilson_df, holm_df, cochran_result)
    summary_path = os.path.join(TABLES_DIR, "statistical_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"[module4] Saved statistical summary to {summary_path}")

    # --- Figures ---
    models = list(correctness_df.columns)
    plot_wilson_forest(wilson_df, os.path.join(FIGURES_DIR, "wilson_ci_forest_plot.png"))

    if not pairwise_df.empty:
        pvalue_matrix = build_pvalue_matrix(pairwise_df, models, "p_value")
        plot_pvalue_heatmap(pvalue_matrix, "Pairwise McNemar Test p-values — PAMAP2",
                            os.path.join(FIGURES_DIR, "mcnemar_pvalue_heatmap.png"))
        plot_significance_heatmap(holm_df, models,
                                  os.path.join(FIGURES_DIR, "holm_significance_heatmap.png"))

    plot_accuracy_with_significance(wilson_df, summary_df,
                                    os.path.join(FIGURES_DIR, "accuracy_with_significance.png"))
    print("[module4] Saved all figures at 300 dpi.")

    # --- Report ---
    generate_report(wilson_df, cochran_result, pairwise_df, holm_df, summary_df, excluded_models)

    print("\n" + "=" * 50)
    print("MODULE 4 COMPLETE")
    print("=" * 50)


if __name__ == "__main__":
    main()

MODULE 4 START — Statistical Validation (PAMAP2)
[module4] Verified Module 1 outputs are present. Proceeding read-only.
[module4] Reconstructed Module 1's held-out test split: 1487 test windows, 475 features.
[module4] Successfully loaded 7 / 7 trained models.
[module4] Built correctness matrix: 1487 samples x 7 models.

[module4] Saved Wilson confidence intervals to C:\Users\Rohan Jaiswal\pamap2_statistics\results\tables\wilson_confidence_intervals.csv
             model  accuracy  wilson_ci_lower  wilson_ci_upper
           XGBoost  0.977808         0.968998         0.984155
         LinearSVM  0.973100         0.963578         0.980184
      RandomForest  0.971083         0.961276         0.978461
               KNN  0.967048         0.956703         0.974985
LogisticRegression  0.965703         0.955187         0.973819
      DecisionTree  0.936113         0.922525         0.947454
        GaussianNB  0.903161         0.887074         0.917170

[module4] Cochran's Q: {'k_models': 7

In [3]:
# ============================================================================
# MHEALTH DATASET — MODULE 1
# Dataset Preparation, Feature Engineering, Model Training & Performance Evaluation
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# STANDARDIZED DEFAULTS (fixed for ALL datasets: UCI HAR, WISDM, PAMAP2, MHEALTH, MotionSense)
# --------------------------------------------------------------------------
# - Windowing: dataset-native sampling rate, 50% overlap. For MHEALTH (50 Hz),
#   we use a 5-second window (250 samples, step = 125 samples), consistent
#   with the original MHEALTH literature (Banos et al., 2014).
# - Null/no-activity class (label 0) is EXCLUDED prior to windowing.
# - Windows are generated PER SUBJECT, PER CONTIGUOUS ACTIVITY BLOCK, so a
#   window never mixes two subjects or two activities.
# - Feature engineering (identical methodology for every dataset):
#     * Time-domain:  mean, std, min, max, range, median, variance, RMS, IQR,
#                     zero-crossing-rate  (10 features per channel)
#     * Frequency-domain: spectral energy, spectral entropy, dominant
#                     frequency, spectral centroid  (4 features per channel)
#     * Cross-axis/cross-sensor correlation features for tri-axial sensor
#       groups (Pearson r between x-y, y-z, x-z) + ECG lead-pair correlation
# - Train/test strategy: stratified 80/20 split (random_state=42).
#   MODEL SELECTION: GridSearchCV (5-fold Stratified CV) over a documented
#   hyperparameter grid per classifier, run on the training set. The
#   best_estimator_ from each grid search is then:
#     (a) evaluated once on the held-out test set, and
#     (b) re-evaluated via a fresh 5-fold Stratified CV (accuracy) to report
#         CV mean/std for the selected hyperparameters.
# - Metrics: Accuracy, Precision (macro), Recall (macro), Macro-F1,
#   Confusion Matrix, full Classification Report, 5-fold CV accuracy (mean±std)
# - Figures: 300 dpi, saved as PNG.
# - Folder convention: ./mhealth_outputs/{raw_data,features,models,results,figures,reports}/
# - ENCODING: All text file writes and CSV exports use encoding="utf-8"
#   explicitly, to avoid platform-dependent default-encoding errors (e.g.
#   cp1252 on Windows failing on characters like arrows, sigma, etc.)
# ============================================================================

import os
import sys
import json
import time
import zipfile
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib

warnings.filterwarnings("ignore")

# Ensure xgboost is available
try:
    from xgboost import XGBClassifier
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "--quiet"])
    from xgboost import XGBClassifier

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME   = "MHEALTH"
BASE_DIR       = Path("./mhealth_outputs")
RAW_DIR        = BASE_DIR / "raw_data"
FEATURES_DIR   = BASE_DIR / "features"
MODELS_DIR     = BASE_DIR / "models"
RESULTS_DIR    = BASE_DIR / "results"
FIGURES_DIR    = BASE_DIR / "figures"
REPORTS_DIR    = BASE_DIR / "reports"

for d in [RAW_DIR, FEATURES_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLING_RATE   = 50           # Hz, official MHEALTH sampling rate
WINDOW_SECONDS  = 5             # standardized default (documented above)
WINDOW_SIZE     = SAMPLING_RATE * WINDOW_SECONDS   # 250 samples
STEP_SIZE       = WINDOW_SIZE // 2                  # 50% overlap -> 125 samples

TEST_SIZE       = 0.2
RANDOM_STATE    = 42
CV_FOLDS        = 5

FIGURE_DPI      = 300
ENCODING        = "utf-8"

ACTIVITY_MAP = {
    1: "Standing still", 2: "Sitting and relaxing", 3: "Lying down",
    4: "Walking", 5: "Climbing stairs", 6: "Waist bends forward",
    7: "Frontal elevation of arms", 8: "Knees bending (crouching)",
    9: "Cycling", 10: "Jogging", 11: "Running", 12: "Jump front & back"
}
NULL_LABEL = 0

COLUMN_NAMES = [
    "chest_acc_x", "chest_acc_y", "chest_acc_z",
    "ecg_1", "ecg_2",
    "ankle_acc_x", "ankle_acc_y", "ankle_acc_z",
    "ankle_gyro_x", "ankle_gyro_y", "ankle_gyro_z",
    "ankle_mag_x", "ankle_mag_y", "ankle_mag_z",
    "arm_acc_x", "arm_acc_y", "arm_acc_z",
    "arm_gyro_x", "arm_gyro_y", "arm_gyro_z",
    "arm_mag_x", "arm_mag_y", "arm_mag_z",
    "label"
]
SENSOR_CHANNELS = [c for c in COLUMN_NAMES if c != "label"]

TRIAXIAL_GROUPS = {
    "chest_acc": ["chest_acc_x", "chest_acc_y", "chest_acc_z"],
    "ankle_acc": ["ankle_acc_x", "ankle_acc_y", "ankle_acc_z"],
    "ankle_gyro": ["ankle_gyro_x", "ankle_gyro_y", "ankle_gyro_z"],
    "ankle_mag": ["ankle_mag_x", "ankle_mag_y", "ankle_mag_z"],
    "arm_acc": ["arm_acc_x", "arm_acc_y", "arm_acc_z"],
    "arm_gyro": ["arm_gyro_x", "arm_gyro_y", "arm_gyro_z"],
    "arm_mag": ["arm_mag_x", "arm_mag_y", "arm_mag_z"],
}
ECG_PAIR = ["ecg_1", "ecg_2"]

print(f"[CONFIG] {DATASET_NAME} | window={WINDOW_SECONDS}s ({WINDOW_SIZE} samples) "
      f"| step={STEP_SIZE} samples (50% overlap) | fs={SAMPLING_RATE}Hz")

# ============================================================================
# 2. DOWNLOAD & EXTRACT OFFICIAL DATASET
# ============================================================================
def download_mhealth(raw_dir: Path) -> Path:
    subject_files_exist = list(raw_dir.rglob("mHealth_subject*.log"))
    if subject_files_exist:
        print(f"[DATA] Found {len(subject_files_exist)} existing subject log files. Skipping download.")
        return raw_dir

    urls = [
        "https://archive.ics.uci.edu/static/public/319/mhealth+dataset.zip",
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00319/MHEALTHDATASET.zip",
    ]
    zip_path = raw_dir / "mhealth.zip"
    downloaded = False
    for url in urls:
        try:
            print(f"[DATA] Attempting download from: {url}")
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                f.write(r.content)
            downloaded = True
            print("[DATA] Download successful.")
            break
        except Exception as e:
            print(f"[DATA] Failed: {e}")

    if not downloaded:
        raise RuntimeError(
            "Could not automatically download the official MHEALTH dataset. "
            "Please manually download it from "
            "https://archive.ics.uci.edu/dataset/319/mhealth+dataset and place the "
            "extracted 'mHealth_subject*.log' files into ./mhealth_outputs/raw_data/"
        )

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(raw_dir)
    print(f"[DATA] Extracted to {raw_dir}")
    return raw_dir


raw_path = download_mhealth(RAW_DIR)
subject_logs = sorted(raw_path.rglob("mHealth_subject*.log"))
if len(subject_logs) == 0:
    raise RuntimeError("No MHEALTH subject log files found after extraction.")
print(f"[DATA] Located {len(subject_logs)} subject files.")

# ============================================================================
# 3. LOAD RAW DATA (PER SUBJECT)
# ============================================================================
all_subjects = {}
for f in subject_logs:
    subj_id = f.stem.replace("mHealth_subject", "")
    df = pd.read_csv(f, sep=r"\s+", header=None, names=COLUMN_NAMES)
    df["subject"] = int(subj_id)
    all_subjects[int(subj_id)] = df
    print(f"[DATA] Subject {subj_id}: {len(df)} samples")

raw_df = pd.concat(all_subjects.values(), ignore_index=True)
raw_df = raw_df[raw_df["label"] != NULL_LABEL].reset_index(drop=True)
print(f"[DATA] Total samples after removing null class: {len(raw_df)}")

# ============================================================================
# 4. FEATURE ENGINEERING FUNCTIONS
# ============================================================================
def time_domain_features(sig: np.ndarray) -> dict:
    mean = np.mean(sig)
    std = np.std(sig)
    mn = np.min(sig)
    mx = np.max(sig)
    med = np.median(sig)
    var = np.var(sig)
    rms = np.sqrt(np.mean(sig ** 2))
    q75, q25 = np.percentile(sig, [75, 25])
    iqr = q75 - q25
    zcr = np.sum(np.diff(np.sign(sig)) != 0) / len(sig)
    return {
        "mean": mean, "std": std, "min": mn, "max": mx, "range": mx - mn,
        "median": med, "var": var, "rms": rms, "iqr": iqr, "zcr": zcr
    }

def freq_domain_features(sig: np.ndarray, fs: int) -> dict:
    n = len(sig)
    fft_vals = np.abs(rfft(sig - np.mean(sig)))
    freqs = rfftfreq(n, d=1.0 / fs)
    power = fft_vals ** 2
    total_power = np.sum(power) + 1e-12

    energy = np.sum(power) / n
    prob = power / total_power
    entropy = -np.sum(prob * np.log2(prob + 1e-12))

    if len(freqs) > 1:
        dom_idx = np.argmax(fft_vals[1:]) + 1
        dominant_freq = freqs[dom_idx]
    else:
        dominant_freq = 0.0

    centroid = np.sum(freqs * power) / total_power

    return {
        "spec_energy": energy, "spec_entropy": entropy,
        "dominant_freq": dominant_freq, "spec_centroid": centroid
    }

def correlation_features(window_df: pd.DataFrame) -> dict:
    feats = {}
    for group_name, cols in TRIAXIAL_GROUPS.items():
        x, y, z = [window_df[c].values for c in cols]
        feats[f"{group_name}_corr_xy"] = safe_corr(x, y)
        feats[f"{group_name}_corr_yz"] = safe_corr(y, z)
        feats[f"{group_name}_corr_xz"] = safe_corr(x, z)
    e1, e2 = [window_df[c].values for c in ECG_PAIR]
    feats["ecg_corr"] = safe_corr(e1, e2)
    return feats

def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-9 or np.std(b) < 1e-9:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else r

def extract_window_features(window_df: pd.DataFrame) -> dict:
    feats = {}
    for ch in SENSOR_CHANNELS:
        sig = window_df[ch].values.astype(float)
        td = time_domain_features(sig)
        fd = freq_domain_features(sig, SAMPLING_RATE)
        for k, v in td.items():
            feats[f"{ch}_{k}"] = v
        for k, v in fd.items():
            feats[f"{ch}_{k}"] = v
    feats.update(correlation_features(window_df))
    return feats

# ============================================================================
# 5. SLIDING WINDOW SEGMENTATION (per subject, per contiguous activity block)
# ============================================================================
def segment_windows(df: pd.DataFrame) -> list:
    windows = []
    for subj_id, subj_df in df.groupby("subject"):
        subj_df = subj_df.reset_index(drop=True)
        block_id = (subj_df["label"] != subj_df["label"].shift()).cumsum()
        for _, block in subj_df.groupby(block_id):
            label = block["label"].iloc[0]
            n = len(block)
            if n < WINDOW_SIZE:
                continue
            start = 0
            while start + WINDOW_SIZE <= n:
                win = block.iloc[start:start + WINDOW_SIZE]
                windows.append((win, label, subj_id))
                start += STEP_SIZE
    return windows

print("[SEGMENT] Generating sliding windows...")
windows = segment_windows(raw_df)
print(f"[SEGMENT] Total windows generated: {len(windows)}")

# ============================================================================
# 6. BUILD FEATURE MATRIX
# ============================================================================
print("[FEATURES] Extracting features from each window (this may take a few minutes)...")
feature_rows = []
labels = []
subjects = []
t0 = time.time()
for i, (win_df, label, subj_id) in enumerate(windows):
    feats = extract_window_features(win_df)
    feature_rows.append(feats)
    labels.append(label)
    subjects.append(subj_id)
    if (i + 1) % 500 == 0:
        print(f"  ...{i+1}/{len(windows)} windows processed")

feature_df = pd.DataFrame(feature_rows)
feature_df["label"] = labels
feature_df["activity_name"] = feature_df["label"].map(ACTIVITY_MAP)
feature_df["subject"] = subjects
print(f"[FEATURES] Extraction complete in {time.time()-t0:.1f}s. "
      f"Feature matrix shape: {feature_df.shape}")

feature_csv_path = FEATURES_DIR / "mhealth_features.csv"
feature_df.to_csv(feature_csv_path, index=False, encoding=ENCODING)
with open(FEATURES_DIR / "mhealth_feature_names.json", "w", encoding=ENCODING) as f:
    json.dump([c for c in feature_df.columns if c not in ["label", "activity_name", "subject"]], f, indent=2)
print(f"[FEATURES] Saved to {feature_csv_path}")

# ============================================================================
# 7. TRAIN/TEST SPLIT + SCALING
# ============================================================================
feature_cols = [c for c in feature_df.columns if c not in ["label", "activity_name", "subject"]]
X = feature_df[feature_cols].values
y_raw = feature_df["label"].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"[SPLIT] Train: {X_train.shape}, Test: {X_test.shape}")

np.savez(RESULTS_DIR / "mhealth_train_test_data.npz",
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train, y_test=y_test)
joblib.dump(scaler, MODELS_DIR / "mhealth_scaler.pkl")
joblib.dump(le, MODELS_DIR / "mhealth_label_encoder.pkl")

# ============================================================================
# 8. DEFINE MODELS + HYPERPARAMETER GRIDS (GridSearchCV, identical across datasets)
# ============================================================================
base_models = {
    "Logistic_Regression": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    "Decision_Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random_Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "KNN": KNeighborsClassifier(),
    "Gaussian_NB": GaussianNB(),
    "Linear_SVM": LinearSVC(random_state=RANDOM_STATE, max_iter=5000),
    "XGBoost": XGBClassifier(
        use_label_encoder=False, eval_metric="mlogloss",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

param_grids = {
    "Logistic_Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs"],
    },
    "Decision_Tree": {
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "criterion": ["gini", "entropy"],
    },
    "Random_Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5],
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    },
    "Gaussian_NB": {
        "var_smoothing": [1e-9, 1e-8, 1e-7],
    },
    "Linear_SVM": {
        "C": [0.01, 0.1, 1, 10],
    },
    "XGBoost": {
        "n_estimators": [100, 200],
        "max_depth": [3, 6, 9],
        "learning_rate": [0.05, 0.1, 0.2],
    },
}

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# ============================================================================
# 9. GRIDSEARCHCV, BEST-ESTIMATOR TEST EVALUATION, POST-HOC 5-FOLD CV
# ============================================================================
all_metrics = {}
all_reports = {}
predictions_store = {}
best_params_store = {}

for name, base_model in base_models.items():
    print(f"\n[MODEL] GridSearchCV for {name}...")
    t0 = time.time()

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grids[name],
        cv=skf,
        scoring="accuracy",
        n_jobs=-1,
        refit=True,
    )
    grid.fit(X_train_scaled, y_train)
    grid_search_time = time.time() - t0

    best_model = grid.best_estimator_
    best_params_store[name] = grid.best_params_
    print(f"  Best params: {grid.best_params_} (grid search took {grid_search_time:.1f}s)")

    t0 = time.time()
    best_model.fit(X_train_scaled, y_train)
    train_time = time.time() - t0

    y_pred = best_model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=skf, scoring="accuracy", n_jobs=-1)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(
        y_test, y_pred, target_names=[str(le.inverse_transform([c])[0]) for c in sorted(set(y_test))],
        output_dict=True, zero_division=0
    )

    all_metrics[name] = {
        "accuracy": acc, "precision_macro": prec, "recall_macro": rec,
        "f1_macro": f1, "cv_accuracy_mean": cv_scores.mean(),
        "cv_accuracy_std": cv_scores.std(), "train_time_sec": train_time,
        "grid_search_time_sec": grid_search_time, "best_params": grid.best_params_
    }
    all_reports[name] = report
    predictions_store[name] = y_pred

    joblib.dump(best_model, MODELS_DIR / f"mhealth_{name}.pkl")

    plt.figure(figsize=(10, 8))
    activity_labels = [ACTIVITY_MAP[c] for c in le.inverse_transform(sorted(set(y_test)))]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=activity_labels, yticklabels=activity_labels)
    plt.title(f"MHEALTH — Confusion Matrix — {name} (GridSearchCV best estimator)")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"mhealth_confusion_matrix_{name}.png", dpi=FIGURE_DPI)
    plt.close()

    print(f"  Test: Accuracy={acc:.4f} | Macro-F1={f1:.4f} | "
          f"CV Acc={cv_scores.mean():.4f}±{cv_scores.std():.4f} | Train time={train_time:.2f}s")

with open(MODELS_DIR / "mhealth_best_hyperparameters.json", "w", encoding=ENCODING) as f:
    json.dump(best_params_store, f, indent=2, default=str)

# ============================================================================
# 10. SAVE METRICS + PREDICTIONS
# ============================================================================
with open(RESULTS_DIR / "mhealth_metrics.json", "w", encoding=ENCODING) as f:
    json.dump(all_metrics, f, indent=2, default=str)

with open(RESULTS_DIR / "mhealth_classification_reports.json", "w", encoding=ENCODING) as f:
    json.dump(all_reports, f, indent=2)

pred_df = pd.DataFrame(predictions_store)
pred_df["y_true"] = y_test
pred_df.to_csv(RESULTS_DIR / "mhealth_test_predictions.csv", index=False, encoding=ENCODING)

metrics_df = pd.DataFrame(all_metrics).T
metrics_df.to_csv(RESULTS_DIR / "mhealth_metrics_summary.csv", encoding=ENCODING)

# ============================================================================
# 11. MODEL COMPARISON FIGURE
# ============================================================================
plt.figure(figsize=(12, 6))
metrics_df_sorted = metrics_df.sort_values("accuracy", ascending=False)
x = np.arange(len(metrics_df_sorted))
width = 0.35
plt.bar(x - width/2, metrics_df_sorted["accuracy"].astype(float), width, label="Test Accuracy")
plt.bar(x + width/2, metrics_df_sorted["f1_macro"].astype(float), width, label="Macro F1")
plt.xticks(x, metrics_df_sorted.index, rotation=45, ha="right")
plt.ylabel("Score")
plt.title("MHEALTH — Model Comparison (Accuracy vs Macro-F1, GridSearchCV best estimators)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_model_comparison.png", dpi=FIGURE_DPI)
plt.close()

# ============================================================================
# 12. GENERATE MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MHEALTH Dataset — Module 1 Report",
    "## Dataset Preparation, Feature Engineering, Model Training & Evaluation\n",
    f"- Total windows: {len(windows)}",
    f"- Window size: {WINDOW_SECONDS}s ({WINDOW_SIZE} samples @ {SAMPLING_RATE}Hz), 50% overlap",
    f"- Feature count: {len(feature_cols)}",
    f"- Train/Test split: {1-TEST_SIZE:.0%}/{TEST_SIZE:.0%} stratified, random_state={RANDOM_STATE}",
    f"- Model selection: GridSearchCV with {CV_FOLDS}-fold Stratified CV per classifier",
    f"- Post-selection evaluation: best_estimator_ scored once on held-out test set, "
    f"then re-validated via fresh {CV_FOLDS}-fold Stratified CV\n",
    "## Best Hyperparameters per Model\n",
    json.dumps(best_params_store, indent=2, default=str),
    "\n## Model Performance Summary\n",
    metrics_df_sorted.drop(columns=["best_params"], errors="ignore").to_markdown(),
]
with open(REPORTS_DIR / "mhealth_module1_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print("\n" + "=" * 70)
print("MODULE 1 (MHEALTH) COMPLETE — GridSearchCV methodology applied")
print("=" * 70)
print(metrics_df_sorted[["accuracy", "f1_macro", "cv_accuracy_mean"]])
print(f"\nAll artifacts saved under: {BASE_DIR.resolve()}")

[CONFIG] MHEALTH | window=5s (250 samples) | step=125 samples (50% overlap) | fs=50Hz
[DATA] Found 10 existing subject log files. Skipping download.
[DATA] Located 10 subject files.
[DATA] Subject 1: 161280 samples
[DATA] Subject 10: 98304 samples
[DATA] Subject 2: 130561 samples
[DATA] Subject 3: 122112 samples
[DATA] Subject 4: 116736 samples
[DATA] Subject 5: 119808 samples
[DATA] Subject 6: 98304 samples
[DATA] Subject 7: 104448 samples
[DATA] Subject 8: 129024 samples
[DATA] Subject 9: 135168 samples
[DATA] Total samples after removing null class: 343195
[SEGMENT] Generating sliding windows...
[SEGMENT] Total windows generated: 2561
[FEATURES] Extracting features from each window (this may take a few minutes)...
  ...500/2561 windows processed
  ...1000/2561 windows processed
  ...1500/2561 windows processed
  ...2000/2561 windows processed
  ...2500/2561 windows processed
[FEATURES] Extraction complete in 34.1s. Feature matrix shape: (2561, 347)
[FEATURES] Saved to mhealth_output

In [4]:
# ============================================================================
# MHEALTH DATASET — MODULE 2
# TinyML-Oriented Deployment Analysis
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# READ-ONLY with respect to Module 1. This script:
#   - Loads trained models, scaler, label encoder, feature/split data, and
#     metrics produced by Module 1 from ./mhealth_outputs/
#   - Performs NO retraining, NO preprocessing, NO re-evaluation
#   - Writes ALL new outputs to ./mhealth_tinyml/{results,figures,reports}/
#
# STANDARDIZED TinyML METHODOLOGY (fixed for ALL datasets going forward)
# --------------------------------------------------------------------------
# Model Size (KB):      actual on-disk size of the joblib-serialized model
#                        (proxy for the exported/quantization-ready artifact)
# RAM Estimate (KB):     heuristic runtime working-memory footprint, computed
#                        per model family (see estimate_ram_flash()):
#                          - Logistic Regression / Linear SVM: coefficient +
#                            intercept matrix (n_classes x n_features), 4 bytes/value
#                          - Gaussian NB: per-class mean + variance + prior, 4 bytes/value
#                          - Decision Tree: node_count x 16 bytes/node
#                            (feature_idx, threshold, left_child, right_child)
#                          - Random Forest: sum of node_count x 16 bytes across all trees
#                          - XGBoost: sum of leaves/nodes across boosters x 16 bytes
#                          - KNN: entire training set stored (n_train x n_features x 4 bytes)
#                            — inherently RAM-heavy, reflecting real TinyML infeasibility
# Flash Estimate (KB):   on-disk serialized model size x 1.15 (10-15% packaging/
#                        alignment overhead), representing static storage in
#                        microcontroller flash/ROM
# Inference Latency:     mean wall-clock time (ms) to predict a single window,
#                        averaged over 200 held-out test samples, 5 repeats
# Training Time:         reused directly from Module 1 (metrics.json), no retraining
#
# Deployment Friendliness Score (0-100):
#   Weighted composite of five min-max normalized components (computed across
#   the 7 models of THIS dataset):
#     0.25 x (1 - norm(RAM))        # lower RAM is better
#     0.20 x (1 - norm(Flash))      # lower Flash is better
#     0.25 x (1 - norm(Latency))    # lower latency is better
#     0.15 x (1 - norm(ModelSize))  # lower size is better
#     0.15 x norm(TestAccuracy)     # higher accuracy is better
#   Score = 100 x weighted sum
#
# Deployment Tier (fixed thresholds, identical across datasets):
#   Tier 1 - Excellent   : Score >= 80  (fits low-end MCU, e.g. Arduino Uno/Nano, <=32KB RAM)
#   Tier 2 - Good        : 65 <= Score < 80  (mid-range MCU, e.g. ESP32/Cortex-M4, <=256KB RAM)
#   Tier 3 - Moderate    : 45 <= Score < 65  (Cortex-M7/Pico-class or constrained SBC)
#   Tier 4 - Poor        : Score < 45  (requires SBC/edge GPU, e.g. Raspberry Pi)
#
# Figures: 300 dpi PNG. Folder convention: ./mhealth_tinyml/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME     = "MHEALTH"

SOURCE_DIR       = Path("./mhealth_outputs")          # READ-ONLY
SOURCE_MODELS    = SOURCE_DIR / "models"
SOURCE_RESULTS   = SOURCE_DIR / "results"
SOURCE_FEATURES  = SOURCE_DIR / "features"

OUTPUT_DIR       = Path("./mhealth_tinyml")            # WRITE-ONLY here
RESULTS_DIR      = OUTPUT_DIR / "results"
FIGURES_DIR      = OUTPUT_DIR / "figures"
REPORTS_DIR      = OUTPUT_DIR / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FIGURE_DPI       = 300
ENCODING         = "utf-8"
LATENCY_SAMPLES  = 200     # number of test samples used per latency measurement
LATENCY_REPEATS  = 5       # repeats, mean latency reported

MODEL_NAMES = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

BYTES_PER_NODE = 16   # heuristic: feature_idx(4) + threshold(4) + left(4) + right(4)

print(f"[CONFIG] {DATASET_NAME} — Module 2 (TinyML Deployment Analysis)")
print(f"[CONFIG] Reading Module 1 artifacts from: {SOURCE_DIR.resolve()}")
print(f"[CONFIG] Writing Module 2 artifacts to:  {OUTPUT_DIR.resolve()}")

# ============================================================================
# 2. VALIDATE MODULE 1 ARTIFACTS EXIST
# ============================================================================
required_paths = [
    SOURCE_MODELS / "mhealth_scaler.pkl",
    SOURCE_MODELS / "mhealth_label_encoder.pkl",
    SOURCE_RESULTS / "mhealth_train_test_data.npz",
    SOURCE_RESULTS / "mhealth_metrics.json",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required Module 1 artifacts:\n" + "\n".join(missing) +
        "\nPlease run Module 1 for MHEALTH first."
    )

model_paths = {name: SOURCE_MODELS / f"mhealth_{name}.pkl" for name in MODEL_NAMES}
missing_models = [name for name, p in model_paths.items() if not p.exists()]
if missing_models:
    raise FileNotFoundError(f"Missing trained model files for: {missing_models}")

# ============================================================================
# 3. LOAD MODULE 1 ARTIFACTS (READ-ONLY)
# ============================================================================
scaler = joblib.load(SOURCE_MODELS / "mhealth_scaler.pkl")
label_encoder = joblib.load(SOURCE_MODELS / "mhealth_label_encoder.pkl")

split_data = np.load(SOURCE_RESULTS / "mhealth_train_test_data.npz")
X_train, X_test = split_data["X_train"], split_data["X_test"]
y_train, y_test = split_data["y_train"], split_data["y_test"]

with open(SOURCE_RESULTS / "mhealth_metrics.json", "r", encoding=ENCODING) as f:
    module1_metrics = json.load(f)

trained_models = {name: joblib.load(path) for name, path in model_paths.items()}

print(f"[LOAD] Loaded {len(trained_models)} trained models, scaler, label encoder, "
      f"and test set of shape {X_test.shape}")

# ============================================================================
# 4. HELPER FUNCTIONS
# ============================================================================
def model_disk_size_kb(model_path: Path) -> float:
    return os.path.getsize(model_path) / 1024.0


def estimate_ram_flash_kb(name: str, model, disk_size_kb: float,
                           n_features: int, n_train: int):
    """
    Heuristic RAM/Flash estimation per model family.
    RAM = runtime working-memory footprint required to hold the model's
          parameters in memory during inference (weights/tree structures/
          stored training data, as applicable).
    Flash = on-disk serialized size x 1.15 (packaging/alignment overhead),
            representing static storage footprint.
    """
    flash_kb = disk_size_kb * 1.15

    if name in ("Logistic_Regression", "Linear_SVM"):
        n_params = model.coef_.size + model.intercept_.size
        ram_kb = (n_params * 4) / 1024.0

    elif name == "Gaussian_NB":
        n_params = model.theta_.size + model.var_.size + model.class_prior_.size
        ram_kb = (n_params * 4) / 1024.0

    elif name == "Decision_Tree":
        n_nodes = model.tree_.node_count
        ram_kb = (n_nodes * BYTES_PER_NODE) / 1024.0

    elif name == "Random_Forest":
        total_nodes = sum(est.tree_.node_count for est in model.estimators_)
        ram_kb = (total_nodes * BYTES_PER_NODE) / 1024.0

    elif name == "XGBoost":
        try:
            df_trees = model.get_booster().trees_to_dataframe()
            total_nodes = len(df_trees)
        except Exception:
            total_nodes = model.n_estimators * (2 ** (model.max_depth + 1))
        ram_kb = (total_nodes * BYTES_PER_NODE) / 1024.0

    elif name == "KNN":
        ram_kb = (n_train * n_features * 4) / 1024.0
        flash_kb = ram_kb * 1.15

    else:
        ram_kb = disk_size_kb

    return ram_kb, flash_kb


def measure_latency_ms(model, X: np.ndarray, n_samples: int, repeats: int) -> float:
    """Mean per-sample inference latency (ms), averaged over repeats."""
    n_samples = min(n_samples, X.shape[0])
    idx = np.random.RandomState(42).choice(X.shape[0], size=n_samples, replace=False)
    X_sub = X[idx]

    per_repeat_times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for i in range(n_samples):
            model.predict(X_sub[i:i + 1])
        t1 = time.perf_counter()
        per_repeat_times.append((t1 - t0) / n_samples)

    return float(np.mean(per_repeat_times) * 1000.0)  # ms


def minmax_norm(series: pd.Series) -> pd.Series:
    rng = series.max() - series.min()
    if rng < 1e-12:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / rng


def assign_tier(score: float) -> str:
    if score >= 80:
        return "Tier 1 - Excellent"
    elif score >= 65:
        return "Tier 2 - Good"
    elif score >= 45:
        return "Tier 3 - Moderate"
    else:
        return "Tier 4 - Poor"

# ============================================================================
# 5. COMPUTE TINYML METRICS FOR EACH MODEL
# ============================================================================
n_features = X_test.shape[1]
n_train = X_train.shape[0]

records = []
print("\n[TINYML] Profiling models (size, RAM, flash, latency)...")
for name in MODEL_NAMES:
    model = trained_models[name]
    disk_kb = model_disk_size_kb(model_paths[name])
    ram_kb, flash_kb = estimate_ram_flash_kb(name, model, disk_kb, n_features, n_train)
    latency_ms = measure_latency_ms(model, X_test, LATENCY_SAMPLES, LATENCY_REPEATS)

    m1 = module1_metrics[name]
    records.append({
        "Model": name,
        "Test_Accuracy": m1["accuracy"],
        "Macro_F1": m1["f1_macro"],
        "Training_Time_sec": m1["train_time_sec"],
        "Model_Size_KB": disk_kb,
        "RAM_Estimate_KB": ram_kb,
        "Flash_Estimate_KB": flash_kb,
        "Inference_Latency_ms": latency_ms,
    })
    print(f"  {name:22s} | size={disk_kb:9.2f} KB | RAM={ram_kb:10.2f} KB | "
          f"flash={flash_kb:9.2f} KB | latency={latency_ms:7.4f} ms")

tinyml_df = pd.DataFrame(records).set_index("Model")

# ============================================================================
# 6. DEPLOYMENT FRIENDLINESS SCORE + TIER
# ============================================================================
norm_ram     = minmax_norm(tinyml_df["RAM_Estimate_KB"])
norm_flash   = minmax_norm(tinyml_df["Flash_Estimate_KB"])
norm_latency = minmax_norm(tinyml_df["Inference_Latency_ms"])
norm_size    = minmax_norm(tinyml_df["Model_Size_KB"])
norm_acc     = minmax_norm(tinyml_df["Test_Accuracy"])

tinyml_df["Deployment_Friendliness_Score"] = 100 * (
    0.25 * (1 - norm_ram) +
    0.20 * (1 - norm_flash) +
    0.25 * (1 - norm_latency) +
    0.15 * (1 - norm_size) +
    0.15 * norm_acc
)
tinyml_df["Deployment_Tier"] = tinyml_df["Deployment_Friendliness_Score"].apply(assign_tier)

ranking_df = tinyml_df.sort_values("Deployment_Friendliness_Score", ascending=False).copy()
ranking_df.insert(0, "TinyML_Rank", range(1, len(ranking_df) + 1))

print("\n[SCORE] Deployment Friendliness ranking (best -> worst):")
print(ranking_df[["TinyML_Rank", "Deployment_Friendliness_Score", "Deployment_Tier"]])

# ============================================================================
# 7. SAVE TABLES
# ============================================================================
tinyml_df.to_csv(RESULTS_DIR / "mhealth_tinyml_comparison.csv", encoding=ENCODING)
ranking_df.to_csv(RESULTS_DIR / "mhealth_tinyml_ranking.csv", encoding=ENCODING)

with open(RESULTS_DIR / "mhealth_tinyml_metrics.json", "w", encoding=ENCODING) as f:
    json.dump(tinyml_df.reset_index().to_dict(orient="records"), f, indent=2, default=str)

print(f"\n[SAVE] Tables written to {RESULTS_DIR}")

# ============================================================================
# 8. FIGURES (300 dpi)
# ============================================================================
plot_order = ranking_df.index.tolist()

def bar_plot(series_name, ylabel, title, filename, color="#4C72B0", log_scale=False):
    plt.figure(figsize=(11, 6))
    values = ranking_df.loc[plot_order, series_name]
    bars = plt.bar(plot_order, values, color=color)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    if log_scale:
        plt.yscale("log")
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                  f"{val:,.2f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=FIGURE_DPI)
    plt.close()

bar_plot("Model_Size_KB", "Size (KB)",
          "MHEALTH — Model Size by Classifier (sorted by deployment score)",
          "mhealth_tinyml_model_size.png", color="#4C72B0", log_scale=True)

bar_plot("RAM_Estimate_KB", "Estimated RAM (KB)",
          "MHEALTH — Estimated RAM Usage by Classifier",
          "mhealth_tinyml_ram_estimate.png", color="#DD8452", log_scale=True)

bar_plot("Flash_Estimate_KB", "Estimated Flash (KB)",
          "MHEALTH — Estimated Flash Usage by Classifier",
          "mhealth_tinyml_flash_estimate.png", color="#55A868", log_scale=True)

bar_plot("Inference_Latency_ms", "Latency (ms/sample)",
          "MHEALTH — Inference Latency by Classifier",
          "mhealth_tinyml_latency.png", color="#C44E52", log_scale=True)

bar_plot("Deployment_Friendliness_Score", "Score (0-100)",
          "MHEALTH — Deployment Friendliness Score by Classifier",
          "mhealth_tinyml_deployment_score.png", color="#8172B2")

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
panels = [
    ("Model_Size_KB", "Size (KB)", "Model Size", axes[0, 0], True),
    ("RAM_Estimate_KB", "RAM (KB)", "Estimated RAM", axes[0, 1], True),
    ("Inference_Latency_ms", "Latency (ms)", "Inference Latency", axes[1, 0], True),
    ("Deployment_Friendliness_Score", "Score", "Deployment Friendliness Score", axes[1, 1], False),
]
for col, ylabel, title, ax, logsc in panels:
    values = ranking_df.loc[plot_order, col]
    ax.bar(plot_order, values, color="#4C72B0")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    if logsc:
        ax.set_yscale("log")
plt.suptitle("MHEALTH — TinyML Deployment Analysis Summary", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_tinyml_summary_panel.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(8, 6))
tier_counts = ranking_df["Deployment_Tier"].value_counts().reindex(
    ["Tier 1 - Excellent", "Tier 2 - Good", "Tier 3 - Moderate", "Tier 4 - Poor"]
).fillna(0)
sns.barplot(x=tier_counts.index, y=tier_counts.values, palette="viridis")
plt.ylabel("Number of Models")
plt.title("MHEALTH — Deployment Tier Distribution")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_tinyml_tier_distribution.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 6 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 9. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MHEALTH Dataset — Module 2 Report",
    "## TinyML-Oriented Deployment Analysis\n",
    "This module reuses trained models and evaluation artifacts from Module 1 "
    "without any retraining, preprocessing, or re-evaluation.\n",
    "### Methodology Summary",
    "- **Model Size**: on-disk joblib serialized size (KB)",
    "- **RAM Estimate**: heuristic runtime memory footprint per model family "
    "(coefficients/tree nodes/stored training samples)",
    "- **Flash Estimate**: serialized size x 1.15 packaging overhead",
    f"- **Inference Latency**: mean per-sample prediction time (ms), "
    f"{LATENCY_SAMPLES} samples x {LATENCY_REPEATS} repeats",
    "- **Training Time**: reused directly from Module 1 (no retraining)",
    "- **Deployment Friendliness Score**: weighted composite (25% RAM, 20% Flash, "
    "25% Latency, 15% Size, 15% Accuracy), min-max normalized across this dataset's 7 models",
    "- **Deployment Tier**: Tier 1 (>=80) Excellent, Tier 2 (65-80) Good, "
    "Tier 3 (45-65) Moderate, Tier 4 (<45) Poor\n",
    "## TinyML Comparison Table\n",
    tinyml_df.round(4).to_markdown(),
    "\n## Deployment Ranking (Best to Worst)\n",
    ranking_df[[
        "TinyML_Rank", "Deployment_Friendliness_Score", "Deployment_Tier",
        "Model_Size_KB", "RAM_Estimate_KB", "Flash_Estimate_KB",
        "Inference_Latency_ms", "Test_Accuracy"
    ]].round(4).to_markdown(),
    "\n## Recommended Model for Constrained MCU Deployment\n",
    f"**{ranking_df.index[0]}** - Score: {ranking_df.iloc[0]['Deployment_Friendliness_Score']:.2f} "
    f"({ranking_df.iloc[0]['Deployment_Tier']})",
]
with open(REPORTS_DIR / "mhealth_module2_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'mhealth_module2_report.md'}")

# ============================================================================
# 10. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 2 COMPLETE")
print("=" * 50)

[CONFIG] MHEALTH — Module 2 (TinyML Deployment Analysis)
[CONFIG] Reading Module 1 artifacts from: C:\Users\Rohan Jaiswal\mhealth_outputs
[CONFIG] Writing Module 2 artifacts to:  C:\Users\Rohan Jaiswal\mhealth_tinyml
[LOAD] Loaded 7 trained models, scaler, label encoder, and test set of shape (513, 344)

[TINYML] Profiling models (size, RAM, flash, latency)...
  Logistic_Regression    | size=    33.26 KB | RAM=     16.17 KB | flash=    38.25 KB | latency= 0.1014 ms
  Decision_Tree          | size=     6.49 KB | RAM=      0.52 KB | flash=     7.47 KB | latency= 0.1231 ms
  Random_Forest          | size=   981.77 KB | RAM=     93.34 KB | flash=  1129.04 KB | latency=17.2495 ms
  KNN                    | size=  5520.86 KB | RAM=   2752.00 KB | flash=  3164.80 KB | latency= 0.9198 ms
  Gaussian_NB            | size=    65.49 KB | RAM=     32.30 KB | flash=    75.31 KB | latency= 0.3216 ms
  Linear_SVM             | size=    33.13 KB | RAM=     16.17 KB | flash=    38.10 KB | latency= 0.105

In [5]:
# ============================================================================
# MHEALTH DATASET — MODULE 3
# Robustness Analysis
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# READ-ONLY with respect to Module 1. This script:
#   - Loads trained models, scaler, label encoder, feature names, and test
#     split from ./mhealth_outputs/
#   - Performs NO retraining, NO preprocessing, NO re-segmentation
#   - Writes ALL new outputs to ./mhealth_robustness/{results,figures,reports}/
#
# STANDARDIZED ROBUSTNESS METHODOLOGY (fixed for ALL datasets going forward)
# --------------------------------------------------------------------------
# 1) GAUSSIAN NOISE INJECTION
#    X_noisy = X_test + N(0, sigma^2) x per-feature std(X_test)
#    Levels tested: sigma in {0.1, 0.2, 0.3}
#
# 2) FEATURE REMOVAL (missingness)
#    Random subset of feature COLUMNS zeroed (scaled mean).
#    Levels: 10%, 20%, 30% of features removed (random_state=42)
#
# 3) SENSOR PERTURBATION (simulated sensor failure)
#    All feature columns of one on-body sensor LOCATION zeroed.
#    Locations tested: chest (acc+ecg), ankle (acc+gyro+mag), arm (acc+gyro+mag)
#
# EVALUATION METRIC: Accuracy/F1 Retention % = (perturbed / clean) x 100
#
# OVERALL ROBUSTNESS SCORE: mean of Accuracy- and F1-Retention % averaged
# across all 9 perturbation conditions per model.
#
# Figures: 300 dpi PNG. Folder convention:
#    ./mhealth_robustness/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME     = "MHEALTH"

SOURCE_DIR       = Path("./mhealth_outputs")           # READ-ONLY
SOURCE_MODELS    = SOURCE_DIR / "models"
SOURCE_RESULTS   = SOURCE_DIR / "results"
SOURCE_FEATURES  = SOURCE_DIR / "features"

OUTPUT_DIR       = Path("./mhealth_robustness")        # WRITE-ONLY here
RESULTS_DIR      = OUTPUT_DIR / "results"
FIGURES_DIR      = OUTPUT_DIR / "figures"
REPORTS_DIR      = OUTPUT_DIR / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FIGURE_DPI       = 300
ENCODING         = "utf-8"
RANDOM_STATE     = 42

NOISE_LEVELS       = [0.1, 0.2, 0.3]
FEATURE_REMOVAL_PCTS = [0.10, 0.20, 0.30]
SENSOR_LOCATIONS  = ["chest", "ankle", "arm"]

MODEL_NAMES = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

print(f"[CONFIG] {DATASET_NAME} — Module 3 (Robustness Analysis)")
print(f"[CONFIG] Reading Module 1 artifacts from: {SOURCE_DIR.resolve()}")
print(f"[CONFIG] Writing Module 3 artifacts to:  {OUTPUT_DIR.resolve()}")

# ============================================================================
# 2. VALIDATE MODULE 1 ARTIFACTS EXIST
# ============================================================================
required_paths = [
    SOURCE_MODELS / "mhealth_scaler.pkl",
    SOURCE_MODELS / "mhealth_label_encoder.pkl",
    SOURCE_RESULTS / "mhealth_train_test_data.npz",
    SOURCE_FEATURES / "mhealth_feature_names.json",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required Module 1 artifacts:\n" + "\n".join(missing) +
        "\nPlease run Module 1 for MHEALTH first."
    )

model_paths = {name: SOURCE_MODELS / f"mhealth_{name}.pkl" for name in MODEL_NAMES}
missing_models = [name for name, p in model_paths.items() if not p.exists()]
if missing_models:
    raise FileNotFoundError(f"Missing trained model files for: {missing_models}")

# ============================================================================
# 3. LOAD MODULE 1 ARTIFACTS (READ-ONLY)
# ============================================================================
scaler = joblib.load(SOURCE_MODELS / "mhealth_scaler.pkl")
label_encoder = joblib.load(SOURCE_MODELS / "mhealth_label_encoder.pkl")

split_data = np.load(SOURCE_RESULTS / "mhealth_train_test_data.npz")
X_test, y_test = split_data["X_test"], split_data["y_test"]

with open(SOURCE_FEATURES / "mhealth_feature_names.json", "r", encoding=ENCODING) as f:
    feature_names = json.load(f)

trained_models = {name: joblib.load(path) for name, path in model_paths.items()}

print(f"[LOAD] Loaded {len(trained_models)} trained models and test set "
      f"of shape {X_test.shape} ({len(feature_names)} features)")

assert len(feature_names) == X_test.shape[1], \
    "Feature name count mismatch with X_test columns."

# ============================================================================
# 4. MAP FEATURE COLUMNS TO SENSOR LOCATIONS (for sensor-dropout perturbation)
# ============================================================================
def sensor_location_of(feat_name: str) -> str:
    if feat_name.startswith("chest_") or feat_name.startswith("ecg"):
        return "chest"
    if feat_name.startswith("ankle_"):
        return "ankle"
    if feat_name.startswith("arm_"):
        return "arm"
    return "other"

location_map = {loc: [] for loc in SENSOR_LOCATIONS + ["other"]}
for idx, fname in enumerate(feature_names):
    location_map[sensor_location_of(fname)].append(idx)

for loc in SENSOR_LOCATIONS:
    print(f"[MAP] Sensor '{loc}': {len(location_map[loc])} feature columns")
if location_map["other"]:
    print(f"[MAP] Unmapped/cross-sensor columns (not zeroed by sensor dropout): "
          f"{len(location_map['other'])}")

# ============================================================================
# 5. PERTURBATION FUNCTIONS
# ============================================================================
def add_gaussian_noise(X: np.ndarray, sigma_frac: float, seed: int) -> np.ndarray:
    rng = np.random.RandomState(seed)
    per_feature_std = X.std(axis=0, keepdims=True)
    noise = rng.normal(loc=0.0, scale=sigma_frac * per_feature_std, size=X.shape)
    return X + noise

def remove_features(X: np.ndarray, pct: float, seed: int) -> np.ndarray:
    rng = np.random.RandomState(seed)
    n_features = X.shape[1]
    n_remove = int(round(pct * n_features))
    cols_to_zero = rng.choice(n_features, size=n_remove, replace=False)
    X_pert = X.copy()
    X_pert[:, cols_to_zero] = 0.0
    return X_pert

def perturb_sensor(X: np.ndarray, location: str) -> np.ndarray:
    X_pert = X.copy()
    cols = location_map.get(location, [])
    if cols:
        X_pert[:, cols] = 0.0
    return X_pert

# ============================================================================
# 6. BASELINE (CLEAN) EVALUATION
# ============================================================================
print("\n[BASELINE] Computing clean-test-set performance for each model...")
baseline = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    baseline[name] = {"accuracy": acc, "f1_macro": f1}
    print(f"  {name:22s} | clean acc={acc:.4f} | clean F1={f1:.4f}")

# ============================================================================
# 7. RUN ALL PERTURBATION CONDITIONS FOR ALL MODELS
# ============================================================================
records = []
print("\n[ROBUSTNESS] Running perturbation conditions...")

for name, model in trained_models.items():
    clean_acc = baseline[name]["accuracy"]
    clean_f1 = baseline[name]["f1_macro"]

    for sigma in NOISE_LEVELS:
        X_pert = add_gaussian_noise(X_test, sigma, seed=RANDOM_STATE)
        y_pred = model.predict(X_pert)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        records.append({
            "Model": name, "Perturbation_Type": "Gaussian_Noise",
            "Condition": f"sigma={sigma}",
            "Accuracy": acc, "F1_Macro": f1,
            "Accuracy_Retention_%": 100 * acc / clean_acc if clean_acc > 0 else 0,
            "F1_Retention_%": 100 * f1 / clean_f1 if clean_f1 > 0 else 0,
        })

    for pct in FEATURE_REMOVAL_PCTS:
        X_pert = remove_features(X_test, pct, seed=RANDOM_STATE)
        y_pred = model.predict(X_pert)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        records.append({
            "Model": name, "Perturbation_Type": "Feature_Removal",
            "Condition": f"{int(pct*100)}%_removed",
            "Accuracy": acc, "F1_Macro": f1,
            "Accuracy_Retention_%": 100 * acc / clean_acc if clean_acc > 0 else 0,
            "F1_Retention_%": 100 * f1 / clean_f1 if clean_f1 > 0 else 0,
        })

    for loc in SENSOR_LOCATIONS:
        X_pert = perturb_sensor(X_test, loc)
        y_pred = model.predict(X_pert)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        records.append({
            "Model": name, "Perturbation_Type": "Sensor_Dropout",
            "Condition": f"{loc}_disabled",
            "Accuracy": acc, "F1_Macro": f1,
            "Accuracy_Retention_%": 100 * acc / clean_acc if clean_acc > 0 else 0,
            "F1_Retention_%": 100 * f1 / clean_f1 if clean_f1 > 0 else 0,
        })

    print(f"  {name:22s} | 9 perturbation conditions evaluated")

robustness_df = pd.DataFrame(records)

baseline_records = [
    {"Model": name, "Perturbation_Type": "Baseline", "Condition": "clean_test_set",
     "Accuracy": b["accuracy"], "F1_Macro": b["f1_macro"],
     "Accuracy_Retention_%": 100.0, "F1_Retention_%": 100.0}
    for name, b in baseline.items()
]
robustness_df = pd.concat([pd.DataFrame(baseline_records), robustness_df], ignore_index=True)

# ============================================================================
# 8. OVERALL ROBUSTNESS SCORE + RANKING
# ============================================================================
perturbed_only = robustness_df[robustness_df["Perturbation_Type"] != "Baseline"]

summary_records = []
for name in MODEL_NAMES:
    subset = perturbed_only[perturbed_only["Model"] == name]
    overall_score = subset[["Accuracy_Retention_%", "F1_Retention_%"]].mean().mean()

    per_type = subset.groupby("Perturbation_Type")[
        ["Accuracy_Retention_%", "F1_Retention_%"]
    ].mean().mean(axis=1)

    summary_records.append({
        "Model": name,
        "Clean_Accuracy": baseline[name]["accuracy"],
        "Clean_F1_Macro": baseline[name]["f1_macro"],
        "Avg_Retention_Gaussian_Noise_%": per_type.get("Gaussian_Noise", np.nan),
        "Avg_Retention_Feature_Removal_%": per_type.get("Feature_Removal", np.nan),
        "Avg_Retention_Sensor_Dropout_%": per_type.get("Sensor_Dropout", np.nan),
        "Overall_Robustness_Score": overall_score,
    })

summary_df = pd.DataFrame(summary_records).set_index("Model")
ranking_df = summary_df.sort_values("Overall_Robustness_Score", ascending=False).copy()
ranking_df.insert(0, "Robustness_Rank", range(1, len(ranking_df) + 1))

print("\n[RANKING] Overall Robustness Ranking (best -> worst):")
print(ranking_df[["Robustness_Rank", "Overall_Robustness_Score"]])

# ============================================================================
# 9. SAVE TABLES
# ============================================================================
robustness_df.to_csv(RESULTS_DIR / "mhealth_robustness_results.csv", index=False, encoding=ENCODING)
ranking_df.to_csv(RESULTS_DIR / "mhealth_robustness_ranking.csv", encoding=ENCODING)

with open(RESULTS_DIR / "mhealth_robustness_metrics.json", "w", encoding=ENCODING) as f:
    json.dump({
        "detailed_results": robustness_df.to_dict(orient="records"),
        "summary_ranking": ranking_df.reset_index().to_dict(orient="records"),
    }, f, indent=2, default=str)

print(f"\n[SAVE] Tables written to {RESULTS_DIR}")

# ============================================================================
# 10. FIGURES (300 dpi)
# ============================================================================
plot_order = ranking_df.index.tolist()

plt.figure(figsize=(11, 6))
values = ranking_df.loc[plot_order, "Overall_Robustness_Score"]
bars = plt.bar(plot_order, values, color="#4C72B0")
plt.ylabel("Overall Robustness Score (%)")
plt.title("MHEALTH — Overall Robustness Score by Classifier")
plt.xticks(rotation=45, ha="right")
plt.axhline(100, color="gray", linestyle="--", linewidth=1, label="Clean baseline (100%)")
plt.legend()
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
              f"{val:.1f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_robustness_overall_score.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(13, 7))
retention_cols = [
    "Avg_Retention_Gaussian_Noise_%",
    "Avg_Retention_Feature_Removal_%",
    "Avg_Retention_Sensor_Dropout_%",
]
x = np.arange(len(plot_order))
width = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]
labels = ["Gaussian Noise", "Feature Removal", "Sensor Dropout"]
for i, (col, color, label) in enumerate(zip(retention_cols, colors, labels)):
    plt.bar(x + (i - 1) * width, ranking_df.loc[plot_order, col], width,
            label=label, color=color)
plt.xticks(x, plot_order, rotation=45, ha="right")
plt.ylabel("Retention (%)")
plt.title("MHEALTH — Accuracy/F1 Retention by Perturbation Type")
plt.axhline(100, color="gray", linestyle="--", linewidth=1)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_robustness_by_perturbation_type.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(10, 6))
noise_subset = robustness_df[robustness_df["Perturbation_Type"] == "Gaussian_Noise"]
for name in MODEL_NAMES:
    sub = noise_subset[noise_subset["Model"] == name].copy()
    sub["sigma"] = sub["Condition"].str.replace("sigma=", "").astype(float)
    sub = sub.sort_values("sigma")
    plt.plot(sub["sigma"], sub["Accuracy_Retention_%"], marker="o", label=name)
plt.xlabel("Gaussian Noise Level (sigma, fraction of feature std)")
plt.ylabel("Accuracy Retention (%)")
plt.title("MHEALTH — Accuracy Retention vs Gaussian Noise Level")
plt.axhline(100, color="gray", linestyle="--", linewidth=1)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_robustness_noise_curve.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(10, 6))
fr_subset = robustness_df[robustness_df["Perturbation_Type"] == "Feature_Removal"]
for name in MODEL_NAMES:
    sub = fr_subset[fr_subset["Model"] == name].copy()
    sub["pct"] = sub["Condition"].str.replace("%_removed", "").astype(float)
    sub = sub.sort_values("pct")
    plt.plot(sub["pct"], sub["Accuracy_Retention_%"], marker="o", label=name)
plt.xlabel("Feature Columns Removed (%)")
plt.ylabel("Accuracy Retention (%)")
plt.title("MHEALTH — Accuracy Retention vs Feature Removal Percentage")
plt.axhline(100, color="gray", linestyle="--", linewidth=1)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_robustness_feature_removal_curve.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(9, 7))
sd_subset = robustness_df[robustness_df["Perturbation_Type"] == "Sensor_Dropout"]
pivot = sd_subset.pivot(index="Model", columns="Condition", values="Accuracy_Retention_%")
pivot = pivot.reindex(plot_order)
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100,
            cbar_kws={"label": "Accuracy Retention (%)"})
plt.title("MHEALTH — Accuracy Retention under Simulated Sensor Dropout")
plt.ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_robustness_sensor_dropout_heatmap.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 5 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 11. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MHEALTH Dataset — Module 3 Report",
    "## Robustness Analysis\n",
    "This module reuses trained models and the held-out test split from "
    "Module 1 without any retraining, preprocessing, or re-segmentation.\n",
    "### Methodology Summary",
    f"- **Gaussian Noise Injection**: additive noise at sigma in "
    f"{NOISE_LEVELS} (fraction of per-feature std)",
    f"- **Feature Removal**: {[f'{int(p*100)}%' for p in FEATURE_REMOVAL_PCTS]} "
    "of feature columns zeroed (random_state=42, identical subset across models per level)",
    f"- **Sensor Perturbation**: simulated failure of {SENSOR_LOCATIONS} on-body "
    "sensor locations (all associated feature columns zeroed)",
    "- **Retention Metric**: `100 x perturbed_metric / clean_metric` for both "
    "accuracy and macro-F1",
    "- **Overall Robustness Score**: mean of accuracy- and F1-retention across "
    "all 9 perturbation conditions per model\n",
    "## Clean Baseline Performance\n",
    pd.DataFrame(baseline).T.round(4).to_markdown(),
    "\n## Robustness Summary & Ranking (Best to Worst)\n",
    ranking_df.round(2).to_markdown(),
    "\n## Full Perturbation Results\n",
    "See `mhealth_robustness_results.csv` for the complete per-condition table "
    "(9 conditions x 7 models = 63 rows, plus 7 clean-baseline rows).\n",
    "## Most Robust Model\n",
    f"**{ranking_df.index[0]}** - Overall Robustness Score: "
    f"{ranking_df.iloc[0]['Overall_Robustness_Score']:.2f}%",
]
with open(REPORTS_DIR / "mhealth_module3_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'mhealth_module3_report.md'}")

# ============================================================================
# 12. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 3 COMPLETE")
print("=" * 50)

[CONFIG] MHEALTH — Module 3 (Robustness Analysis)
[CONFIG] Reading Module 1 artifacts from: C:\Users\Rohan Jaiswal\mhealth_outputs
[CONFIG] Writing Module 3 artifacts to:  C:\Users\Rohan Jaiswal\mhealth_robustness
[LOAD] Loaded 7 trained models and test set of shape (513, 344) (344 features)
[MAP] Sensor 'chest': 74 feature columns
[MAP] Sensor 'ankle': 135 feature columns
[MAP] Sensor 'arm': 135 feature columns

[BASELINE] Computing clean-test-set performance for each model...
  Logistic_Regression    | clean acc=1.0000 | clean F1=1.0000
  Decision_Tree          | clean acc=0.9922 | clean F1=0.9889
  Random_Forest          | clean acc=1.0000 | clean F1=1.0000
  KNN                    | clean acc=1.0000 | clean F1=1.0000
  Gaussian_NB            | clean acc=0.9727 | clean F1=0.9743
  Linear_SVM             | clean acc=1.0000 | clean F1=1.0000
  XGBoost                | clean acc=0.9961 | clean F1=0.9964

[ROBUSTNESS] Running perturbation conditions...
  Logistic_Regression    | 9 pertu

In [6]:
# ============================================================================
# MHEALTH DATASET — MODULE 4
# Statistical Validation
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# READ-ONLY with respect to Module 1. This script:
#   - Loads test-set predictions, ground truth, and metrics produced by
#     Module 1 from ./mhealth_outputs/
#   - Performs NO retraining, NO preprocessing, NO re-evaluation
#   - Writes ALL new outputs to ./mhealth_statistics/{results,figures,reports}/
#
# STANDARDIZED STATISTICAL METHODOLOGY (fixed for ALL datasets going forward)
# --------------------------------------------------------------------------
# 1) WILSON 95% CONFIDENCE INTERVALS (per model, on test-set accuracy)
# 2) COCHRAN'S Q TEST (equal accuracy across all 7 models)
# 3) PAIRWISE MCNEMAR TESTS (21 pairs, exact binomial or chi-square corrected)
# 4) HOLM-BONFERRONI CORRECTION (alpha=0.05, applied to 21 McNemar p-values)
# 5) STATISTICAL SUMMARY & RANKING (accuracy, CI width, significant wins)
#
# Figures: 300 dpi PNG. Folder convention:
#    ./mhealth_statistics/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import json
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats as sp_stats

try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels", "--quiet"])
    from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME     = "MHEALTH"

SOURCE_DIR       = Path("./mhealth_outputs")           # READ-ONLY
SOURCE_RESULTS   = SOURCE_DIR / "results"

OUTPUT_DIR       = Path("./mhealth_statistics")        # WRITE-ONLY here
RESULTS_DIR      = OUTPUT_DIR / "results"
FIGURES_DIR      = OUTPUT_DIR / "figures"
REPORTS_DIR      = OUTPUT_DIR / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FIGURE_DPI       = 300
ENCODING         = "utf-8"
ALPHA            = 0.05
Z_95             = 1.959963985

MODEL_NAMES = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

print(f"[CONFIG] {DATASET_NAME} — Module 4 (Statistical Validation)")
print(f"[CONFIG] Reading Module 1 artifacts from: {SOURCE_DIR.resolve()}")
print(f"[CONFIG] Writing Module 4 artifacts to:  {OUTPUT_DIR.resolve()}")

# ============================================================================
# 2. VALIDATE MODULE 1 ARTIFACTS EXIST
# ============================================================================
required_paths = [
    SOURCE_RESULTS / "mhealth_test_predictions.csv",
    SOURCE_RESULTS / "mhealth_metrics.json",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required Module 1 artifacts:\n" + "\n".join(missing) +
        "\nPlease run Module 1 for MHEALTH first."
    )

# ============================================================================
# 3. LOAD MODULE 1 ARTIFACTS (READ-ONLY)
# ============================================================================
pred_df = pd.read_csv(SOURCE_RESULTS / "mhealth_test_predictions.csv", encoding=ENCODING)

with open(SOURCE_RESULTS / "mhealth_metrics.json", "r", encoding=ENCODING) as f:
    module1_metrics = json.load(f)

missing_cols = [c for c in MODEL_NAMES + ["y_true"] if c not in pred_df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in test_predictions.csv: {missing_cols}")

y_true = pred_df["y_true"].values
n_samples = len(y_true)

correct_matrix = pd.DataFrame({
    name: (pred_df[name].values == y_true).astype(int) for name in MODEL_NAMES
})

print(f"[LOAD] Loaded predictions for {len(MODEL_NAMES)} models on "
      f"{n_samples} held-out test samples.")

# ============================================================================
# 4. WILSON 95% CONFIDENCE INTERVALS
# ============================================================================
def wilson_ci(successes: int, n: int, z: float = Z_95):
    p_hat = successes / n
    denom = 1 + (z ** 2) / n
    center = (p_hat + (z ** 2) / (2 * n)) / denom
    margin = (z / denom) * np.sqrt((p_hat * (1 - p_hat) / n) + (z ** 2) / (4 * n ** 2))
    return p_hat, max(0.0, center - margin), min(1.0, center + margin)

print("\n[WILSON] Computing 95% Wilson confidence intervals...")
wilson_records = []
for name in MODEL_NAMES:
    successes = int(correct_matrix[name].sum())
    p_hat, lower, upper = wilson_ci(successes, n_samples)
    wilson_records.append({
        "Model": name,
        "Accuracy": p_hat,
        "Wilson_CI_Lower": lower,
        "Wilson_CI_Upper": upper,
        "CI_Width": upper - lower,
        "N_Correct": successes,
        "N_Total": n_samples,
    })
    print(f"  {name:22s} | acc={p_hat:.4f} | 95% CI=[{lower:.4f}, {upper:.4f}]")

wilson_df = pd.DataFrame(wilson_records).set_index("Model")
wilson_df.to_csv(RESULTS_DIR / "mhealth_wilson_confidence_intervals.csv", encoding=ENCODING)

# ============================================================================
# 5. COCHRAN'S Q TEST
# ============================================================================
print("\n[COCHRAN'S Q] Testing H0: all models have equal accuracy...")

def cochrans_q(binary_matrix: pd.DataFrame):
    k = binary_matrix.shape[1]
    col_sums = binary_matrix.sum(axis=0).values
    row_sums = binary_matrix.sum(axis=1).values
    L = col_sums.sum()

    numerator = k * (k - 1) * np.sum((col_sums - L / k) ** 2)
    denominator = k * L - np.sum(row_sums ** 2)
    Q = numerator / denominator if denominator != 0 else 0.0
    df = k - 1
    p_value = 1 - sp_stats.chi2.cdf(Q, df)
    return Q, df, p_value

Q_stat, Q_df, Q_pvalue = cochrans_q(correct_matrix)
cochran_result = {
    "Q_statistic": Q_stat,
    "degrees_of_freedom": Q_df,
    "p_value": Q_pvalue,
    "significant_at_0.05": bool(Q_pvalue < ALPHA),
    "interpretation": (
        "Reject H0: at least one model's accuracy differs significantly from the others."
        if Q_pvalue < ALPHA else
        "Fail to reject H0: no significant difference in accuracy detected across models."
    ),
}
print(f"  Q = {Q_stat:.4f}, df = {Q_df}, p = {Q_pvalue:.6f}")
print(f"  {cochran_result['interpretation']}")

with open(RESULTS_DIR / "mhealth_cochrans_q_results.json", "w", encoding=ENCODING) as f:
    json.dump(cochran_result, f, indent=2)

# ============================================================================
# 6. PAIRWISE MCNEMAR TESTS
# ============================================================================
print("\n[MCNEMAR] Running pairwise McNemar tests for all model pairs...")
mcnemar_records = []

for model_a, model_b in combinations(MODEL_NAMES, 2):
    correct_a = correct_matrix[model_a].values
    correct_b = correct_matrix[model_b].values

    both_correct = int(np.sum((correct_a == 1) & (correct_b == 1)))
    a_only = int(np.sum((correct_a == 1) & (correct_b == 0)))
    b_only = int(np.sum((correct_a == 0) & (correct_b == 1)))
    both_wrong = int(np.sum((correct_a == 0) & (correct_b == 0)))

    contingency = [[both_correct, a_only], [b_only, both_wrong]]
    discordant = a_only + b_only
    exact = discordant < 25

    result = mcnemar(contingency, exact=exact, correction=True)
    mcnemar_records.append({
        "Model_A": model_a, "Model_B": model_b,
        "A_only_correct": a_only, "B_only_correct": b_only,
        "Discordant_pairs": discordant,
        "Statistic": result.statistic,
        "p_value": result.pvalue,
        "Test_Type": "exact_binomial" if exact else "chi_square_corrected",
    })

mcnemar_df = pd.DataFrame(mcnemar_records)
print(f"  Completed {len(mcnemar_df)} pairwise tests.")

# ============================================================================
# 7. HOLM-BONFERRONI CORRECTION
# ============================================================================
print("\n[HOLM-BONFERRONI] Applying multiple-comparison correction...")

def holm_bonferroni(p_values: np.ndarray, alpha: float = 0.05):
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        factor = m - rank
        adj_p = p_values[idx] * factor
        running_max = max(running_max, adj_p)
        adjusted[idx] = min(running_max, 1.0)
    significant = adjusted < alpha
    return adjusted, significant

adjusted_pvals, significant_flags = holm_bonferroni(mcnemar_df["p_value"].values, ALPHA)
mcnemar_df["Holm_Adjusted_p_value"] = adjusted_pvals
mcnemar_df["Significant_after_Holm"] = significant_flags

n_significant = int(significant_flags.sum())
print(f"  {n_significant} / {len(mcnemar_df)} pairs remain significant after "
      f"Holm-Bonferroni correction (alpha={ALPHA}).")

mcnemar_df.to_csv(RESULTS_DIR / "mhealth_pairwise_mcnemar_holm.csv", index=False, encoding=ENCODING)

# ============================================================================
# 8. STATISTICAL SUMMARY & MODEL RANKING
# ============================================================================
print("\n[SUMMARY] Building statistical summary and ranking...")

win_counts = {name: 0 for name in MODEL_NAMES}
for _, row in mcnemar_df.iterrows():
    if row["Significant_after_Holm"]:
        if row["A_only_correct"] > row["B_only_correct"]:
            win_counts[row["Model_A"]] += 1
        elif row["B_only_correct"] > row["A_only_correct"]:
            win_counts[row["Model_B"]] += 1

summary_records = []
for name in MODEL_NAMES:
    summary_records.append({
        "Model": name,
        "Accuracy": wilson_df.loc[name, "Accuracy"],
        "Wilson_CI_Lower": wilson_df.loc[name, "Wilson_CI_Lower"],
        "Wilson_CI_Upper": wilson_df.loc[name, "Wilson_CI_Upper"],
        "CI_Width": wilson_df.loc[name, "CI_Width"],
        "Significant_Pairwise_Wins": win_counts[name],
        "Macro_F1": module1_metrics[name]["f1_macro"],
        "CV_Accuracy_Mean": module1_metrics[name]["cv_accuracy_mean"],
        "CV_Accuracy_Std": module1_metrics[name]["cv_accuracy_std"],
    })

summary_df = pd.DataFrame(summary_records).set_index("Model")
ranking_df = summary_df.sort_values(
    ["Significant_Pairwise_Wins", "Accuracy"], ascending=[False, False]
).copy()
ranking_df.insert(0, "Statistical_Rank", range(1, len(ranking_df) + 1))

ranking_df.to_csv(RESULTS_DIR / "mhealth_statistical_summary.csv", encoding=ENCODING)

with open(RESULTS_DIR / "mhealth_statistical_validation_full.json", "w", encoding=ENCODING) as f:
    json.dump({
        "wilson_confidence_intervals": wilson_df.reset_index().to_dict(orient="records"),
        "cochrans_q_test": cochran_result,
        "pairwise_mcnemar_holm": mcnemar_df.to_dict(orient="records"),
        "statistical_summary_ranking": ranking_df.reset_index().to_dict(orient="records"),
    }, f, indent=2, default=str)

print("\n[RANKING] Statistical ranking (best -> worst):")
print(ranking_df[["Statistical_Rank", "Accuracy", "Significant_Pairwise_Wins"]])

# ============================================================================
# 9. FIGURES (300 dpi)
# ============================================================================
plot_order = ranking_df.index.tolist()

plt.figure(figsize=(10, 7))
y_pos = np.arange(len(plot_order))
accs = wilson_df.loc[plot_order, "Accuracy"].values
lowers = wilson_df.loc[plot_order, "Wilson_CI_Lower"].values
uppers = wilson_df.loc[plot_order, "Wilson_CI_Upper"].values
errors = np.vstack([accs - lowers, uppers - accs])
plt.errorbar(accs, y_pos, xerr=errors, fmt="o", color="#4C72B0",
             ecolor="#4C72B0", elinewidth=2, capsize=4, markersize=7)
plt.yticks(y_pos, plot_order)
plt.xlabel("Accuracy (95% Wilson CI)")
plt.title("MHEALTH — Wilson 95% Confidence Intervals by Classifier")
plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_wilson_ci_forest_plot.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(9, 8))
pval_matrix = pd.DataFrame(np.ones((len(MODEL_NAMES), len(MODEL_NAMES))),
                           index=MODEL_NAMES, columns=MODEL_NAMES)
for _, row in mcnemar_df.iterrows():
    pval_matrix.loc[row["Model_A"], row["Model_B"]] = row["Holm_Adjusted_p_value"]
    pval_matrix.loc[row["Model_B"], row["Model_A"]] = row["Holm_Adjusted_p_value"]
np.fill_diagonal(pval_matrix.values, np.nan)

sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f", cmap="coolwarm_r",
            vmin=0, vmax=1, cbar_kws={"label": "Holm-adjusted p-value"},
            mask=pval_matrix.isna())
plt.title("MHEALTH — Pairwise McNemar Test (Holm-Bonferroni adjusted p-values)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_mcnemar_holm_heatmap.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(10, 6))
wins = ranking_df.loc[plot_order, "Significant_Pairwise_Wins"]
bars = plt.bar(plot_order, wins, color="#55A868")
plt.ylabel("Number of Significant Pairwise Wins (Holm-adjusted)")
plt.title("MHEALTH — Statistically Significant Pairwise Wins by Classifier")
plt.xticks(rotation=45, ha="right")
for bar, val in zip(bars, wins):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
              f"{int(val)}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_significant_wins.png", dpi=FIGURE_DPI)
plt.close()

plt.figure(figsize=(9, 7))
plt.scatter(wilson_df.loc[plot_order, "Accuracy"], wilson_df.loc[plot_order, "CI_Width"],
            s=100, color="#C44E52")
for name in plot_order:
    plt.annotate(name, (wilson_df.loc[name, "Accuracy"], wilson_df.loc[name, "CI_Width"]),
                 fontsize=8, xytext=(5, 5), textcoords="offset points")
plt.xlabel("Accuracy")
plt.ylabel("Wilson 95% CI Width")
plt.title("MHEALTH — Accuracy vs Confidence Interval Width")
plt.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mhealth_accuracy_vs_ci_width.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 4 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 10. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MHEALTH Dataset — Module 4 Report",
    "## Statistical Validation\n",
    "This module reuses test-set predictions and metrics from Module 1 "
    "without any retraining, preprocessing, or re-evaluation.\n",
    "### Methodology Summary",
    "- **Wilson 95% Confidence Intervals**: computed per model on test-set "
    "accuracy (Bernoulli correct/incorrect trials)",
    "- **Cochran's Q Test**: tests equality of accuracy across all 7 models "
    "on the same test instances",
    "- **Pairwise McNemar Tests**: 21 pairwise comparisons using discordant "
    "correct/incorrect pairs (exact binomial if discordant count < 25, else "
    "chi-square with continuity correction)",
    "- **Holm-Bonferroni Correction**: applied to the 21 McNemar p-values, "
    f"alpha = {ALPHA}",
    "- **Statistical Ranking**: models ranked by number of statistically "
    "significant pairwise wins, then by raw accuracy\n",
    "## Wilson 95% Confidence Intervals\n",
    wilson_df.round(4).to_markdown(),
    "\n## Cochran's Q Test\n",
    f"- Q statistic: {Q_stat:.4f}",
    f"- Degrees of freedom: {Q_df}",
    f"- p-value: {Q_pvalue:.6f}",
    f"- {cochran_result['interpretation']}\n",
    "## Pairwise McNemar Tests (Holm-Bonferroni Adjusted)\n",
    mcnemar_df.round(4).to_markdown(index=False),
    "\n## Statistical Summary & Ranking (Best to Worst)\n",
    ranking_df.round(4).to_markdown(),
    "\n## Statistically Best-Validated Model\n",
    f"**{ranking_df.index[0]}** - Accuracy: {ranking_df.iloc[0]['Accuracy']:.4f}, "
    f"95% CI: [{ranking_df.iloc[0]['Wilson_CI_Lower']:.4f}, "
    f"{ranking_df.iloc[0]['Wilson_CI_Upper']:.4f}], "
    f"Significant pairwise wins: {int(ranking_df.iloc[0]['Significant_Pairwise_Wins'])}/6",
]
with open(REPORTS_DIR / "mhealth_module4_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'mhealth_module4_report.md'}")

# ============================================================================
# 11. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 4 COMPLETE")
print("=" * 50)

[CONFIG] MHEALTH — Module 4 (Statistical Validation)
[CONFIG] Reading Module 1 artifacts from: C:\Users\Rohan Jaiswal\mhealth_outputs
[CONFIG] Writing Module 4 artifacts to:  C:\Users\Rohan Jaiswal\mhealth_statistics
[LOAD] Loaded predictions for 7 models on 513 held-out test samples.

[WILSON] Computing 95% Wilson confidence intervals...
  Logistic_Regression    | acc=1.0000 | 95% CI=[0.9926, 1.0000]
  Decision_Tree          | acc=0.9922 | 95% CI=[0.9801, 0.9970]
  Random_Forest          | acc=1.0000 | 95% CI=[0.9926, 1.0000]
  KNN                    | acc=1.0000 | 95% CI=[0.9926, 1.0000]
  Gaussian_NB            | acc=0.9727 | 95% CI=[0.9547, 0.9837]
  Linear_SVM             | acc=1.0000 | 95% CI=[0.9926, 1.0000]
  XGBoost                | acc=0.9961 | 95% CI=[0.9859, 0.9989]

[COCHRAN'S Q] Testing H0: all models have equal accuracy...
  Q = 55.6000, df = 6, p = 0.000000
  Reject H0: at least one model's accuracy differs significantly from the others.

[MCNEMAR] Running pairwise McNe

In [1]:
# ============================================================================
# MOTIONSENSE DATASET — MODULE 1
# Dataset Preparation, Feature Engineering, Model Training & Performance Evaluation
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# STANDARDIZED DEFAULTS (fixed for ALL datasets: UCI HAR, WISDM, PAMAP2, MHEALTH, MotionSense)
# --------------------------------------------------------------------------
# - Windowing: dataset-native sampling rate, 50% overlap. MotionSense's
#   DeviceMotion stream is sampled at 50 Hz, so we use the same 5-second
#   window (250 samples, step = 125 samples) as PAMAP2/MHEALTH for
#   cross-dataset consistency.
# - No null/no-activity class exists in MotionSense (all 6 classes are
#   clean, fully-labeled activities), so no null-class removal is needed.
# - Data organization: A_DeviceMotion_data/{activity}_{trial}/sub_{id}.csv.
#   Each CSV file is ALREADY a single continuous recording for one subject,
#   one activity, one trial — so sliding windows are generated PER
#   (subject, trial) FILE, never crossing file boundaries.
# - Feature engineering (identical methodology for every dataset):
#     * Time-domain:  mean, std, min, max, range, median, variance, RMS, IQR,
#                     zero-crossing-rate  (10 features per channel)
#     * Frequency-domain: spectral energy, spectral entropy, dominant
#                     frequency, spectral centroid  (4 features per channel)
#     * Cross-axis correlation features for the four 3-dimensional sensor
#       groups (attitude, gravity, rotationRate, userAcceleration): Pearson
#       r between each pair of the three components
# - Train/test strategy: stratified 80/20 split (random_state=42).
#   MODEL SELECTION: GridSearchCV (5-fold Stratified CV) over a documented
#   hyperparameter grid per classifier, run on the training set. The
#   best_estimator_ from each grid search is then:
#     (a) evaluated once on the held-out test set, and
#     (b) re-evaluated via a fresh 5-fold Stratified CV (accuracy) to report
#         CV mean/std for the selected hyperparameters.
# - Metrics: Accuracy, Precision (macro), Recall (macro), Macro-F1,
#   Confusion Matrix, full Classification Report, 5-fold CV accuracy (mean±std)
# - Figures: 300 dpi, saved as PNG.
# - Folder convention: ./motionsense_outputs/{raw_data,features,models,results,figures,reports}/
# - ENCODING: All text file writes and CSV exports use encoding="utf-8"
#   explicitly, to avoid platform-dependent default-encoding errors (e.g.
#   cp1252 on Windows failing on special characters).
# ============================================================================

import os
import sys
import json
import time
import zipfile
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib

warnings.filterwarnings("ignore")

# Ensure xgboost is available
try:
    from xgboost import XGBClassifier
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "--quiet"])
    from xgboost import XGBClassifier

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME   = "MotionSense"
BASE_DIR       = Path("./motionsense_outputs")
RAW_DIR        = BASE_DIR / "raw_data"
FEATURES_DIR   = BASE_DIR / "features"
MODELS_DIR     = BASE_DIR / "models"
RESULTS_DIR    = BASE_DIR / "results"
FIGURES_DIR    = BASE_DIR / "figures"
REPORTS_DIR    = BASE_DIR / "reports"

for d in [RAW_DIR, FEATURES_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLING_RATE   = 50            # Hz, official MotionSense DeviceMotion sampling rate
WINDOW_SECONDS  = 5              # standardized default (documented above)
WINDOW_SIZE     = SAMPLING_RATE * WINDOW_SECONDS   # 250 samples
STEP_SIZE       = WINDOW_SIZE // 2                  # 50% overlap -> 125 samples

TEST_SIZE       = 0.2
RANDOM_STATE    = 42
CV_FOLDS        = 5

FIGURE_DPI      = 300
ENCODING        = "utf-8"

# Activity folder-prefix -> full activity name
ACTIVITY_CODE_MAP = {
    "dws": "Downstairs",
    "ups": "Upstairs",
    "wlk": "Walking",
    "jog": "Jogging",
    "sit": "Sitting",
    "std": "Standing",
}

SENSOR_CHANNELS = [
    "attitude.roll", "attitude.pitch", "attitude.yaw",
    "gravity.x", "gravity.y", "gravity.z",
    "rotationRate.x", "rotationRate.y", "rotationRate.z",
    "userAcceleration.x", "userAcceleration.y", "userAcceleration.z",
]

TRIAXIAL_GROUPS = {
    "attitude": ["attitude.roll", "attitude.pitch", "attitude.yaw"],
    "gravity": ["gravity.x", "gravity.y", "gravity.z"],
    "rotationRate": ["rotationRate.x", "rotationRate.y", "rotationRate.z"],
    "userAcceleration": ["userAcceleration.x", "userAcceleration.y", "userAcceleration.z"],
}

print(f"[CONFIG] {DATASET_NAME} | window={WINDOW_SECONDS}s ({WINDOW_SIZE} samples) "
      f"| step={STEP_SIZE} samples (50% overlap) | fs={SAMPLING_RATE}Hz")

# ============================================================================
# 2. DOWNLOAD & EXTRACT OFFICIAL DATASET
# ============================================================================
def download_motionsense(raw_dir: Path) -> Path:
    existing_csvs = list(raw_dir.rglob("sub_*.csv"))
    if existing_csvs:
        print(f"[DATA] Found {len(existing_csvs)} existing subject CSV files. Skipping download.")
        return raw_dir

    urls = [
        "https://github.com/mmalekzadeh/motion-sense/raw/master/data/A_DeviceMotion_data.zip",
        "https://raw.githubusercontent.com/mmalekzadeh/motion-sense/master/data/A_DeviceMotion_data.zip",
    ]
    zip_path = raw_dir / "A_DeviceMotion_data.zip"
    downloaded = False
    for url in urls:
        try:
            print(f"[DATA] Attempting download from: {url}")
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                f.write(r.content)
            downloaded = True
            print("[DATA] Download successful.")
            break
        except Exception as e:
            print(f"[DATA] Failed: {e}")

    if not downloaded:
        raise RuntimeError(
            "Could not automatically download the official MotionSense dataset. "
            "Please manually download 'A_DeviceMotion_data.zip' from "
            "https://github.com/mmalekzadeh/motion-sense (data folder) and "
            "extract it into ./motionsense_outputs/raw_data/"
        )

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(raw_dir)
    print(f"[DATA] Extracted to {raw_dir}")
    return raw_dir


raw_path = download_motionsense(RAW_DIR)
trial_dirs = sorted([d for d in raw_path.rglob("*") if d.is_dir() and "_" in d.name
                     and d.name.split("_")[0] in ACTIVITY_CODE_MAP])
if len(trial_dirs) == 0:
    raise RuntimeError("No MotionSense activity trial folders found after extraction.")
print(f"[DATA] Located {len(trial_dirs)} activity trial folders.")

# ============================================================================
# 3. LOAD RAW DATA (PER SUBJECT, PER TRIAL FILE)
# ============================================================================
all_frames = []
for trial_dir in trial_dirs:
    activity_code = trial_dir.name.split("_")[0]
    activity_name = ACTIVITY_CODE_MAP[activity_code]
    trial_id = trial_dir.name

    subject_files = sorted(trial_dir.glob("sub_*.csv"))
    for f in subject_files:
        subj_id = int(f.stem.replace("sub_", ""))
        df = pd.read_csv(f, index_col=0)
        # Some files may have minor column ordering differences; enforce order
        df = df[SENSOR_CHANNELS]
        df["label"] = activity_name
        df["subject"] = subj_id
        df["trial_id"] = trial_id
        all_frames.append(df)

raw_df = pd.concat(all_frames, ignore_index=True)
print(f"[DATA] Loaded {len(all_frames)} subject-trial files. "
      f"Total samples: {len(raw_df)}")
print(f"[DATA] Activity distribution:\n{raw_df['label'].value_counts()}")

# ============================================================================
# 4. FEATURE ENGINEERING FUNCTIONS
# ============================================================================
def time_domain_features(sig: np.ndarray) -> dict:
    mean = np.mean(sig)
    std = np.std(sig)
    mn = np.min(sig)
    mx = np.max(sig)
    med = np.median(sig)
    var = np.var(sig)
    rms = np.sqrt(np.mean(sig ** 2))
    q75, q25 = np.percentile(sig, [75, 25])
    iqr = q75 - q25
    zcr = np.sum(np.diff(np.sign(sig)) != 0) / len(sig)
    return {
        "mean": mean, "std": std, "min": mn, "max": mx, "range": mx - mn,
        "median": med, "var": var, "rms": rms, "iqr": iqr, "zcr": zcr
    }

def freq_domain_features(sig: np.ndarray, fs: int) -> dict:
    n = len(sig)
    fft_vals = np.abs(rfft(sig - np.mean(sig)))
    freqs = rfftfreq(n, d=1.0 / fs)
    power = fft_vals ** 2
    total_power = np.sum(power) + 1e-12

    energy = np.sum(power) / n
    prob = power / total_power
    entropy = -np.sum(prob * np.log2(prob + 1e-12))

    if len(freqs) > 1:
        dom_idx = np.argmax(fft_vals[1:]) + 1
        dominant_freq = freqs[dom_idx]
    else:
        dominant_freq = 0.0

    centroid = np.sum(freqs * power) / total_power

    return {
        "spec_energy": energy, "spec_entropy": entropy,
        "dominant_freq": dominant_freq, "spec_centroid": centroid
    }

def correlation_features(window_df: pd.DataFrame) -> dict:
    feats = {}
    for group_name, cols in TRIAXIAL_GROUPS.items():
        a, b, c = [window_df[col].values for col in cols]
        feats[f"{group_name}_corr_xy"] = safe_corr(a, b)
        feats[f"{group_name}_corr_yz"] = safe_corr(b, c)
        feats[f"{group_name}_corr_xz"] = safe_corr(a, c)
    return feats

def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-9 or np.std(b) < 1e-9:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else r

def extract_window_features(window_df: pd.DataFrame) -> dict:
    feats = {}
    for ch in SENSOR_CHANNELS:
        sig = window_df[ch].values.astype(float)
        td = time_domain_features(sig)
        fd = freq_domain_features(sig, SAMPLING_RATE)
        for k, v in td.items():
            feats[f"{ch}_{k}"] = v
        for k, v in fd.items():
            feats[f"{ch}_{k}"] = v
    feats.update(correlation_features(window_df))
    return feats

# ============================================================================
# 5. SLIDING WINDOW SEGMENTATION (per subject, per trial FILE — already
#    contiguous single-activity recordings, so no label-change detection
#    is required, unlike MHEALTH/PAMAP2)
# ============================================================================
def segment_windows(df: pd.DataFrame) -> list:
    windows = []
    for (subj_id, trial_id), group in df.groupby(["subject", "trial_id"]):
        group = group.reset_index(drop=True)
        label = group["label"].iloc[0]
        n = len(group)
        if n < WINDOW_SIZE:
            continue
        start = 0
        while start + WINDOW_SIZE <= n:
            win = group.iloc[start:start + WINDOW_SIZE]
            windows.append((win, label, subj_id, trial_id))
            start += STEP_SIZE
    return windows

print("[SEGMENT] Generating sliding windows...")
windows = segment_windows(raw_df)
print(f"[SEGMENT] Total windows generated: {len(windows)}")

# ============================================================================
# 6. BUILD FEATURE MATRIX
# ============================================================================
print("[FEATURES] Extracting features from each window (this may take a few minutes)...")
feature_rows = []
labels = []
subjects = []
trial_ids = []
t0 = time.time()
for i, (win_df, label, subj_id, trial_id) in enumerate(windows):
    feats = extract_window_features(win_df)
    feature_rows.append(feats)
    labels.append(label)
    subjects.append(subj_id)
    trial_ids.append(trial_id)
    if (i + 1) % 500 == 0:
        print(f"  ...{i+1}/{len(windows)} windows processed")

feature_df = pd.DataFrame(feature_rows)
feature_df["label"] = labels
feature_df["subject"] = subjects
feature_df["trial_id"] = trial_ids
print(f"[FEATURES] Extraction complete in {time.time()-t0:.1f}s. "
      f"Feature matrix shape: {feature_df.shape}")

feature_csv_path = FEATURES_DIR / "motionsense_features.csv"
feature_df.to_csv(feature_csv_path, index=False, encoding=ENCODING)
with open(FEATURES_DIR / "motionsense_feature_names.json", "w", encoding=ENCODING) as f:
    json.dump([c for c in feature_df.columns if c not in ["label", "subject", "trial_id"]], f, indent=2)
print(f"[FEATURES] Saved to {feature_csv_path}")

# ============================================================================
# 7. TRAIN/TEST SPLIT + SCALING
# ============================================================================
feature_cols = [c for c in feature_df.columns if c not in ["label", "subject", "trial_id"]]
X = feature_df[feature_cols].values
y_raw = feature_df["label"].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"[SPLIT] Train: {X_train.shape}, Test: {X_test.shape}")

np.savez(RESULTS_DIR / "motionsense_train_test_data.npz",
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train, y_test=y_test)
joblib.dump(scaler, MODELS_DIR / "motionsense_scaler.pkl")
joblib.dump(le, MODELS_DIR / "motionsense_label_encoder.pkl")

# ============================================================================
# 8. DEFINE MODELS + HYPERPARAMETER GRIDS (GridSearchCV, identical across datasets)
# ============================================================================
base_models = {
    "Logistic_Regression": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    "Decision_Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random_Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "KNN": KNeighborsClassifier(),
    "Gaussian_NB": GaussianNB(),
    "Linear_SVM": LinearSVC(random_state=RANDOM_STATE, max_iter=5000),
    "XGBoost": XGBClassifier(
        use_label_encoder=False, eval_metric="mlogloss",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

param_grids = {
    "Logistic_Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs"],
    },
    "Decision_Tree": {
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "criterion": ["gini", "entropy"],
    },
    "Random_Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5],
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    },
    "Gaussian_NB": {
        "var_smoothing": [1e-9, 1e-8, 1e-7],
    },
    "Linear_SVM": {
        "C": [0.01, 0.1, 1, 10],
    },
    "XGBoost": {
        "n_estimators": [100, 200],
        "max_depth": [3, 6, 9],
        "learning_rate": [0.05, 0.1, 0.2],
    },
}

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# ============================================================================
# 9. GRIDSEARCHCV, BEST-ESTIMATOR TEST EVALUATION, POST-HOC 5-FOLD CV
# ============================================================================
all_metrics = {}
all_reports = {}
predictions_store = {}
best_params_store = {}

for name, base_model in base_models.items():
    print(f"\n[MODEL] GridSearchCV for {name}...")
    t0 = time.time()

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grids[name],
        cv=skf,
        scoring="accuracy",
        n_jobs=-1,
        refit=True,
    )
    grid.fit(X_train_scaled, y_train)
    grid_search_time = time.time() - t0

    best_model = grid.best_estimator_
    best_params_store[name] = grid.best_params_
    print(f"  Best params: {grid.best_params_} (grid search took {grid_search_time:.1f}s)")

    t0 = time.time()
    best_model.fit(X_train_scaled, y_train)
    train_time = time.time() - t0

    y_pred = best_model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=skf, scoring="accuracy", n_jobs=-1)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(
        y_test, y_pred, target_names=[str(c) for c in le.classes_],
        output_dict=True, zero_division=0
    )

    all_metrics[name] = {
        "accuracy": acc, "precision_macro": prec, "recall_macro": rec,
        "f1_macro": f1, "cv_accuracy_mean": cv_scores.mean(),
        "cv_accuracy_std": cv_scores.std(), "train_time_sec": train_time,
        "grid_search_time_sec": grid_search_time, "best_params": grid.best_params_
    }
    all_reports[name] = report
    predictions_store[name] = y_pred

    joblib.dump(best_model, MODELS_DIR / f"motionsense_{name}.pkl")

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f"MotionSense — Confusion Matrix — {name} (GridSearchCV best estimator)")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"motionsense_confusion_matrix_{name}.png", dpi=FIGURE_DPI)
    plt.close()

    print(f"  Test: Accuracy={acc:.4f} | Macro-F1={f1:.4f} | "
          f"CV Acc={cv_scores.mean():.4f}±{cv_scores.std():.4f} | Train time={train_time:.2f}s")

with open(MODELS_DIR / "motionsense_best_hyperparameters.json", "w", encoding=ENCODING) as f:
    json.dump(best_params_store, f, indent=2, default=str)

# ============================================================================
# 10. SAVE METRICS + PREDICTIONS
# ============================================================================
with open(RESULTS_DIR / "motionsense_metrics.json", "w", encoding=ENCODING) as f:
    json.dump(all_metrics, f, indent=2, default=str)

with open(RESULTS_DIR / "motionsense_classification_reports.json", "w", encoding=ENCODING) as f:
    json.dump(all_reports, f, indent=2)

pred_df = pd.DataFrame(predictions_store)
pred_df["y_true"] = y_test
pred_df.to_csv(RESULTS_DIR / "motionsense_test_predictions.csv", index=False, encoding=ENCODING)

metrics_df = pd.DataFrame(all_metrics).T
metrics_df.to_csv(RESULTS_DIR / "motionsense_metrics_summary.csv", encoding=ENCODING)

# ============================================================================
# 11. MODEL COMPARISON FIGURE
# ============================================================================
plt.figure(figsize=(12, 6))
metrics_df_sorted = metrics_df.sort_values("accuracy", ascending=False)
x = np.arange(len(metrics_df_sorted))
width = 0.35
plt.bar(x - width/2, metrics_df_sorted["accuracy"].astype(float), width, label="Test Accuracy")
plt.bar(x + width/2, metrics_df_sorted["f1_macro"].astype(float), width, label="Macro F1")
plt.xticks(x, metrics_df_sorted.index, rotation=45, ha="right")
plt.ylabel("Score")
plt.title("MotionSense — Model Comparison (Accuracy vs Macro-F1, GridSearchCV best estimators)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_model_comparison.png", dpi=FIGURE_DPI)
plt.close()

# ============================================================================
# 12. GENERATE MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MotionSense Dataset — Module 1 Report",
    "## Dataset Preparation, Feature Engineering, Model Training & Evaluation\n",
    f"- Total windows: {len(windows)}",
    f"- Window size: {WINDOW_SECONDS}s ({WINDOW_SIZE} samples @ {SAMPLING_RATE}Hz), 50% overlap",
    f"- Feature count: {len(feature_cols)}",
    f"- Activity classes: {list(le.classes_)}",
    f"- Train/Test split: {1-TEST_SIZE:.0%}/{TEST_SIZE:.0%} stratified, random_state={RANDOM_STATE}",
    f"- Model selection: GridSearchCV with {CV_FOLDS}-fold Stratified CV per classifier",
    f"- Post-selection evaluation: best_estimator_ scored once on held-out test set, "
    f"then re-validated via fresh {CV_FOLDS}-fold Stratified CV\n",
    "## Best Hyperparameters per Model\n",
    json.dumps(best_params_store, indent=2, default=str),
    "\n## Model Performance Summary\n",
    metrics_df_sorted.drop(columns=["best_params"], errors="ignore").to_markdown(),
]
with open(REPORTS_DIR / "motionsense_module1_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print("\n" + "=" * 70)
print("MODULE 1 (MOTIONSENSE) COMPLETE — GridSearchCV methodology applied")
print("=" * 70)
print(metrics_df_sorted[["accuracy", "f1_macro", "cv_accuracy_mean"]])
print(f"\nAll artifacts saved under: {BASE_DIR.resolve()}")

[CONFIG] MotionSense | window=5s (250 samples) | step=125 samples (50% overlap) | fs=50Hz
[DATA] Attempting download from: https://github.com/mmalekzadeh/motion-sense/raw/master/data/A_DeviceMotion_data.zip
[DATA] Download successful.
[DATA] Extracted to motionsense_outputs\raw_data
[DATA] Located 18 activity trial folders.
[DATA] Loaded 360 subject-trial files. Total samples: 1412865
[DATA] Activity distribution:
label
Walking       344288
Sitting       338778
Standing      306427
Upstairs      157285
Jogging       134231
Downstairs    131856
Name: count, dtype: int64
[SEGMENT] Generating sliding windows...
[SEGMENT] Total windows generated: 10765
[FEATURES] Extracting features from each window (this may take a few minutes)...
  ...500/10765 windows processed
  ...1000/10765 windows processed
  ...1500/10765 windows processed
  ...2000/10765 windows processed
  ...2500/10765 windows processed
  ...3000/10765 windows processed
  ...3500/10765 windows processed
  ...4000/10765 windows p

In [2]:
# ============================================================================
# MOTIONSENSE DATASET — MODULE 2
# TinyML-Oriented Deployment Analysis
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# READ-ONLY with respect to Module 1. This script:
#   - Loads trained models, scaler, label encoder, feature/split data, and
#     metrics produced by Module 1 from ./motionsense_outputs/
#   - Performs NO retraining, NO preprocessing, NO re-evaluation
#   - Writes ALL new outputs to ./motionsense_tinyml/{results,figures,reports}/
#
# STANDARDIZED TinyML METHODOLOGY (fixed for ALL datasets, identical to
# PAMAP2 and MHEALTH)
# --------------------------------------------------------------------------
# Model Size (KB):      actual on-disk size of the joblib-serialized model
#                        (proxy for the exported/quantization-ready artifact)
# RAM Estimate (KB):     heuristic runtime working-memory footprint, computed
#                        per model family (see estimate_ram_flash_kb()):
#                          - Logistic Regression / Linear SVM: coefficient +
#                            intercept matrix (n_classes x n_features), 4 bytes/value
#                          - Gaussian NB: per-class mean + variance + prior, 4 bytes/value
#                          - Decision Tree: node_count x 16 bytes/node
#                            (feature_idx, threshold, left_child, right_child)
#                          - Random Forest: sum of node_count x 16 bytes across all trees
#                          - XGBoost: sum of leaves/nodes across boosters x 16 bytes
#                          - KNN: entire training set stored (n_train x n_features x 4 bytes)
#                            — inherently RAM-heavy, reflecting real TinyML infeasibility
# Flash Estimate (KB):   on-disk serialized model size x 1.15 (10-15% packaging/
#                        alignment overhead), representing static storage in
#                        microcontroller flash/ROM
# Inference Latency:     mean wall-clock time (ms) to predict a single window,
#                        averaged over 200 held-out test samples, 5 repeats
# Training Time:         reused directly from Module 1 (metrics.json), no retraining
#
# Deployment Friendliness Score (0-100):
#   Weighted composite of five min-max normalized components (computed across
#   the 7 models of THIS dataset):
#     0.25 x (1 - norm(RAM))        # lower RAM is better
#     0.20 x (1 - norm(Flash))      # lower Flash is better
#     0.25 x (1 - norm(Latency))    # lower latency is better
#     0.15 x (1 - norm(ModelSize))  # lower size is better
#     0.15 x norm(TestAccuracy)     # higher accuracy is better
#   Score = 100 x weighted sum
#
# Deployment Tier (fixed thresholds, identical across datasets):
#   Tier 1 - Excellent   : Score >= 80  (fits low-end MCU, e.g. Arduino Uno/Nano, <=32KB RAM)
#   Tier 2 - Good        : 65 <= Score < 80  (mid-range MCU, e.g. ESP32/Cortex-M4, <=256KB RAM)
#   Tier 3 - Moderate    : 45 <= Score < 65  (Cortex-M7/Pico-class or constrained SBC)
#   Tier 4 - Poor        : Score < 45  (requires SBC/edge GPU, e.g. Raspberry Pi)
#
# Figures: 300 dpi PNG. Folder convention: ./motionsense_tinyml/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME     = "MotionSense"

SOURCE_DIR       = Path("./motionsense_outputs")       # READ-ONLY
SOURCE_MODELS    = SOURCE_DIR / "models"
SOURCE_RESULTS   = SOURCE_DIR / "results"
SOURCE_FEATURES  = SOURCE_DIR / "features"

OUTPUT_DIR       = Path("./motionsense_tinyml")        # WRITE-ONLY here
RESULTS_DIR      = OUTPUT_DIR / "results"
FIGURES_DIR      = OUTPUT_DIR / "figures"
REPORTS_DIR      = OUTPUT_DIR / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FIGURE_DPI       = 300
ENCODING         = "utf-8"
LATENCY_SAMPLES  = 200     # number of test samples used per latency measurement
LATENCY_REPEATS  = 5       # repeats, mean latency reported

MODEL_NAMES = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

BYTES_PER_NODE = 16   # heuristic: feature_idx(4) + threshold(4) + left(4) + right(4)

print(f"[CONFIG] {DATASET_NAME} — Module 2 (TinyML Deployment Analysis)")
print(f"[CONFIG] Reading Module 1 artifacts from: {SOURCE_DIR.resolve()}")
print(f"[CONFIG] Writing Module 2 artifacts to:  {OUTPUT_DIR.resolve()}")

# ============================================================================
# 2. VALIDATE MODULE 1 ARTIFACTS EXIST
# ============================================================================
required_paths = [
    SOURCE_MODELS / "motionsense_scaler.pkl",
    SOURCE_MODELS / "motionsense_label_encoder.pkl",
    SOURCE_RESULTS / "motionsense_train_test_data.npz",
    SOURCE_RESULTS / "motionsense_metrics.json",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required Module 1 artifacts:\n" + "\n".join(missing) +
        "\nPlease run Module 1 for MotionSense first."
    )

model_paths = {name: SOURCE_MODELS / f"motionsense_{name}.pkl" for name in MODEL_NAMES}
missing_models = [name for name, p in model_paths.items() if not p.exists()]
if missing_models:
    raise FileNotFoundError(f"Missing trained model files for: {missing_models}")

# ============================================================================
# 3. LOAD MODULE 1 ARTIFACTS (READ-ONLY)
# ============================================================================
scaler = joblib.load(SOURCE_MODELS / "motionsense_scaler.pkl")
label_encoder = joblib.load(SOURCE_MODELS / "motionsense_label_encoder.pkl")

split_data = np.load(SOURCE_RESULTS / "motionsense_train_test_data.npz")
X_train, X_test = split_data["X_train"], split_data["X_test"]
y_train, y_test = split_data["y_train"], split_data["y_test"]

with open(SOURCE_RESULTS / "motionsense_metrics.json", "r", encoding=ENCODING) as f:
    module1_metrics = json.load(f)

trained_models = {name: joblib.load(path) for name, path in model_paths.items()}

print(f"[LOAD] Loaded {len(trained_models)} trained models, scaler, label encoder, "
      f"and test set of shape {X_test.shape}")

# ============================================================================
# 4. HELPER FUNCTIONS
# ============================================================================
def model_disk_size_kb(model_path: Path) -> float:
    return os.path.getsize(model_path) / 1024.0


def estimate_ram_flash_kb(name: str, model, disk_size_kb: float,
                           n_features: int, n_train: int):
    """
    Heuristic RAM/Flash estimation per model family.
    RAM = runtime working-memory footprint required to hold the model's
          parameters in memory during inference (weights/tree structures/
          stored training data, as applicable).
    Flash = on-disk serialized size x 1.15 (packaging/alignment overhead),
            representing static storage footprint.
    """
    flash_kb = disk_size_kb * 1.15

    if name in ("Logistic_Regression", "Linear_SVM"):
        n_params = model.coef_.size + model.intercept_.size
        ram_kb = (n_params * 4) / 1024.0

    elif name == "Gaussian_NB":
        n_params = model.theta_.size + model.var_.size + model.class_prior_.size
        ram_kb = (n_params * 4) / 1024.0

    elif name == "Decision_Tree":
        n_nodes = model.tree_.node_count
        ram_kb = (n_nodes * BYTES_PER_NODE) / 1024.0

    elif name == "Random_Forest":
        total_nodes = sum(est.tree_.node_count for est in model.estimators_)
        ram_kb = (total_nodes * BYTES_PER_NODE) / 1024.0

    elif name == "XGBoost":
        try:
            df_trees = model.get_booster().trees_to_dataframe()
            total_nodes = len(df_trees)
        except Exception:
            total_nodes = model.n_estimators * (2 ** (model.max_depth + 1))
        ram_kb = (total_nodes * BYTES_PER_NODE) / 1024.0

    elif name == "KNN":
        # Must store the entire training set in memory
        ram_kb = (n_train * n_features * 4) / 1024.0
        flash_kb = ram_kb * 1.15  # training set also needs persistent storage

    else:
        ram_kb = disk_size_kb  # fallback

    return ram_kb, flash_kb


def measure_latency_ms(model, X: np.ndarray, n_samples: int, repeats: int) -> float:
    """Mean per-sample inference latency (ms), averaged over repeats."""
    n_samples = min(n_samples, X.shape[0])
    idx = np.random.RandomState(42).choice(X.shape[0], size=n_samples, replace=False)
    X_sub = X[idx]

    per_repeat_times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for i in range(n_samples):
            model.predict(X_sub[i:i + 1])
        t1 = time.perf_counter()
        per_repeat_times.append((t1 - t0) / n_samples)

    return float(np.mean(per_repeat_times) * 1000.0)  # ms


def minmax_norm(series: pd.Series) -> pd.Series:
    rng = series.max() - series.min()
    if rng < 1e-12:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / rng


def assign_tier(score: float) -> str:
    if score >= 80:
        return "Tier 1 - Excellent"
    elif score >= 65:
        return "Tier 2 - Good"
    elif score >= 45:
        return "Tier 3 - Moderate"
    else:
        return "Tier 4 - Poor"

# ============================================================================
# 5. COMPUTE TINYML METRICS FOR EACH MODEL
# ============================================================================
n_features = X_test.shape[1]
n_train = X_train.shape[0]

records = []
print("\n[TINYML] Profiling models (size, RAM, flash, latency)...")
for name in MODEL_NAMES:
    model = trained_models[name]
    disk_kb = model_disk_size_kb(model_paths[name])
    ram_kb, flash_kb = estimate_ram_flash_kb(name, model, disk_kb, n_features, n_train)
    latency_ms = measure_latency_ms(model, X_test, LATENCY_SAMPLES, LATENCY_REPEATS)

    m1 = module1_metrics[name]
    records.append({
        "Model": name,
        "Test_Accuracy": m1["accuracy"],
        "Macro_F1": m1["f1_macro"],
        "Training_Time_sec": m1["train_time_sec"],
        "Model_Size_KB": disk_kb,
        "RAM_Estimate_KB": ram_kb,
        "Flash_Estimate_KB": flash_kb,
        "Inference_Latency_ms": latency_ms,
    })
    print(f"  {name:22s} | size={disk_kb:9.2f} KB | RAM={ram_kb:10.2f} KB | "
          f"flash={flash_kb:9.2f} KB | latency={latency_ms:7.4f} ms")

tinyml_df = pd.DataFrame(records).set_index("Model")

# ============================================================================
# 6. DEPLOYMENT FRIENDLINESS SCORE + TIER
# ============================================================================
norm_ram     = minmax_norm(tinyml_df["RAM_Estimate_KB"])
norm_flash   = minmax_norm(tinyml_df["Flash_Estimate_KB"])
norm_latency = minmax_norm(tinyml_df["Inference_Latency_ms"])
norm_size    = minmax_norm(tinyml_df["Model_Size_KB"])
norm_acc     = minmax_norm(tinyml_df["Test_Accuracy"])

tinyml_df["Deployment_Friendliness_Score"] = 100 * (
    0.25 * (1 - norm_ram) +
    0.20 * (1 - norm_flash) +
    0.25 * (1 - norm_latency) +
    0.15 * (1 - norm_size) +
    0.15 * norm_acc
)
tinyml_df["Deployment_Tier"] = tinyml_df["Deployment_Friendliness_Score"].apply(assign_tier)

ranking_df = tinyml_df.sort_values("Deployment_Friendliness_Score", ascending=False).copy()
ranking_df.insert(0, "TinyML_Rank", range(1, len(ranking_df) + 1))

print("\n[SCORE] Deployment Friendliness ranking (best -> worst):")
print(ranking_df[["TinyML_Rank", "Deployment_Friendliness_Score", "Deployment_Tier"]])

# ============================================================================
# 7. SAVE TABLES
# ============================================================================
tinyml_df.to_csv(RESULTS_DIR / "motionsense_tinyml_comparison.csv", encoding=ENCODING)
ranking_df.to_csv(RESULTS_DIR / "motionsense_tinyml_ranking.csv", encoding=ENCODING)

with open(RESULTS_DIR / "motionsense_tinyml_metrics.json", "w", encoding=ENCODING) as f:
    json.dump(tinyml_df.reset_index().to_dict(orient="records"), f, indent=2, default=str)

print(f"\n[SAVE] Tables written to {RESULTS_DIR}")

# ============================================================================
# 8. FIGURES (300 dpi)
# ============================================================================
plot_order = ranking_df.index.tolist()

def bar_plot(series_name, ylabel, title, filename, color="#4C72B0", log_scale=False):
    plt.figure(figsize=(11, 6))
    values = ranking_df.loc[plot_order, series_name]
    bars = plt.bar(plot_order, values, color=color)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    if log_scale:
        plt.yscale("log")
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                  f"{val:,.2f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=FIGURE_DPI)
    plt.close()

bar_plot("Model_Size_KB", "Size (KB)",
          "MotionSense — Model Size by Classifier (sorted by deployment score)",
          "motionsense_tinyml_model_size.png", color="#4C72B0", log_scale=True)

bar_plot("RAM_Estimate_KB", "Estimated RAM (KB)",
          "MotionSense — Estimated RAM Usage by Classifier",
          "motionsense_tinyml_ram_estimate.png", color="#DD8452", log_scale=True)

bar_plot("Flash_Estimate_KB", "Estimated Flash (KB)",
          "MotionSense — Estimated Flash Usage by Classifier",
          "motionsense_tinyml_flash_estimate.png", color="#55A868", log_scale=True)

bar_plot("Inference_Latency_ms", "Latency (ms/sample)",
          "MotionSense — Inference Latency by Classifier",
          "motionsense_tinyml_latency.png", color="#C44E52", log_scale=True)

bar_plot("Deployment_Friendliness_Score", "Score (0-100)",
          "MotionSense — Deployment Friendliness Score by Classifier",
          "motionsense_tinyml_deployment_score.png", color="#8172B2")

# Combined multi-panel figure
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
panels = [
    ("Model_Size_KB", "Size (KB)", "Model Size", axes[0, 0], True),
    ("RAM_Estimate_KB", "RAM (KB)", "Estimated RAM", axes[0, 1], True),
    ("Inference_Latency_ms", "Latency (ms)", "Inference Latency", axes[1, 0], True),
    ("Deployment_Friendliness_Score", "Score", "Deployment Friendliness Score", axes[1, 1], False),
]
for col, ylabel, title, ax, logsc in panels:
    values = ranking_df.loc[plot_order, col]
    ax.bar(plot_order, values, color="#4C72B0")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    if logsc:
        ax.set_yscale("log")
plt.suptitle("MotionSense — TinyML Deployment Analysis Summary", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_tinyml_summary_panel.png", dpi=FIGURE_DPI)
plt.close()

# Tier distribution
plt.figure(figsize=(8, 6))
tier_counts = ranking_df["Deployment_Tier"].value_counts().reindex(
    ["Tier 1 - Excellent", "Tier 2 - Good", "Tier 3 - Moderate", "Tier 4 - Poor"]
).fillna(0)
sns.barplot(x=tier_counts.index, y=tier_counts.values, palette="viridis")
plt.ylabel("Number of Models")
plt.title("MotionSense — Deployment Tier Distribution")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_tinyml_tier_distribution.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 6 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 9. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MotionSense Dataset — Module 2 Report",
    "## TinyML-Oriented Deployment Analysis\n",
    "This module reuses trained models and evaluation artifacts from Module 1 "
    "without any retraining, preprocessing, or re-evaluation.\n",
    "### Methodology Summary",
    "- **Model Size**: on-disk joblib serialized size (KB)",
    "- **RAM Estimate**: heuristic runtime memory footprint per model family "
    "(coefficients/tree nodes/stored training samples)",
    "- **Flash Estimate**: serialized size x 1.15 packaging overhead",
    f"- **Inference Latency**: mean per-sample prediction time (ms), "
    f"{LATENCY_SAMPLES} samples x {LATENCY_REPEATS} repeats",
    "- **Training Time**: reused directly from Module 1 (no retraining)",
    "- **Deployment Friendliness Score**: weighted composite (25% RAM, 20% Flash, "
    "25% Latency, 15% Size, 15% Accuracy), min-max normalized across this dataset's 7 models",
    "- **Deployment Tier**: Tier 1 (>=80) Excellent, Tier 2 (65-80) Good, "
    "Tier 3 (45-65) Moderate, Tier 4 (<45) Poor\n",
    "## TinyML Comparison Table\n",
    tinyml_df.round(4).to_markdown(),
    "\n## Deployment Ranking (Best to Worst)\n",
    ranking_df[[
        "TinyML_Rank", "Deployment_Friendliness_Score", "Deployment_Tier",
        "Model_Size_KB", "RAM_Estimate_KB", "Flash_Estimate_KB",
        "Inference_Latency_ms", "Test_Accuracy"
    ]].round(4).to_markdown(),
    "\n## Recommended Model for Constrained MCU Deployment\n",
    f"**{ranking_df.index[0]}** - Score: {ranking_df.iloc[0]['Deployment_Friendliness_Score']:.2f} "
    f"({ranking_df.iloc[0]['Deployment_Tier']})",
]
with open(REPORTS_DIR / "motionsense_module2_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'motionsense_module2_report.md'}")

# ============================================================================
# 10. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 2 COMPLETE")
print("=" * 50)

[CONFIG] MotionSense — Module 2 (TinyML Deployment Analysis)
[CONFIG] Reading Module 1 artifacts from: C:\Users\Rohan Jaiswal\motionsense_outputs
[CONFIG] Writing Module 2 artifacts to:  C:\Users\Rohan Jaiswal\motionsense_tinyml
[LOAD] Loaded 7 trained models, scaler, label encoder, and test set of shape (2153, 180)

[TINYML] Profiling models (size, RAM, flash, latency)...
  Logistic_Regression    | size=     9.35 KB | RAM=      4.24 KB | flash=    10.75 KB | latency= 0.1144 ms
  Decision_Tree          | size=    25.90 KB | RAM=      3.52 KB | flash=    29.78 KB | latency= 0.0867 ms
  Random_Forest          | size= 11582.38 KB | RAM=   1636.34 KB | flash= 13319.74 KB | latency=42.5052 ms
  KNN                    | size= 12178.72 KB | RAM=   6055.31 KB | flash=  6963.61 KB | latency= 1.4472 ms
  Gaussian_NB            | size=    17.73 KB | RAM=      8.46 KB | flash=    20.38 KB | latency= 0.1966 ms
  Linear_SVM             | size=     9.22 KB | RAM=      4.24 KB | flash=    10.60 KB | l

In [3]:
# ============================================================================
# MOTIONSENSE DATASET — MODULE 3
# Robustness Analysis
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# READ-ONLY with respect to Module 1. This script:
#   - Loads trained models, scaler, label encoder, feature names, and test
#     split from ./motionsense_outputs/
#   - Performs NO retraining, NO preprocessing, NO re-segmentation
#   - Writes ALL new outputs to ./motionsense_robustness/{results,figures,reports}/
#
# STANDARDIZED ROBUSTNESS METHODOLOGY (fixed for ALL datasets, identical to
# PAMAP2 and MHEALTH)
# --------------------------------------------------------------------------
# 1) GAUSSIAN NOISE INJECTION
#    X_noisy = X_test + N(0, sigma^2) x per-feature std(X_test)
#    Levels tested: sigma in {0.1, 0.2, 0.3}
#
# 2) FEATURE REMOVAL (missingness)
#    Random subset of feature COLUMNS zeroed (scaled mean).
#    Levels: 10%, 20%, 30% of features removed (random_state=42)
#
# 3) SENSOR PERTURBATION (simulated sensor/channel failure)
#    All feature columns belonging to one MotionSense sensor GROUP are
#    zeroed out entirely, simulating that sensor stream failing. Unlike
#    MHEALTH (which has on-body sensor LOCATIONS: chest/ankle/arm),
#    MotionSense's DeviceMotion stream is a single phone-worn sensor with
#    four logical measurement groups, so the "sensor" dropout here targets
#    those groups instead:
#      - attitude          (roll, pitch, yaw)
#      - gravity           (x, y, z)
#      - rotationRate      (x, y, z)  [gyroscope]
#      - userAcceleration  (x, y, z)  [linear accelerometer]
#
# EVALUATION METRIC: Accuracy/F1 Retention % = (perturbed / clean) x 100
#
# OVERALL ROBUSTNESS SCORE: mean of Accuracy- and F1-Retention % averaged
# across all 10 perturbation conditions per model (3 noise + 3 feature-
# removal + 4 sensor-group dropout, one more than MHEALTH's 3-location
# scheme since MotionSense has 4 sensor groups instead of 3 body locations).
#
# Figures: 300 dpi PNG. Folder convention:
#    ./motionsense_robustness/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME     = "MotionSense"

SOURCE_DIR       = Path("./motionsense_outputs")        # READ-ONLY
SOURCE_MODELS    = SOURCE_DIR / "models"
SOURCE_RESULTS   = SOURCE_DIR / "results"
SOURCE_FEATURES  = SOURCE_DIR / "features"

OUTPUT_DIR       = Path("./motionsense_robustness")     # WRITE-ONLY here
RESULTS_DIR      = OUTPUT_DIR / "results"
FIGURES_DIR      = OUTPUT_DIR / "figures"
REPORTS_DIR      = OUTPUT_DIR / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FIGURE_DPI       = 300
ENCODING         = "utf-8"
RANDOM_STATE     = 42

NOISE_LEVELS          = [0.1, 0.2, 0.3]
FEATURE_REMOVAL_PCTS  = [0.10, 0.20, 0.30]
SENSOR_GROUPS         = ["attitude", "gravity", "rotationRate", "userAcceleration"]

MODEL_NAMES = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

print(f"[CONFIG] {DATASET_NAME} — Module 3 (Robustness Analysis)")
print(f"[CONFIG] Reading Module 1 artifacts from: {SOURCE_DIR.resolve()}")
print(f"[CONFIG] Writing Module 3 artifacts to:  {OUTPUT_DIR.resolve()}")

# ============================================================================
# 2. VALIDATE MODULE 1 ARTIFACTS EXIST
# ============================================================================
required_paths = [
    SOURCE_MODELS / "motionsense_scaler.pkl",
    SOURCE_MODELS / "motionsense_label_encoder.pkl",
    SOURCE_RESULTS / "motionsense_train_test_data.npz",
    SOURCE_FEATURES / "motionsense_feature_names.json",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required Module 1 artifacts:\n" + "\n".join(missing) +
        "\nPlease run Module 1 for MotionSense first."
    )

model_paths = {name: SOURCE_MODELS / f"motionsense_{name}.pkl" for name in MODEL_NAMES}
missing_models = [name for name, p in model_paths.items() if not p.exists()]
if missing_models:
    raise FileNotFoundError(f"Missing trained model files for: {missing_models}")

# ============================================================================
# 3. LOAD MODULE 1 ARTIFACTS (READ-ONLY)
# ============================================================================
scaler = joblib.load(SOURCE_MODELS / "motionsense_scaler.pkl")
label_encoder = joblib.load(SOURCE_MODELS / "motionsense_label_encoder.pkl")

split_data = np.load(SOURCE_RESULTS / "motionsense_train_test_data.npz")
X_test, y_test = split_data["X_test"], split_data["y_test"]

with open(SOURCE_FEATURES / "motionsense_feature_names.json", "r", encoding=ENCODING) as f:
    feature_names = json.load(f)

trained_models = {name: joblib.load(path) for name, path in model_paths.items()}

print(f"[LOAD] Loaded {len(trained_models)} trained models and test set "
      f"of shape {X_test.shape} ({len(feature_names)} features)")

assert len(feature_names) == X_test.shape[1], \
    "Feature name count mismatch with X_test columns."

# ============================================================================
# 4. MAP FEATURE COLUMNS TO SENSOR GROUPS (for sensor-dropout perturbation)
# ============================================================================
def sensor_group_of(feat_name: str) -> str:
    if feat_name.startswith("attitude."):
        return "attitude"
    if feat_name.startswith("gravity."):
        return "gravity"
    if feat_name.startswith("rotationRate."):
        return "rotationRate"
    if feat_name.startswith("userAcceleration."):
        return "userAcceleration"
    return "other"  # cross-axis correlation features already belong to one
                     # of the four groups by naming convention (e.g.
                     # "gravity_corr_xy"), so we also match on that prefix

def sensor_group_of_extended(feat_name: str) -> str:
    base = sensor_group_of(feat_name)
    if base != "other":
        return base
    for group in SENSOR_GROUPS:
        if feat_name.startswith(f"{group}_corr"):
            return group
    return "other"

location_map = {loc: [] for loc in SENSOR_GROUPS + ["other"]}
for idx, fname in enumerate(feature_names):
    location_map[sensor_group_of_extended(fname)].append(idx)

for loc in SENSOR_GROUPS:
    print(f"[MAP] Sensor group '{loc}': {len(location_map[loc])} feature columns")
if location_map["other"]:
    print(f"[MAP] Unmapped columns (not zeroed by sensor dropout): "
          f"{len(location_map['other'])}")

# ============================================================================
# 5. PERTURBATION FUNCTIONS
# ============================================================================
def add_gaussian_noise(X: np.ndarray, sigma_frac: float, seed: int) -> np.ndarray:
    rng = np.random.RandomState(seed)
    per_feature_std = X.std(axis=0, keepdims=True)
    noise = rng.normal(loc=0.0, scale=sigma_frac * per_feature_std, size=X.shape)
    return X + noise

def remove_features(X: np.ndarray, pct: float, seed: int) -> np.ndarray:
    rng = np.random.RandomState(seed)
    n_features = X.shape[1]
    n_remove = int(round(pct * n_features))
    cols_to_zero = rng.choice(n_features, size=n_remove, replace=False)
    X_pert = X.copy()
    X_pert[:, cols_to_zero] = 0.0   # 0.0 == scaled mean (StandardScaler)
    return X_pert

def perturb_sensor(X: np.ndarray, group: str) -> np.ndarray:
    X_pert = X.copy()
    cols = location_map.get(group, [])
    if cols:
        X_pert[:, cols] = 0.0
    return X_pert

# ============================================================================
# 6. BASELINE (CLEAN) EVALUATION
# ============================================================================
print("\n[BASELINE] Computing clean-test-set performance for each model...")
baseline = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    baseline[name] = {"accuracy": acc, "f1_macro": f1}
    print(f"  {name:22s} | clean acc={acc:.4f} | clean F1={f1:.4f}")

# ============================================================================
# 7. RUN ALL PERTURBATION CONDITIONS FOR ALL MODELS
# ============================================================================
records = []
print("\n[ROBUSTNESS] Running perturbation conditions...")

for name, model in trained_models.items():
    clean_acc = baseline[name]["accuracy"]
    clean_f1 = baseline[name]["f1_macro"]

    # --- 1) Gaussian noise injection ---
    for sigma in NOISE_LEVELS:
        X_pert = add_gaussian_noise(X_test, sigma, seed=RANDOM_STATE)
        y_pred = model.predict(X_pert)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        records.append({
            "Model": name, "Perturbation_Type": "Gaussian_Noise",
            "Condition": f"sigma={sigma}",
            "Accuracy": acc, "F1_Macro": f1,
            "Accuracy_Retention_%": 100 * acc / clean_acc if clean_acc > 0 else 0,
            "F1_Retention_%": 100 * f1 / clean_f1 if clean_f1 > 0 else 0,
        })

    # --- 2) Feature removal ---
    for pct in FEATURE_REMOVAL_PCTS:
        X_pert = remove_features(X_test, pct, seed=RANDOM_STATE)
        y_pred = model.predict(X_pert)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        records.append({
            "Model": name, "Perturbation_Type": "Feature_Removal",
            "Condition": f"{int(pct*100)}%_removed",
            "Accuracy": acc, "F1_Macro": f1,
            "Accuracy_Retention_%": 100 * acc / clean_acc if clean_acc > 0 else 0,
            "F1_Retention_%": 100 * f1 / clean_f1 if clean_f1 > 0 else 0,
        })

    # --- 3) Sensor group perturbation (simulated sensor stream failure) ---
    for group in SENSOR_GROUPS:
        X_pert = perturb_sensor(X_test, group)
        y_pred = model.predict(X_pert)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        records.append({
            "Model": name, "Perturbation_Type": "Sensor_Dropout",
            "Condition": f"{group}_disabled",
            "Accuracy": acc, "F1_Macro": f1,
            "Accuracy_Retention_%": 100 * acc / clean_acc if clean_acc > 0 else 0,
            "F1_Retention_%": 100 * f1 / clean_f1 if clean_f1 > 0 else 0,
        })

    print(f"  {name:22s} | {3 + 3 + len(SENSOR_GROUPS)} perturbation conditions evaluated")

robustness_df = pd.DataFrame(records)

# Add clean baseline rows for completeness/reference
baseline_records = [
    {"Model": name, "Perturbation_Type": "Baseline", "Condition": "clean_test_set",
     "Accuracy": b["accuracy"], "F1_Macro": b["f1_macro"],
     "Accuracy_Retention_%": 100.0, "F1_Retention_%": 100.0}
    for name, b in baseline.items()
]
robustness_df = pd.concat([pd.DataFrame(baseline_records), robustness_df], ignore_index=True)

# ============================================================================
# 8. OVERALL ROBUSTNESS SCORE + RANKING
# ============================================================================
perturbed_only = robustness_df[robustness_df["Perturbation_Type"] != "Baseline"]

summary_records = []
for name in MODEL_NAMES:
    subset = perturbed_only[perturbed_only["Model"] == name]
    overall_score = subset[["Accuracy_Retention_%", "F1_Retention_%"]].mean().mean()

    per_type = subset.groupby("Perturbation_Type")[
        ["Accuracy_Retention_%", "F1_Retention_%"]
    ].mean().mean(axis=1)

    summary_records.append({
        "Model": name,
        "Clean_Accuracy": baseline[name]["accuracy"],
        "Clean_F1_Macro": baseline[name]["f1_macro"],
        "Avg_Retention_Gaussian_Noise_%": per_type.get("Gaussian_Noise", np.nan),
        "Avg_Retention_Feature_Removal_%": per_type.get("Feature_Removal", np.nan),
        "Avg_Retention_Sensor_Dropout_%": per_type.get("Sensor_Dropout", np.nan),
        "Overall_Robustness_Score": overall_score,
    })

summary_df = pd.DataFrame(summary_records).set_index("Model")
ranking_df = summary_df.sort_values("Overall_Robustness_Score", ascending=False).copy()
ranking_df.insert(0, "Robustness_Rank", range(1, len(ranking_df) + 1))

print("\n[RANKING] Overall Robustness Ranking (best -> worst):")
print(ranking_df[["Robustness_Rank", "Overall_Robustness_Score"]])

# ============================================================================
# 9. SAVE TABLES
# ============================================================================
robustness_df.to_csv(RESULTS_DIR / "motionsense_robustness_results.csv", index=False, encoding=ENCODING)
ranking_df.to_csv(RESULTS_DIR / "motionsense_robustness_ranking.csv", encoding=ENCODING)

with open(RESULTS_DIR / "motionsense_robustness_metrics.json", "w", encoding=ENCODING) as f:
    json.dump({
        "detailed_results": robustness_df.to_dict(orient="records"),
        "summary_ranking": ranking_df.reset_index().to_dict(orient="records"),
    }, f, indent=2, default=str)

print(f"\n[SAVE] Tables written to {RESULTS_DIR}")

# ============================================================================
# 10. FIGURES (300 dpi)
# ============================================================================
plot_order = ranking_df.index.tolist()

# --- Overall robustness score bar chart ---
plt.figure(figsize=(11, 6))
values = ranking_df.loc[plot_order, "Overall_Robustness_Score"]
bars = plt.bar(plot_order, values, color="#4C72B0")
plt.ylabel("Overall Robustness Score (%)")
plt.title("MotionSense — Overall Robustness Score by Classifier")
plt.xticks(rotation=45, ha="right")
plt.axhline(100, color="gray", linestyle="--", linewidth=1, label="Clean baseline (100%)")
plt.legend()
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
              f"{val:.1f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_robustness_overall_score.png", dpi=FIGURE_DPI)
plt.close()

# --- Retention by perturbation type (grouped bar) ---
plt.figure(figsize=(13, 7))
retention_cols = [
    "Avg_Retention_Gaussian_Noise_%",
    "Avg_Retention_Feature_Removal_%",
    "Avg_Retention_Sensor_Dropout_%",
]
x = np.arange(len(plot_order))
width = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]
labels = ["Gaussian Noise", "Feature Removal", "Sensor Dropout"]
for i, (col, color, label) in enumerate(zip(retention_cols, colors, labels)):
    plt.bar(x + (i - 1) * width, ranking_df.loc[plot_order, col], width,
            label=label, color=color)
plt.xticks(x, plot_order, rotation=45, ha="right")
plt.ylabel("Retention (%)")
plt.title("MotionSense — Accuracy/F1 Retention by Perturbation Type")
plt.axhline(100, color="gray", linestyle="--", linewidth=1)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_robustness_by_perturbation_type.png", dpi=FIGURE_DPI)
plt.close()

# --- Noise-level degradation curves ---
plt.figure(figsize=(10, 6))
noise_subset = robustness_df[robustness_df["Perturbation_Type"] == "Gaussian_Noise"]
for name in MODEL_NAMES:
    sub = noise_subset[noise_subset["Model"] == name].copy()
    sub["sigma"] = sub["Condition"].str.replace("sigma=", "").astype(float)
    sub = sub.sort_values("sigma")
    plt.plot(sub["sigma"], sub["Accuracy_Retention_%"], marker="o", label=name)
plt.xlabel("Gaussian Noise Level (sigma, fraction of feature std)")
plt.ylabel("Accuracy Retention (%)")
plt.title("MotionSense — Accuracy Retention vs Gaussian Noise Level")
plt.axhline(100, color="gray", linestyle="--", linewidth=1)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_robustness_noise_curve.png", dpi=FIGURE_DPI)
plt.close()

# --- Feature-removal degradation curves ---
plt.figure(figsize=(10, 6))
fr_subset = robustness_df[robustness_df["Perturbation_Type"] == "Feature_Removal"]
for name in MODEL_NAMES:
    sub = fr_subset[fr_subset["Model"] == name].copy()
    sub["pct"] = sub["Condition"].str.replace("%_removed", "").astype(float)
    sub = sub.sort_values("pct")
    plt.plot(sub["pct"], sub["Accuracy_Retention_%"], marker="o", label=name)
plt.xlabel("Feature Columns Removed (%)")
plt.ylabel("Accuracy Retention (%)")
plt.title("MotionSense — Accuracy Retention vs Feature Removal Percentage")
plt.axhline(100, color="gray", linestyle="--", linewidth=1)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_robustness_feature_removal_curve.png", dpi=FIGURE_DPI)
plt.close()

# --- Sensor group dropout heatmap ---
plt.figure(figsize=(9, 7))
sd_subset = robustness_df[robustness_df["Perturbation_Type"] == "Sensor_Dropout"]
pivot = sd_subset.pivot(index="Model", columns="Condition", values="Accuracy_Retention_%")
pivot = pivot.reindex(plot_order)
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100,
            cbar_kws={"label": "Accuracy Retention (%)"})
plt.title("MotionSense — Accuracy Retention under Simulated Sensor-Group Dropout")
plt.ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_robustness_sensor_dropout_heatmap.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 5 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 11. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MotionSense Dataset — Module 3 Report",
    "## Robustness Analysis\n",
    "This module reuses trained models and the held-out test split from "
    "Module 1 without any retraining, preprocessing, or re-segmentation.\n",
    "### Methodology Summary",
    f"- **Gaussian Noise Injection**: additive noise at sigma in "
    f"{NOISE_LEVELS} (fraction of per-feature std)",
    f"- **Feature Removal**: {[f'{int(p*100)}%' for p in FEATURE_REMOVAL_PCTS]} "
    "of feature columns zeroed (random_state=42, identical subset across models per level)",
    f"- **Sensor Perturbation**: simulated failure of the {SENSOR_GROUPS} "
    "MotionSense DeviceMotion sensor groups (all associated feature columns zeroed). "
    "This adapts MHEALTH/PAMAP2's on-body sensor-location dropout to MotionSense's "
    "single-device four-channel-group structure.",
    "- **Retention Metric**: `100 x perturbed_metric / clean_metric` for both "
    "accuracy and macro-F1",
    "- **Overall Robustness Score**: mean of accuracy- and F1-retention across "
    "all 10 perturbation conditions per model (3 noise + 3 feature-removal + "
    "4 sensor-group dropout)\n",
    "## Clean Baseline Performance\n",
    pd.DataFrame(baseline).T.round(4).to_markdown(),
    "\n## Robustness Summary & Ranking (Best to Worst)\n",
    ranking_df.round(2).to_markdown(),
    "\n## Full Perturbation Results\n",
    "See `motionsense_robustness_results.csv` for the complete per-condition table "
    "(10 conditions x 7 models = 70 rows, plus 7 clean-baseline rows).\n",
    "## Most Robust Model\n",
    f"**{ranking_df.index[0]}** - Overall Robustness Score: "
    f"{ranking_df.iloc[0]['Overall_Robustness_Score']:.2f}%",
]
with open(REPORTS_DIR / "motionsense_module3_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'motionsense_module3_report.md'}")

# ============================================================================
# 12. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 3 COMPLETE")
print("=" * 50)

[CONFIG] MotionSense — Module 3 (Robustness Analysis)
[CONFIG] Reading Module 1 artifacts from: C:\Users\Rohan Jaiswal\motionsense_outputs
[CONFIG] Writing Module 3 artifacts to:  C:\Users\Rohan Jaiswal\motionsense_robustness
[LOAD] Loaded 7 trained models and test set of shape (2153, 180) (180 features)
[MAP] Sensor group 'attitude': 45 feature columns
[MAP] Sensor group 'gravity': 45 feature columns
[MAP] Sensor group 'rotationRate': 45 feature columns
[MAP] Sensor group 'userAcceleration': 45 feature columns

[BASELINE] Computing clean-test-set performance for each model...
  Logistic_Regression    | clean acc=0.9889 | clean F1=0.9841
  Decision_Tree          | clean acc=0.9786 | clean F1=0.9695
  Random_Forest          | clean acc=0.9921 | clean F1=0.9881
  KNN                    | clean acc=0.9963 | clean F1=0.9948
  Gaussian_NB            | clean acc=0.9257 | clean F1=0.9068
  Linear_SVM             | clean acc=0.9902 | clean F1=0.9859
  XGBoost                | clean acc=0.9930 

In [4]:
# ============================================================================
# MOTIONSENSE DATASET — MODULE 4
# Statistical Validation
# Standardized HAR Framework (identical methodology across all 5 datasets)
# ============================================================================
#
# READ-ONLY with respect to Module 1. This script:
#   - Loads test-set predictions, ground truth, and metrics produced by
#     Module 1 from ./motionsense_outputs/
#   - Performs NO retraining, NO preprocessing, NO re-evaluation
#   - Writes ALL new outputs to ./motionsense_statistics/{results,figures,reports}/
#
# STANDARDIZED STATISTICAL METHODOLOGY (fixed for ALL datasets, identical to
# PAMAP2 and MHEALTH)
# --------------------------------------------------------------------------
# 1) WILSON 95% CONFIDENCE INTERVALS (per model, on test-set accuracy)
# 2) COCHRAN'S Q TEST (equal accuracy across all 7 models)
# 3) PAIRWISE MCNEMAR TESTS (21 pairs, exact binomial or chi-square corrected)
# 4) HOLM-BONFERRONI CORRECTION (alpha=0.05, applied to 21 McNemar p-values)
# 5) STATISTICAL SUMMARY & RANKING (accuracy, CI width, significant wins)
#
# Figures: 300 dpi PNG. Folder convention:
#    ./motionsense_statistics/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import json
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats as sp_stats

try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels", "--quiet"])
    from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATASET_NAME     = "MotionSense"

SOURCE_DIR       = Path("./motionsense_outputs")        # READ-ONLY
SOURCE_RESULTS   = SOURCE_DIR / "results"

OUTPUT_DIR       = Path("./motionsense_statistics")     # WRITE-ONLY here
RESULTS_DIR      = OUTPUT_DIR / "results"
FIGURES_DIR      = OUTPUT_DIR / "figures"
REPORTS_DIR      = OUTPUT_DIR / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FIGURE_DPI       = 300
ENCODING         = "utf-8"
ALPHA            = 0.05
Z_95             = 1.959963985

MODEL_NAMES = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

print(f"[CONFIG] {DATASET_NAME} — Module 4 (Statistical Validation)")
print(f"[CONFIG] Reading Module 1 artifacts from: {SOURCE_DIR.resolve()}")
print(f"[CONFIG] Writing Module 4 artifacts to:  {OUTPUT_DIR.resolve()}")

# ============================================================================
# 2. VALIDATE MODULE 1 ARTIFACTS EXIST
# ============================================================================
required_paths = [
    SOURCE_RESULTS / "motionsense_test_predictions.csv",
    SOURCE_RESULTS / "motionsense_metrics.json",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required Module 1 artifacts:\n" + "\n".join(missing) +
        "\nPlease run Module 1 for MotionSense first."
    )

# ============================================================================
# 3. LOAD MODULE 1 ARTIFACTS (READ-ONLY)
# ============================================================================
pred_df = pd.read_csv(SOURCE_RESULTS / "motionsense_test_predictions.csv", encoding=ENCODING)

with open(SOURCE_RESULTS / "motionsense_metrics.json", "r", encoding=ENCODING) as f:
    module1_metrics = json.load(f)

missing_cols = [c for c in MODEL_NAMES + ["y_true"] if c not in pred_df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in test_predictions.csv: {missing_cols}")

y_true = pred_df["y_true"].values
n_samples = len(y_true)

correct_matrix = pd.DataFrame({
    name: (pred_df[name].values == y_true).astype(int) for name in MODEL_NAMES
})

print(f"[LOAD] Loaded predictions for {len(MODEL_NAMES)} models on "
      f"{n_samples} held-out test samples.")

# ============================================================================
# 4. WILSON 95% CONFIDENCE INTERVALS
# ============================================================================
def wilson_ci(successes: int, n: int, z: float = Z_95):
    p_hat = successes / n
    denom = 1 + (z ** 2) / n
    center = (p_hat + (z ** 2) / (2 * n)) / denom
    margin = (z / denom) * np.sqrt((p_hat * (1 - p_hat) / n) + (z ** 2) / (4 * n ** 2))
    return p_hat, max(0.0, center - margin), min(1.0, center + margin)

print("\n[WILSON] Computing 95% Wilson confidence intervals...")
wilson_records = []
for name in MODEL_NAMES:
    successes = int(correct_matrix[name].sum())
    p_hat, lower, upper = wilson_ci(successes, n_samples)
    wilson_records.append({
        "Model": name,
        "Accuracy": p_hat,
        "Wilson_CI_Lower": lower,
        "Wilson_CI_Upper": upper,
        "CI_Width": upper - lower,
        "N_Correct": successes,
        "N_Total": n_samples,
    })
    print(f"  {name:22s} | acc={p_hat:.4f} | 95% CI=[{lower:.4f}, {upper:.4f}]")

wilson_df = pd.DataFrame(wilson_records).set_index("Model")
wilson_df.to_csv(RESULTS_DIR / "motionsense_wilson_confidence_intervals.csv", encoding=ENCODING)

# ============================================================================
# 5. COCHRAN'S Q TEST
# ============================================================================
print("\n[COCHRAN'S Q] Testing H0: all models have equal accuracy...")

def cochrans_q(binary_matrix: pd.DataFrame):
    k = binary_matrix.shape[1]
    col_sums = binary_matrix.sum(axis=0).values
    row_sums = binary_matrix.sum(axis=1).values
    L = col_sums.sum()

    numerator = k * (k - 1) * np.sum((col_sums - L / k) ** 2)
    denominator = k * L - np.sum(row_sums ** 2)
    Q = numerator / denominator if denominator != 0 else 0.0
    df = k - 1
    p_value = 1 - sp_stats.chi2.cdf(Q, df)
    return Q, df, p_value

Q_stat, Q_df, Q_pvalue = cochrans_q(correct_matrix)
cochran_result = {
    "Q_statistic": Q_stat,
    "degrees_of_freedom": Q_df,
    "p_value": Q_pvalue,
    "significant_at_0.05": bool(Q_pvalue < ALPHA),
    "interpretation": (
        "Reject H0: at least one model's accuracy differs significantly from the others."
        if Q_pvalue < ALPHA else
        "Fail to reject H0: no significant difference in accuracy detected across models."
    ),
}
print(f"  Q = {Q_stat:.4f}, df = {Q_df}, p = {Q_pvalue:.6f}")
print(f"  {cochran_result['interpretation']}")

with open(RESULTS_DIR / "motionsense_cochrans_q_results.json", "w", encoding=ENCODING) as f:
    json.dump(cochran_result, f, indent=2)

# ============================================================================
# 6. PAIRWISE MCNEMAR TESTS
# ============================================================================
print("\n[MCNEMAR] Running pairwise McNemar tests for all model pairs...")
mcnemar_records = []

for model_a, model_b in combinations(MODEL_NAMES, 2):
    correct_a = correct_matrix[model_a].values
    correct_b = correct_matrix[model_b].values

    both_correct = int(np.sum((correct_a == 1) & (correct_b == 1)))
    a_only = int(np.sum((correct_a == 1) & (correct_b == 0)))
    b_only = int(np.sum((correct_a == 0) & (correct_b == 1)))
    both_wrong = int(np.sum((correct_a == 0) & (correct_b == 0)))

    contingency = [[both_correct, a_only], [b_only, both_wrong]]
    discordant = a_only + b_only
    exact = discordant < 25

    result = mcnemar(contingency, exact=exact, correction=True)
    mcnemar_records.append({
        "Model_A": model_a, "Model_B": model_b,
        "A_only_correct": a_only, "B_only_correct": b_only,
        "Discordant_pairs": discordant,
        "Statistic": result.statistic,
        "p_value": result.pvalue,
        "Test_Type": "exact_binomial" if exact else "chi_square_corrected",
    })

mcnemar_df = pd.DataFrame(mcnemar_records)
print(f"  Completed {len(mcnemar_df)} pairwise tests.")

# ============================================================================
# 7. HOLM-BONFERRONI CORRECTION
# ============================================================================
print("\n[HOLM-BONFERRONI] Applying multiple-comparison correction...")

def holm_bonferroni(p_values: np.ndarray, alpha: float = 0.05):
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        factor = m - rank
        adj_p = p_values[idx] * factor
        running_max = max(running_max, adj_p)
        adjusted[idx] = min(running_max, 1.0)
    significant = adjusted < alpha
    return adjusted, significant

adjusted_pvals, significant_flags = holm_bonferroni(mcnemar_df["p_value"].values, ALPHA)
mcnemar_df["Holm_Adjusted_p_value"] = adjusted_pvals
mcnemar_df["Significant_after_Holm"] = significant_flags

n_significant = int(significant_flags.sum())
print(f"  {n_significant} / {len(mcnemar_df)} pairs remain significant after "
      f"Holm-Bonferroni correction (alpha={ALPHA}).")

mcnemar_df.to_csv(RESULTS_DIR / "motionsense_pairwise_mcnemar_holm.csv", index=False, encoding=ENCODING)

# ============================================================================
# 8. STATISTICAL SUMMARY & MODEL RANKING
# ============================================================================
print("\n[SUMMARY] Building statistical summary and ranking...")

win_counts = {name: 0 for name in MODEL_NAMES}
for _, row in mcnemar_df.iterrows():
    if row["Significant_after_Holm"]:
        if row["A_only_correct"] > row["B_only_correct"]:
            win_counts[row["Model_A"]] += 1
        elif row["B_only_correct"] > row["A_only_correct"]:
            win_counts[row["Model_B"]] += 1

summary_records = []
for name in MODEL_NAMES:
    summary_records.append({
        "Model": name,
        "Accuracy": wilson_df.loc[name, "Accuracy"],
        "Wilson_CI_Lower": wilson_df.loc[name, "Wilson_CI_Lower"],
        "Wilson_CI_Upper": wilson_df.loc[name, "Wilson_CI_Upper"],
        "CI_Width": wilson_df.loc[name, "CI_Width"],
        "Significant_Pairwise_Wins": win_counts[name],
        "Macro_F1": module1_metrics[name]["f1_macro"],
        "CV_Accuracy_Mean": module1_metrics[name]["cv_accuracy_mean"],
        "CV_Accuracy_Std": module1_metrics[name]["cv_accuracy_std"],
    })

summary_df = pd.DataFrame(summary_records).set_index("Model")
ranking_df = summary_df.sort_values(
    ["Significant_Pairwise_Wins", "Accuracy"], ascending=[False, False]
).copy()
ranking_df.insert(0, "Statistical_Rank", range(1, len(ranking_df) + 1))

ranking_df.to_csv(RESULTS_DIR / "motionsense_statistical_summary.csv", encoding=ENCODING)

with open(RESULTS_DIR / "motionsense_statistical_validation_full.json", "w", encoding=ENCODING) as f:
    json.dump({
        "wilson_confidence_intervals": wilson_df.reset_index().to_dict(orient="records"),
        "cochrans_q_test": cochran_result,
        "pairwise_mcnemar_holm": mcnemar_df.to_dict(orient="records"),
        "statistical_summary_ranking": ranking_df.reset_index().to_dict(orient="records"),
    }, f, indent=2, default=str)

print("\n[RANKING] Statistical ranking (best -> worst):")
print(ranking_df[["Statistical_Rank", "Accuracy", "Significant_Pairwise_Wins"]])

# ============================================================================
# 9. FIGURES (300 dpi)
# ============================================================================
plot_order = ranking_df.index.tolist()

# --- Wilson CI forest plot ---
plt.figure(figsize=(10, 7))
y_pos = np.arange(len(plot_order))
accs = wilson_df.loc[plot_order, "Accuracy"].values
lowers = wilson_df.loc[plot_order, "Wilson_CI_Lower"].values
uppers = wilson_df.loc[plot_order, "Wilson_CI_Upper"].values
errors = np.vstack([accs - lowers, uppers - accs])
plt.errorbar(accs, y_pos, xerr=errors, fmt="o", color="#4C72B0",
             ecolor="#4C72B0", elinewidth=2, capsize=4, markersize=7)
plt.yticks(y_pos, plot_order)
plt.xlabel("Accuracy (95% Wilson CI)")
plt.title("MotionSense — Wilson 95% Confidence Intervals by Classifier")
plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_wilson_ci_forest_plot.png", dpi=FIGURE_DPI)
plt.close()

# --- McNemar p-value heatmap (Holm-adjusted) ---
plt.figure(figsize=(9, 8))
pval_matrix = pd.DataFrame(np.ones((len(MODEL_NAMES), len(MODEL_NAMES))),
                           index=MODEL_NAMES, columns=MODEL_NAMES)
for _, row in mcnemar_df.iterrows():
    pval_matrix.loc[row["Model_A"], row["Model_B"]] = row["Holm_Adjusted_p_value"]
    pval_matrix.loc[row["Model_B"], row["Model_A"]] = row["Holm_Adjusted_p_value"]
np.fill_diagonal(pval_matrix.values, np.nan)

sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f", cmap="coolwarm_r",
            vmin=0, vmax=1, cbar_kws={"label": "Holm-adjusted p-value"},
            mask=pval_matrix.isna())
plt.title("MotionSense — Pairwise McNemar Test (Holm-Bonferroni adjusted p-values)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_mcnemar_holm_heatmap.png", dpi=FIGURE_DPI)
plt.close()

# --- Significant pairwise wins bar chart ---
plt.figure(figsize=(10, 6))
wins = ranking_df.loc[plot_order, "Significant_Pairwise_Wins"]
bars = plt.bar(plot_order, wins, color="#55A868")
plt.ylabel("Number of Significant Pairwise Wins (Holm-adjusted)")
plt.title("MotionSense — Statistically Significant Pairwise Wins by Classifier")
plt.xticks(rotation=45, ha="right")
for bar, val in zip(bars, wins):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
              f"{int(val)}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_significant_wins.png", dpi=FIGURE_DPI)
plt.close()

# --- Accuracy vs CI width scatter ---
plt.figure(figsize=(9, 7))
plt.scatter(wilson_df.loc[plot_order, "Accuracy"], wilson_df.loc[plot_order, "CI_Width"],
            s=100, color="#C44E52")
for name in plot_order:
    plt.annotate(name, (wilson_df.loc[name, "Accuracy"], wilson_df.loc[name, "CI_Width"]),
                 fontsize=8, xytext=(5, 5), textcoords="offset points")
plt.xlabel("Accuracy")
plt.ylabel("Wilson 95% CI Width")
plt.title("MotionSense — Accuracy vs Confidence Interval Width")
plt.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "motionsense_accuracy_vs_ci_width.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 4 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 10. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# MotionSense Dataset — Module 4 Report",
    "## Statistical Validation\n",
    "This module reuses test-set predictions and metrics from Module 1 "
    "without any retraining, preprocessing, or re-evaluation.\n",
    "### Methodology Summary",
    "- **Wilson 95% Confidence Intervals**: computed per model on test-set "
    "accuracy (Bernoulli correct/incorrect trials)",
    "- **Cochran's Q Test**: tests equality of accuracy across all 7 models "
    "on the same test instances",
    "- **Pairwise McNemar Tests**: 21 pairwise comparisons using discordant "
    "correct/incorrect pairs (exact binomial if discordant count < 25, else "
    "chi-square with continuity correction)",
    "- **Holm-Bonferroni Correction**: applied to the 21 McNemar p-values, "
    f"alpha = {ALPHA}",
    "- **Statistical Ranking**: models ranked by number of statistically "
    "significant pairwise wins, then by raw accuracy\n",
    "## Wilson 95% Confidence Intervals\n",
    wilson_df.round(4).to_markdown(),
    "\n## Cochran's Q Test\n",
    f"- Q statistic: {Q_stat:.4f}",
    f"- Degrees of freedom: {Q_df}",
    f"- p-value: {Q_pvalue:.6f}",
    f"- {cochran_result['interpretation']}\n",
    "## Pairwise McNemar Tests (Holm-Bonferroni Adjusted)\n",
    mcnemar_df.round(4).to_markdown(index=False),
    "\n## Statistical Summary & Ranking (Best to Worst)\n",
    ranking_df.round(4).to_markdown(),
    "\n## Statistically Best-Validated Model\n",
    f"**{ranking_df.index[0]}** - Accuracy: {ranking_df.iloc[0]['Accuracy']:.4f}, "
    f"95% CI: [{ranking_df.iloc[0]['Wilson_CI_Lower']:.4f}, "
    f"{ranking_df.iloc[0]['Wilson_CI_Upper']:.4f}], "
    f"Significant pairwise wins: {int(ranking_df.iloc[0]['Significant_Pairwise_Wins'])}/6",
]
with open(REPORTS_DIR / "motionsense_module4_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'motionsense_module4_report.md'}")

# ============================================================================
# 11. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 4 COMPLETE")
print("=" * 50)

[CONFIG] MotionSense — Module 4 (Statistical Validation)
[CONFIG] Reading Module 1 artifacts from: C:\Users\Rohan Jaiswal\motionsense_outputs
[CONFIG] Writing Module 4 artifacts to:  C:\Users\Rohan Jaiswal\motionsense_statistics
[LOAD] Loaded predictions for 7 models on 2153 held-out test samples.

[WILSON] Computing 95% Wilson confidence intervals...
  Logistic_Regression    | acc=0.9889 | 95% CI=[0.9835, 0.9925]
  Decision_Tree          | acc=0.9786 | 95% CI=[0.9716, 0.9839]
  Random_Forest          | acc=0.9921 | 95% CI=[0.9874, 0.9951]
  KNN                    | acc=0.9963 | 95% CI=[0.9927, 0.9981]
  Gaussian_NB            | acc=0.9257 | 95% CI=[0.9138, 0.9360]
  Linear_SVM             | acc=0.9902 | 95% CI=[0.9851, 0.9936]
  XGBoost                | acc=0.9930 | 95% CI=[0.9885, 0.9958]

[COCHRAN'S Q] Testing H0: all models have equal accuracy...
  Q = 524.6560, df = 6, p = 0.000000
  Reject H0: at least one model's accuracy differs significantly from the others.

[MCNEMAR] Running

In [1]:
# ============================================================================
# MODULE 5 — CROSS-DATASET COMPARATIVE ANALYSIS
# Standardized HAR Framework — Final Synthesis Across All 5 Datasets
# (UCI HAR, WISDM, PAMAP2, MHEALTH, MotionSense)
# ============================================================================
#
# READ-ONLY / NO RETRAINING. This script:
#   - For UCI HAR, WISDM, and PAMAP2: uses HARDCODED values transcribed
#     directly from console outputs and uploaded CSVs provided in this
#     conversation. Any metric not present in those sources is explicitly
#     marked "N/A" — never estimated or fabricated.
#   - PAMAP2's TinyML Deployment Friendliness Score/Tier are COMPUTED from
#     PAMAP2's own console-reported raw metrics (size, RAM, flash, latency,
#     accuracy) using the project's standard scoring formula.
#   - PAMAP2's Overall Robustness Score is COMPUTED as worst-case retention
#     (100 x worst_case_accuracy / baseline_accuracy) from the console-
#     reported baseline and worst-case corrupted accuracy per model. This
#     is a DIFFERENT metric granularity than the other 4 datasets (which
#     report MEAN retention across 9-10 perturbation conditions), so it is
#     clearly labeled as such throughout.
#   - PAMAP2's Significant_Pairwise_Wins are derived from the actual
#     uploaded holm_adjusted_results.csv, using the same win-counting rule
#     applied to every other dataset.
#   - For MHEALTH, MotionSense: reads already-saved CSV/JSON artifacts
#     from their existing output folders on disk. No preprocessing, no
#     re-segmentation, no re-training, no re-evaluation.
#   - Writes ALL new outputs to ./cross_dataset_comparison/{results,figures,reports}/
#   - Does NOT modify any existing dataset output folder.
#
# METHODOLOGY
# --------------------------------------------------------------------------
# Four comparison categories, one canonical 7-model set:
#   Logistic_Regression, Decision_Tree, Random_Forest, KNN, Gaussian_NB,
#   Linear_SVM, XGBoost
#
# For each category and each dataset, models are ranked 1 (best) .. 7 (worst)
# on a single primary metric:
#   - Performance   : Test Accuracy (descending)
#   - TinyML        : Deployment Friendliness Score (descending)
#   - Robustness    : Overall Robustness Score (descending)
#   - Statistical   : Significant Pairwise Wins (descending), tie-break by
#                     Accuracy (descending) — identical tie-break rule used
#                     in every per-dataset Module 4 script in this project
#
# Average rank per model per category = mean rank across the datasets where
# that category's primary metric is available for that model (NaN-safe).
#
# Overall combined rank per model = mean of the four category-average ranks
# (a "rank of ranks", giving equal weight to each analysis category).
#
# Figures: 300 dpi PNG. Folder convention:
#    ./cross_dataset_comparison/{results,figures,reports}/
# ENCODING: All text/CSV writes use encoding="utf-8" explicitly (Windows-safe).
# ============================================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
ENCODING    = "utf-8"
FIGURE_DPI  = 300

OUTPUT_DIR   = Path("./cross_dataset_comparison")
RESULTS_DIR  = OUTPUT_DIR / "results"
FIGURES_DIR  = OUTPUT_DIR / "figures"
REPORTS_DIR  = OUTPUT_DIR / "reports"
for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CANONICAL_MODELS = [
    "Logistic_Regression", "Decision_Tree", "Random_Forest",
    "KNN", "Gaussian_NB", "Linear_SVM", "XGBoost"
]

DATASETS = ["UCI_HAR", "WISDM", "PAMAP2", "MHEALTH", "MotionSense"]

# On-disk locations for the two datasets still loaded from files.
MHEALTH_MODULE1_DIR   = Path("./mhealth_outputs/results")
MHEALTH_MODULE2_DIR   = Path("./mhealth_tinyml/results")
MHEALTH_MODULE3_DIR   = Path("./mhealth_robustness/results")
MHEALTH_MODULE4_DIR   = Path("./mhealth_statistics/results")

MOTIONSENSE_MODULE1_DIR = Path("./motionsense_outputs/results")
MOTIONSENSE_MODULE2_DIR = Path("./motionsense_tinyml/results")
MOTIONSENSE_MODULE3_DIR = Path("./motionsense_robustness/results")
MOTIONSENSE_MODULE4_DIR = Path("./motionsense_statistics/results")

print("[CONFIG] Module 5 — Cross-Dataset Comparative Analysis")
print(f"[CONFIG] Writing all outputs to: {OUTPUT_DIR.resolve()}")
print("[CONFIG] UCI_HAR, WISDM, and PAMAP2 use hardcoded values transcribed from console output/uploaded CSVs.")
print("[CONFIG] MHEALTH and MotionSense are read from their saved output folders (read-only).")

# ============================================================================
# 2. MODEL NAME NORMALIZATION
# ============================================================================
_NAME_MAP = {
    "logistic_regression": "Logistic_Regression", "logisticregression": "Logistic_Regression",
    "decision_tree": "Decision_Tree", "decisiontree": "Decision_Tree",
    "random_forest": "Random_Forest", "randomforest": "Random_Forest",
    "knn": "KNN",
    "gaussian_nb": "Gaussian_NB", "gaussiannb": "Gaussian_NB",
    "gaussian_naive_bayes": "Gaussian_NB", "gaussiannaivebayes": "Gaussian_NB",
    "linear_svm": "Linear_SVM", "linearsvm": "Linear_SVM", "linearsvc": "Linear_SVM",
    "xgboost": "XGBoost",
}

def normalize_model_name(raw_name: str):
    if raw_name is None or (isinstance(raw_name, float) and np.isnan(raw_name)):
        return None
    key = str(raw_name).strip().replace(" ", "_").replace("-", "_").lower()
    return _NAME_MAP.get(key, None)


def safe_read_csv(path: Path):
    if not path.exists():
        print(f"  [WARN] File not found, skipping: {path}")
        return None
    try:
        return pd.read_csv(path, encoding=ENCODING)
    except Exception as e:
        print(f"  [WARN] Could not read {path}: {e}")
        return None


def safe_read_json(path: Path):
    if not path.exists():
        print(f"  [WARN] File not found, skipping: {path}")
        return None
    try:
        with open(path, "r", encoding=ENCODING) as f:
            return json.load(f)
    except Exception as e:
        print(f"  [WARN] Could not read {path}: {e}")
        return None


def compute_win_counts(mcnemar_raw: list, models: list) -> dict:
    """Shared win-counting logic used for every dataset's Statistical
    category: for each significant pair, the model with more discordant-
    correct predictions gets +1 win.
    Tuple format: (model_a, model_b, b_only, a_only, p_raw, p_holm, sig)
      b_only = Discordant (A wrong / B right)
      a_only = Discordant (A right / B wrong)
    """
    wins = {m: 0 for m in models}
    for model_a, model_b, b_only, a_only, p_raw, p_holm, sig in mcnemar_raw:
        if not sig:
            continue
        if a_only > b_only:
            wins[model_a] += 1
        elif b_only > a_only:
            wins[model_b] += 1
    return wins

# ============================================================================
# 3. HARDCODED UCI HAR AND WISDM DATA (transcribed verbatim from console output)
# ============================================================================

# --- 3a. Performance (Module 1 equivalent) ---
UCI_HAR_PERFORMANCE = {
    "Logistic_Regression": {"accuracy": 0.9552, "f1_macro": 0.9551, "cv_accuracy_mean": np.nan, "train_time_sec": 8.6309},
    "Decision_Tree":       {"accuracy": 0.8666, "f1_macro": 0.8660, "cv_accuracy_mean": np.nan, "train_time_sec": 5.8907},
    "Random_Forest":       {"accuracy": 0.9220, "f1_macro": 0.9219, "cv_accuracy_mean": np.nan, "train_time_sec": 8.8161},
    "KNN":                 {"accuracy": 0.8897, "f1_macro": 0.8889, "cv_accuracy_mean": np.nan, "train_time_sec": 0.0183},
    "Gaussian_NB":         {"accuracy": 0.7703, "f1_macro": 0.7688, "cv_accuracy_mean": np.nan, "train_time_sec": 0.0880},
    "Linear_SVM":          {"accuracy": 0.9620, "f1_macro": 0.9620, "cv_accuracy_mean": np.nan, "train_time_sec": 5.7955},
    "XGBoost":             {"accuracy": 0.9477, "f1_macro": 0.9476, "cv_accuracy_mean": np.nan, "train_time_sec": 55.3078},
}

WISDM_PERFORMANCE = {
    "Logistic_Regression": {"accuracy": 0.8799, "f1_macro": 0.8733, "cv_accuracy_mean": np.nan, "train_time_sec": 5.3167},
    "Decision_Tree":       {"accuracy": 0.9283, "f1_macro": 0.9275, "cv_accuracy_mean": np.nan, "train_time_sec": 2.3724},
    "Random_Forest":       {"accuracy": 0.9721, "f1_macro": 0.9715, "cv_accuracy_mean": np.nan, "train_time_sec": 28.2311},
    "KNN":                 {"accuracy": 0.9844, "f1_macro": 0.9842, "cv_accuracy_mean": np.nan, "train_time_sec": 0.0101},
    "Gaussian_NB":         {"accuracy": 0.7371, "f1_macro": 0.7475, "cv_accuracy_mean": np.nan, "train_time_sec": 0.0135},
    "Linear_SVM":          {"accuracy": 0.8691, "f1_macro": 0.8569, "cv_accuracy_mean": np.nan, "train_time_sec": 1.3279},
    "XGBoost":             {"accuracy": 0.9799, "f1_macro": 0.9798, "cv_accuracy_mean": np.nan, "train_time_sec": 13.2943},
}

# --- 3b. TinyML (Module 2 equivalent) ---
UCI_HAR_TINYML = {
    "Decision_Tree":       {"model_size_kb": 24.40,    "latency_ms": 0.2415,  "ram_kb": 12.20,    "flash_kb": 24.40,    "score": 96.84, "tier_raw": "Class 1 - Entry Cortex-M0/M0+"},
    "Logistic_Regression": {"model_size_kb": 27.21,    "latency_ms": 0.2773,  "ram_kb": 13.60,    "flash_kb": 27.21,    "score": 96.45, "tier_raw": "Class 1 - Entry Cortex-M0/M0+"},
    "Linear_SVM":          {"model_size_kb": 27.08,    "latency_ms": 0.3096,  "ram_kb": 13.54,    "flash_kb": 27.08,    "score": 88.36, "tier_raw": "Class 1 - Entry Cortex-M0/M0+"},
    "Gaussian_NB":         {"model_size_kb": 53.44,    "latency_ms": 1.1529,  "ram_kb": 26.72,    "flash_kb": 53.44,    "score": 87.20, "tier_raw": "Class 1 - Entry Cortex-M0/M0+"},
    "XGBoost":             {"model_size_kb": 1226.48,  "latency_ms": 1.9797,  "ram_kb": 613.24,   "flash_kb": 1226.48,  "score": 28.06, "tier_raw": "Not TinyML-suitable as-is"},
    "KNN":                 {"model_size_kb": 32280.69, "latency_ms": 11.3483, "ram_kb": 16140.35, "flash_kb": 32280.69, "score": 12.00, "tier_raw": "Not TinyML-suitable as-is"},
    "Random_Forest":       {"model_size_kb": 2905.67,  "latency_ms": 20.7487, "ram_kb": 1452.83,  "flash_kb": 2905.67,  "score": 8.00,  "tier_raw": "Not TinyML-suitable as-is"},
}

WISDM_TINYML = {
    "Logistic_Regression": {"model_size_kb": 2.37,     "latency_ms": 0.6353,  "ram_kb": 1.18,     "flash_kb": 2.37,     "score": 97.86, "tier_raw": "Class 0 - Ultra-constrained"},
    "Gaussian_NB":         {"model_size_kb": 3.76,     "latency_ms": 0.3118,  "ram_kb": 1.88,     "flash_kb": 3.76,     "score": 94.69, "tier_raw": "Class 0 - Ultra-constrained"},
    "Linear_SVM":          {"model_size_kb": 2.24,     "latency_ms": 0.1878,  "ram_kb": 1.12,     "flash_kb": 2.24,     "score": 91.21, "tier_raw": "Class 0 - Ultra-constrained"},
    "Decision_Tree":       {"model_size_kb": 207.06,   "latency_ms": 0.2660,  "ram_kb": 103.53,   "flash_kb": 207.06,   "score": 78.50, "tier_raw": "Class 2 - Mainstream Cortex-M4/ESP32"},
    "KNN":                 {"model_size_kb": 5371.07,  "latency_ms": 4.7903,  "ram_kb": 2685.54,  "flash_kb": 5371.07,  "score": 27.63, "tier_raw": "Not TinyML-suitable as-is"},
    "XGBoost":             {"model_size_kb": 3329.56,  "latency_ms": 2.6688,  "ram_kb": 1664.78,  "flash_kb": 3329.56,  "score": 25.99, "tier_raw": "Not TinyML-suitable as-is"},
    "Random_Forest":       {"model_size_kb": 44470.37, "latency_ms": 52.2930, "ram_kb": 22235.18, "flash_kb": 44470.37, "score": 8.00,  "tier_raw": "Not TinyML-suitable as-is"},
}

# --- 3c. Robustness (Module 3 equivalent) ---
# Only an AGGREGATE "Avg Retention Under Corruption" was printed for these two
# datasets — no breakdown by perturbation type. Converted to 0-100 scale.
UCI_HAR_ROBUSTNESS_OVERALL = {
    "Logistic_Regression": 98.106, "KNN": 97.624, "Linear_SVM": 96.974, "XGBoost": 96.529,
    "Random_Forest": 96.383, "Decision_Tree": 84.902, "Gaussian_NB": 69.683,
}
WISDM_ROBUSTNESS_OVERALL = {
    "KNN": 97.300, "Random_Forest": 97.185, "XGBoost": 96.110, "Gaussian_NB": 95.760,
    "Decision_Tree": 88.361, "Linear_SVM": 78.901, "Logistic_Regression": 77.456,
}

# --- 3d. Statistical Validation (Module 4 equivalent) ---
UCI_HAR_WILSON = {
    "Linear_SVM":          {"accuracy": 0.9620, "ci_lower": 0.9545, "ci_upper": 0.9683},
    "Logistic_Regression": {"accuracy": 0.9552, "ci_lower": 0.9471, "ci_upper": 0.9621},
    "XGBoost":             {"accuracy": 0.9477, "ci_lower": 0.9391, "ci_upper": 0.9552},
    "Random_Forest":       {"accuracy": 0.9220, "ci_lower": 0.9117, "ci_upper": 0.9311},
    "KNN":                 {"accuracy": 0.8897, "ci_lower": 0.8779, "ci_upper": 0.9005},
    "Decision_Tree":       {"accuracy": 0.8666, "ci_lower": 0.8539, "ci_upper": 0.8784},
    "Gaussian_NB":         {"accuracy": 0.7703, "ci_lower": 0.7547, "ci_upper": 0.7851},
}
WISDM_WILSON = {
    "KNN":                 {"accuracy": 0.9844, "ci_lower": 0.9807, "ci_upper": 0.9873},
    "XGBoost":             {"accuracy": 0.9799, "ci_lower": 0.9758, "ci_upper": 0.9833},
    "Random_Forest":       {"accuracy": 0.9721, "ci_lower": 0.9673, "ci_upper": 0.9762},
    "Decision_Tree":       {"accuracy": 0.9283, "ci_lower": 0.9211, "ci_upper": 0.9349},
    "Logistic_Regression": {"accuracy": 0.8799, "ci_lower": 0.8709, "ci_upper": 0.8883},
    "Linear_SVM":          {"accuracy": 0.8691, "ci_lower": 0.8598, "ci_upper": 0.8779},
    "Gaussian_NB":         {"accuracy": 0.7371, "ci_lower": 0.7252, "ci_upper": 0.7487},
}

UCI_HAR_COCHRAN = {"Q_statistic": 1223.2525, "degrees_of_freedom": 6, "p_value": 0.0}
WISDM_COCHRAN   = {"Q_statistic": 3991.6446, "degrees_of_freedom": 6, "p_value": 0.0}

# Raw pairwise McNemar/Holm tables, transcribed verbatim.
# Tuple format: (model_a, model_b, b_only, a_only, p_raw, p_holm, sig)
#   b_only = Discordant (A wrong / B right)
#   a_only = Discordant (A right / B wrong)
UCI_HAR_MCNEMAR_RAW = [
    ("Decision_Tree", "Gaussian_NB", 217, 501, 0.000000, 0.000000, True),
    ("Decision_Tree", "KNN", 269, 201, 0.001967, 0.005900, True),
    ("Decision_Tree", "Linear_SVM", 324, 43, 0.000000, 0.000000, True),
    ("Decision_Tree", "Logistic_Regression", 319, 58, 0.000000, 0.000000, True),
    ("Decision_Tree", "Random_Forest", 224, 61, 0.000000, 0.000000, True),
    ("Decision_Tree", "XGBoost", 274, 35, 0.000000, 0.000000, True),
    ("Gaussian_NB", "KNN", 524, 172, 0.000000, 0.000000, True),
    ("Gaussian_NB", "Linear_SVM", 630, 65, 0.000000, 0.000000, True),
    ("Gaussian_NB", "Logistic_Regression", 619, 74, 0.000000, 0.000000, True),
    ("Gaussian_NB", "Random_Forest", 552, 105, 0.000000, 0.000000, True),
    ("Gaussian_NB", "XGBoost", 601, 78, 0.000000, 0.000000, True),
    ("KNN", "Linear_SVM", 263, 50, 0.000000, 0.000000, True),
    ("KNN", "Logistic_Regression", 256, 63, 0.000000, 0.000000, True),
    ("KNN", "Random_Forest", 218, 123, 0.000000, 0.000002, True),
    ("KNN", "XGBoost", 251, 80, 0.000000, 0.000000, True),
    ("Linear_SVM", "Logistic_Regression", 26, 46, 0.024461, 0.048922, True),
    ("Linear_SVM", "Random_Forest", 61, 179, 0.000000, 0.000000, True),
    ("Linear_SVM", "XGBoost", 48, 90, 0.000441, 0.001762, True),
    ("Logistic_Regression", "Random_Forest", 70, 168, 0.000000, 0.000000, True),
    ("Logistic_Regression", "XGBoost", 63, 85, 0.083967, 0.083967, False),
    ("Random_Forest", "XGBoost", 121, 45, 0.000000, 0.000000, True),
]

WISDM_MCNEMAR_RAW = [
    ("Decision_Tree", "Gaussian_NB", 133, 1160, 0.000000, 0.000000, True),
    ("Decision_Tree", "KNN", 339, 38, 0.000000, 0.000000, True),
    ("Decision_Tree", "Linear_SVM", 181, 499, 0.000000, 0.000000, True),
    ("Decision_Tree", "Logistic_Regression", 188, 448, 0.000000, 0.000000, True),
    ("Decision_Tree", "Random_Forest", 287, 52, 0.000000, 0.000000, True),
    ("Decision_Tree", "XGBoost", 314, 37, 0.000000, 0.000000, True),
    ("Gaussian_NB", "KNN", 1358, 30, 0.000000, 0.000000, True),
    ("Gaussian_NB", "Linear_SVM", 933, 224, 0.000000, 0.000000, True),
    ("Gaussian_NB", "Logistic_Regression", 942, 175, 0.000000, 0.000000, True),
    ("Gaussian_NB", "Random_Forest", 1296, 34, 0.000000, 0.000000, True),
    ("Gaussian_NB", "XGBoost", 1331, 27, 0.000000, 0.000000, True),
    ("KNN", "Linear_SVM", 28, 647, 0.000000, 0.000000, True),
    ("KNN", "Logistic_Regression", 33, 594, 0.000000, 0.000000, True),
    ("KNN", "Random_Forest", 35, 101, 0.000000, 0.000000, True),
    ("KNN", "XGBoost", 40, 64, 0.023646, 0.023646, True),
    ("Linear_SVM", "Logistic_Regression", 135, 77, 0.000082, 0.000165, True),
    ("Linear_SVM", "Random_Forest", 585, 32, 0.000000, 0.000000, True),
    ("Linear_SVM", "XGBoost", 617, 22, 0.000000, 0.000000, True),
    ("Logistic_Regression", "Random_Forest", 532, 37, 0.000000, 0.000000, True),
    ("Logistic_Regression", "XGBoost", 554, 17, 0.000000, 0.000000, True),
    ("Random_Forest", "XGBoost", 69, 27, 0.000021, 0.000064, True),
]

UCI_HAR_WIN_COUNTS = compute_win_counts(UCI_HAR_MCNEMAR_RAW, CANONICAL_MODELS)
WISDM_WIN_COUNTS   = compute_win_counts(WISDM_MCNEMAR_RAW, CANONICAL_MODELS)

print("\n[HARDCODED] UCI_HAR significant pairwise wins (derived from transcribed McNemar table):")
print(f"  {UCI_HAR_WIN_COUNTS}")
print("[HARDCODED] WISDM significant pairwise wins (derived from transcribed McNemar table):")
print(f"  {WISDM_WIN_COUNTS}")

# ============================================================================
# 3e. HARDCODED PAMAP2 DATA (transcribed from console output + uploaded CSVs)
# ============================================================================

# --- Performance (Module 1) — all fields present in console log ---
PAMAP2_PERFORMANCE = {
    "Logistic_Regression": {"accuracy": 0.9657, "f1_macro": 0.9623, "cv_accuracy_mean": 0.9705, "train_time_sec": 19.5},
    "Decision_Tree":       {"accuracy": 0.9361, "f1_macro": 0.9308, "cv_accuracy_mean": 0.9305, "train_time_sec": 24.3},
    "Random_Forest":       {"accuracy": 0.9711, "f1_macro": 0.9687, "cv_accuracy_mean": 0.9700, "train_time_sec": 108.1},
    "KNN":                 {"accuracy": 0.9670, "f1_macro": 0.9660, "cv_accuracy_mean": 0.9657, "train_time_sec": 5.9},
    "Gaussian_NB":         {"accuracy": 0.9032, "f1_macro": 0.9086, "cv_accuracy_mean": 0.9017, "train_time_sec": 2.4},
    "Linear_SVM":          {"accuracy": 0.9731, "f1_macro": 0.9697, "cv_accuracy_mean": 0.9716, "train_time_sec": 410.5},
    "XGBoost":             {"accuracy": 0.9778, "f1_macro": 0.9739, "cv_accuracy_mean": 0.9783, "train_time_sec": 858.6},
}

# --- TinyML (Module 2) — raw ingredients present in console log.        ---
# --- Deployment_Friendliness_Score/Tier were NOT printed for PAMAP2, so ---
# --- they are COMPUTED here using the exact same documented formula     ---
# --- (25% RAM, 20% Flash, 25% Latency, 15% Size, 15% Accuracy,          ---
# --- min-max normalized across these 7 models), applied to the raw,    ---
# --- console-reported numbers. This is arithmetic on real data, not     ---
# --- estimation/fabrication of a missing measurement.                   ---
PAMAP2_TINYML_RAW = {
    "Logistic_Regression": {"model_size_kb": 57.12,    "latency_ms": 0.4083,  "ram_kb": 72.40,     "flash_kb": 77.12},
    "Decision_Tree":       {"model_size_kb": 48.03,    "latency_ms": 0.2662,  "ram_kb": 61.50,     "flash_kb": 68.03},
    "Random_Forest":       {"model_size_kb": 15908.00, "latency_ms": 36.9590, "ram_kb": 19093.46,  "flash_kb": 15928.00},
    "KNN":                 {"model_size_kb": 22131.61, "latency_ms": 3.9421,  "ram_kb": 26561.78,  "flash_kb": 22151.61},
    "Gaussian_NB":         {"model_size_kb": 101.61,   "latency_ms": 0.6818,  "ram_kb": 125.79,    "flash_kb": 121.61},
    "Linear_SVM":          {"model_size_kb": 56.99,    "latency_ms": 0.2683,  "ram_kb": 72.24,     "flash_kb": 76.99},
    "XGBoost":             {"model_size_kb": 2367.70,  "latency_ms": 2.3327,  "ram_kb": 2845.09,   "flash_kb": 2387.70},
}

def _minmax(values: dict):
    arr = np.array(list(values.values()), dtype=float)
    rng = arr.max() - arr.min()
    if rng < 1e-12:
        return {k: 0.0 for k in values}
    return {k: (v - arr.min()) / rng for k, v in values.items()}

def _assign_tier_local(score: float) -> str:
    if score >= 80:
        return "Tier 1 - Excellent"
    elif score >= 65:
        return "Tier 2 - Good"
    elif score >= 45:
        return "Tier 3 - Moderate"
    else:
        return "Tier 4 - Poor"

def _compute_pamap2_tinyml():
    ram = {m: v["ram_kb"] for m, v in PAMAP2_TINYML_RAW.items()}
    flash = {m: v["flash_kb"] for m, v in PAMAP2_TINYML_RAW.items()}
    latency = {m: v["latency_ms"] for m, v in PAMAP2_TINYML_RAW.items()}
    size = {m: v["model_size_kb"] for m, v in PAMAP2_TINYML_RAW.items()}
    acc = {m: PAMAP2_PERFORMANCE[m]["accuracy"] for m in PAMAP2_TINYML_RAW}

    norm_ram, norm_flash = _minmax(ram), _minmax(flash)
    norm_latency, norm_size, norm_acc = _minmax(latency), _minmax(size), _minmax(acc)

    result = {}
    for m in PAMAP2_TINYML_RAW:
        score = 100 * (
            0.25 * (1 - norm_ram[m]) +
            0.20 * (1 - norm_flash[m]) +
            0.25 * (1 - norm_latency[m]) +
            0.15 * (1 - norm_size[m]) +
            0.15 * norm_acc[m]
        )
        result[m] = {
            "model_size_kb": PAMAP2_TINYML_RAW[m]["model_size_kb"],
            "latency_ms": PAMAP2_TINYML_RAW[m]["latency_ms"],
            "ram_kb": PAMAP2_TINYML_RAW[m]["ram_kb"],
            "flash_kb": PAMAP2_TINYML_RAW[m]["flash_kb"],
            "score": score,
            "tier_raw": _assign_tier_local(score) + " (computed from console-reported raw metrics; "
                        "score/tier were not printed in the original PAMAP2 console log)",
        }
    return result

PAMAP2_TINYML = _compute_pamap2_tinyml()

# --- Robustness (Module 3) — console log prints baseline accuracy and a  ---
# --- single "worst-case corrupted accuracy" per model. Overall_Robustness ---
# --- _Score is COMPUTED as 100 x worst_case / baseline. NOTE: this is a  ---
# --- DIFFERENT metric than the other 4 datasets, whose score is the MEAN ---
# --- retention across 9-10 perturbation conditions. PAMAP2's score       ---
# --- therefore reflects worst-case degradation, not average degradation, ---
# --- and will read lower/harsher than the others for that reason — this  ---
# --- is a genuine measurement difference, not an error. Flagged clearly  ---
# --- in the report.                                                       ---
PAMAP2_ROBUSTNESS_WORST_CASE = {
    "Logistic_Regression": {"baseline": 0.9657, "worst_case": 0.0995},
    "Decision_Tree":       {"baseline": 0.9361, "worst_case": 0.0955},
    "Random_Forest":       {"baseline": 0.9711, "worst_case": 0.0995},
    "KNN":                 {"baseline": 0.9670, "worst_case": 0.0982},
    "Gaussian_NB":         {"baseline": 0.9032, "worst_case": 0.0874},
    "Linear_SVM":          {"baseline": 0.9731, "worst_case": 0.0995},
    "XGBoost":             {"baseline": 0.9778, "worst_case": 0.0955},
}

def _compute_pamap2_robustness():
    result = {}
    for m, vals in PAMAP2_ROBUSTNESS_WORST_CASE.items():
        retention_pct = 100 * vals["worst_case"] / vals["baseline"]
        result[m] = {
            "overall_score": retention_pct,
            "noise_retention": np.nan,               # per-condition breakdown not available
            "feature_removal_retention": np.nan,     # per-condition breakdown not available
            "sensor_retention": np.nan,               # per-condition breakdown not available
        }
    return result

PAMAP2_ROBUSTNESS = _compute_pamap2_robustness()

# --- Statistical Validation (Module 4) — Wilson CI, Cochran's Q, and the ---
# --- full pairwise McNemar/Holm table were all confirmed from uploaded   ---
# --- CSVs (wilson_confidence_intervals.csv, cochran_q_results.csv,       ---
# --- holm_adjusted_results.csv). Significant_Pairwise_Wins is computed   ---
# --- from real data using the same compute_win_counts() helper used for ---
# --- every other dataset.                                                 ---
PAMAP2_WILSON = {
    "XGBoost":             {"accuracy": 0.977808, "ci_lower": 0.968998, "ci_upper": 0.984155},
    "Linear_SVM":          {"accuracy": 0.973100, "ci_lower": 0.963578, "ci_upper": 0.980184},
    "Random_Forest":       {"accuracy": 0.971083, "ci_lower": 0.961276, "ci_upper": 0.978461},
    "KNN":                 {"accuracy": 0.967048, "ci_lower": 0.956703, "ci_upper": 0.974985},
    "Logistic_Regression": {"accuracy": 0.965703, "ci_lower": 0.955187, "ci_upper": 0.973819},
    "Decision_Tree":       {"accuracy": 0.936113, "ci_lower": 0.922525, "ci_upper": 0.947454},
    "Gaussian_NB":         {"accuracy": 0.903161, "ci_lower": 0.887074, "ci_upper": 0.917170},
}
PAMAP2_COCHRAN = {"Q_statistic": 280.17283950617286, "degrees_of_freedom": 6, "p_value": 1.4427901179236016e-57}

# Raw pairwise McNemar/Holm table, transcribed from uploaded holm_adjusted_results.csv
# Confirmed columns: model_a, model_b, n10, n01, n_discordant, statistic,
# p_value, method, holm_adjusted_p_value, significant_after_holm
#   n10 = Model_A correct / Model_B wrong  (a_only)
#   n01 = Model_A wrong / Model_B correct  (b_only)
# Tuple format matches compute_win_counts(): (model_a, model_b, b_only, a_only, p_raw, p_holm, sig)
PAMAP2_MCNEMAR_RAW = [
    ("Gaussian_NB", "XGBoost", 113, 2, 1.094450804683078e-24, 2.298346689834464e-23, True),
    ("Gaussian_NB", "Linear_SVM", 110, 6, 1.1404959724299155e-21, 2.280991944859831e-20, True),
    ("Random_Forest", "Gaussian_NB", 5, 106, 2.275586552857822e-21, 4.3236144504298623e-20, True),
    ("KNN", "Gaussian_NB", 15, 110, 4.184298969701859e-17, 7.531738145463346e-16, True),
    ("Logistic_Regression", "Gaussian_NB", 16, 109, 1.8920847060445003e-16, 3.2165440002756506e-15, True),
    ("Decision_Tree", "XGBoost", 66, 4, 3.079037744278066e-13, 4.926460390844905e-12, True),
    ("Decision_Tree", "Random_Forest", 58, 6, 1.8296295167217228e-10, 2.7444442750825843e-09, True),
    ("Decision_Tree", "Linear_SVM", 68, 13, 1.9731752900753933e-09, 2.7624454061055507e-08, True),
    ("Decision_Tree", "KNN", 62, 16, 3.48287497867635e-07, 4.527737472279255e-06, True),
    ("Logistic_Regression", "Decision_Tree", 20, 64, 2.709645743730636e-06, 3.251574892476763e-05, True),
    ("Decision_Tree", "Gaussian_NB", 45, 94, 4.675011517049424e-05, 0.0005142512668754366, True),
    ("Logistic_Regression", "XGBoost", 31, 13, 0.010381795789701767, 0.10381795789701767, False),
    ("KNN", "XGBoost", 27, 11, 0.014961017675175686, 0.13464915907658118, False),
    ("Random_Forest", "XGBoost", 16, 6, 0.052478790283203125, 0.419830322265625, False),
    ("Logistic_Regression", "Linear_SVM", 19, 8, 0.05429182836685469, 0.419830322265625, False),
    ("KNN", "Linear_SVM", 23, 14, 0.18844541724270242, 1.0, False),
    ("Linear_SVM", "XGBoost", 17, 10, 0.24821307898992026, 1.0, False),
    ("Logistic_Regression", "Random_Forest", 25, 17, 0.28008721081149435, 1.0, False),
    ("Random_Forest", "KNN", 13, 19, 0.3767591178115821, 1.0, False),
    ("Random_Forest", "Linear_SVM", 20, 17, 0.7423083940717163, 1.0, False),
    ("Logistic_Regression", "KNN", 23, 21, 0.8801684549067255, 1.0, False),
]

PAMAP2_WIN_COUNTS = compute_win_counts(PAMAP2_MCNEMAR_RAW, CANONICAL_MODELS)

print("\n[HARDCODED] PAMAP2 data loaded from console-log transcription and uploaded CSVs "
      "(Deployment Score/Tier computed from raw metrics; Robustness computed as "
      "worst-case retention; McNemar win-counts derived from uploaded holm_adjusted_results.csv).")
print(f"  PAMAP2 significant pairwise wins: {PAMAP2_WIN_COUNTS}")

# ============================================================================
# 4. LOADERS — MHEALTH (exact schema known, since this project wrote it)
# ============================================================================
def load_mhealth_performance():
    print("\n[LOAD] MHEALTH Module 1 (Performance)...")
    data = safe_read_json(MHEALTH_MODULE1_DIR / "mhealth_metrics.json")
    result = {m: {"accuracy": np.nan, "f1_macro": np.nan, "cv_accuracy_mean": np.nan, "train_time_sec": np.nan} for m in CANONICAL_MODELS}
    if data is None:
        return result
    for raw_name, vals in data.items():
        norm = normalize_model_name(raw_name)
        if norm is None:
            continue
        result[norm] = {
            "accuracy": vals.get("accuracy", np.nan),
            "f1_macro": vals.get("f1_macro", np.nan),
            "cv_accuracy_mean": vals.get("cv_accuracy_mean", np.nan),
            "train_time_sec": vals.get("train_time_sec", np.nan),
        }
    return result


def load_mhealth_tinyml():
    print("[LOAD] MHEALTH Module 2 (TinyML)...")
    df = safe_read_csv(MHEALTH_MODULE2_DIR / "mhealth_tinyml_ranking.csv")
    result = {m: {"model_size_kb": np.nan, "latency_ms": np.nan, "ram_kb": np.nan,
                  "flash_kb": np.nan, "score": np.nan, "tier_raw": "N/A"} for m in CANONICAL_MODELS}
    if df is None:
        return result
    for _, row in df.iterrows():
        norm = normalize_model_name(row.get("Model"))
        if norm is None:
            continue
        result[norm] = {
            "model_size_kb": row.get("Model_Size_KB", np.nan),
            "latency_ms": row.get("Inference_Latency_ms", np.nan),
            "ram_kb": row.get("RAM_Estimate_KB", np.nan),
            "flash_kb": row.get("Flash_Estimate_KB", np.nan),
            "score": row.get("Deployment_Friendliness_Score", np.nan),
            "tier_raw": row.get("Deployment_Tier", "N/A"),
        }
    return result


def load_mhealth_robustness():
    print("[LOAD] MHEALTH Module 3 (Robustness)...")
    df = safe_read_csv(MHEALTH_MODULE3_DIR / "mhealth_robustness_ranking.csv")
    result = {m: {"overall_score": np.nan, "noise_retention": np.nan,
                  "feature_removal_retention": np.nan, "sensor_retention": np.nan} for m in CANONICAL_MODELS}
    if df is None:
        return result
    for _, row in df.iterrows():
        norm = normalize_model_name(row.get("Model"))
        if norm is None:
            continue
        result[norm] = {
            "overall_score": row.get("Overall_Robustness_Score", np.nan),
            "noise_retention": row.get("Avg_Retention_Gaussian_Noise_%", np.nan),
            "feature_removal_retention": row.get("Avg_Retention_Feature_Removal_%", np.nan),
            "sensor_retention": row.get("Avg_Retention_Sensor_Dropout_%", np.nan),
        }
    return result


def load_mhealth_statistics():
    print("[LOAD] MHEALTH Module 4 (Statistical Validation)...")
    df = safe_read_csv(MHEALTH_MODULE4_DIR / "mhealth_statistical_summary.csv")
    cochran = safe_read_json(MHEALTH_MODULE4_DIR / "mhealth_cochrans_q_results.json")
    wilson_result = {m: {"accuracy": np.nan, "ci_lower": np.nan, "ci_upper": np.nan} for m in CANONICAL_MODELS}
    win_counts = {m: np.nan for m in CANONICAL_MODELS}
    if df is not None:
        for _, row in df.iterrows():
            norm = normalize_model_name(row.get("Model"))
            if norm is None:
                continue
            wilson_result[norm] = {
                "accuracy": row.get("Accuracy", np.nan),
                "ci_lower": row.get("Wilson_CI_Lower", np.nan),
                "ci_upper": row.get("Wilson_CI_Upper", np.nan),
            }
            win_counts[norm] = row.get("Significant_Pairwise_Wins", np.nan)
    cochran_result = {"Q_statistic": np.nan, "degrees_of_freedom": np.nan, "p_value": np.nan}
    if cochran is not None:
        cochran_result = {
            "Q_statistic": cochran.get("Q_statistic", np.nan),
            "degrees_of_freedom": cochran.get("degrees_of_freedom", np.nan),
            "p_value": cochran.get("p_value", np.nan),
        }
    return wilson_result, cochran_result, win_counts


# ============================================================================
# 5. LOADERS — MOTIONSENSE (same schema as MHEALTH)
# ============================================================================
def load_motionsense_performance():
    print("[LOAD] MotionSense Module 1 (Performance)...")
    data = safe_read_json(MOTIONSENSE_MODULE1_DIR / "motionsense_metrics.json")
    result = {m: {"accuracy": np.nan, "f1_macro": np.nan, "cv_accuracy_mean": np.nan, "train_time_sec": np.nan} for m in CANONICAL_MODELS}
    if data is None:
        return result
    for raw_name, vals in data.items():
        norm = normalize_model_name(raw_name)
        if norm is None:
            continue
        result[norm] = {
            "accuracy": vals.get("accuracy", np.nan),
            "f1_macro": vals.get("f1_macro", np.nan),
            "cv_accuracy_mean": vals.get("cv_accuracy_mean", np.nan),
            "train_time_sec": vals.get("train_time_sec", np.nan),
        }
    return result


def load_motionsense_tinyml():
    print("[LOAD] MotionSense Module 2 (TinyML)...")
    df = safe_read_csv(MOTIONSENSE_MODULE2_DIR / "motionsense_tinyml_ranking.csv")
    result = {m: {"model_size_kb": np.nan, "latency_ms": np.nan, "ram_kb": np.nan,
                  "flash_kb": np.nan, "score": np.nan, "tier_raw": "N/A"} for m in CANONICAL_MODELS}
    if df is None:
        return result
    for _, row in df.iterrows():
        norm = normalize_model_name(row.get("Model"))
        if norm is None:
            continue
        result[norm] = {
            "model_size_kb": row.get("Model_Size_KB", np.nan),
            "latency_ms": row.get("Inference_Latency_ms", np.nan),
            "ram_kb": row.get("RAM_Estimate_KB", np.nan),
            "flash_kb": row.get("Flash_Estimate_KB", np.nan),
            "score": row.get("Deployment_Friendliness_Score", np.nan),
            "tier_raw": row.get("Deployment_Tier", "N/A"),
        }
    return result


def load_motionsense_robustness():
    print("[LOAD] MotionSense Module 3 (Robustness)...")
    df = safe_read_csv(MOTIONSENSE_MODULE3_DIR / "motionsense_robustness_ranking.csv")
    result = {m: {"overall_score": np.nan, "noise_retention": np.nan,
                  "feature_removal_retention": np.nan, "sensor_retention": np.nan} for m in CANONICAL_MODELS}
    if df is None:
        return result
    for _, row in df.iterrows():
        norm = normalize_model_name(row.get("Model"))
        if norm is None:
            continue
        result[norm] = {
            "overall_score": row.get("Overall_Robustness_Score", np.nan),
            "noise_retention": row.get("Avg_Retention_Gaussian_Noise_%", np.nan),
            "feature_removal_retention": row.get("Avg_Retention_Feature_Removal_%", np.nan),
            "sensor_retention": row.get("Avg_Retention_Sensor_Dropout_%", np.nan),
        }
    return result


def load_motionsense_statistics():
    print("[LOAD] MotionSense Module 4 (Statistical Validation)...")
    df = safe_read_csv(MOTIONSENSE_MODULE4_DIR / "motionsense_statistical_summary.csv")
    cochran = safe_read_json(MOTIONSENSE_MODULE4_DIR / "motionsense_cochrans_q_results.json")
    wilson_result = {m: {"accuracy": np.nan, "ci_lower": np.nan, "ci_upper": np.nan} for m in CANONICAL_MODELS}
    win_counts = {m: np.nan for m in CANONICAL_MODELS}
    if df is not None:
        for _, row in df.iterrows():
            norm = normalize_model_name(row.get("Model"))
            if norm is None:
                continue
            wilson_result[norm] = {
                "accuracy": row.get("Accuracy", np.nan),
                "ci_lower": row.get("Wilson_CI_Lower", np.nan),
                "ci_upper": row.get("Wilson_CI_Upper", np.nan),
            }
            win_counts[norm] = row.get("Significant_Pairwise_Wins", np.nan)
    cochran_result = {"Q_statistic": np.nan, "degrees_of_freedom": np.nan, "p_value": np.nan}
    if cochran is not None:
        cochran_result = {
            "Q_statistic": cochran.get("Q_statistic", np.nan),
            "degrees_of_freedom": cochran.get("degrees_of_freedom", np.nan),
            "p_value": cochran.get("p_value", np.nan),
        }
    return wilson_result, cochran_result, win_counts


# ============================================================================
# 6. ASSEMBLE ALL DATA
# ============================================================================
print("\n" + "=" * 70)
print("LOADING ALL FIVE DATASETS")
print("=" * 70)

PERFORMANCE = {
    "UCI_HAR": UCI_HAR_PERFORMANCE,
    "WISDM": WISDM_PERFORMANCE,
    "PAMAP2": PAMAP2_PERFORMANCE,
    "MHEALTH": load_mhealth_performance(),
    "MotionSense": load_motionsense_performance(),
}

TINYML = {
    "UCI_HAR": UCI_HAR_TINYML,
    "WISDM": WISDM_TINYML,
    "PAMAP2": PAMAP2_TINYML,
    "MHEALTH": load_mhealth_tinyml(),
    "MotionSense": load_motionsense_tinyml(),
}

mhealth_robustness = load_mhealth_robustness()
motionsense_robustness = load_motionsense_robustness()

ROBUSTNESS = {
    "UCI_HAR": {m: {"overall_score": UCI_HAR_ROBUSTNESS_OVERALL.get(m, np.nan),
                     "noise_retention": np.nan, "feature_removal_retention": np.nan,
                     "sensor_retention": np.nan} for m in CANONICAL_MODELS},
    "WISDM": {m: {"overall_score": WISDM_ROBUSTNESS_OVERALL.get(m, np.nan),
                   "noise_retention": np.nan, "feature_removal_retention": np.nan,
                   "sensor_retention": np.nan} for m in CANONICAL_MODELS},
    "PAMAP2": PAMAP2_ROBUSTNESS,
    "MHEALTH": mhealth_robustness,
    "MotionSense": motionsense_robustness,
}

mhealth_wilson, mhealth_cochran, mhealth_wins = load_mhealth_statistics()
motionsense_wilson, motionsense_cochran, motionsense_wins = load_motionsense_statistics()

STATISTICS_WILSON = {
    "UCI_HAR": UCI_HAR_WILSON, "WISDM": WISDM_WILSON,
    "PAMAP2": PAMAP2_WILSON, "MHEALTH": mhealth_wilson, "MotionSense": motionsense_wilson,
}
STATISTICS_COCHRAN = {
    "UCI_HAR": UCI_HAR_COCHRAN, "WISDM": WISDM_COCHRAN,
    "PAMAP2": PAMAP2_COCHRAN, "MHEALTH": mhealth_cochran, "MotionSense": motionsense_cochran,
}
STATISTICS_WINS = {
    "UCI_HAR": UCI_HAR_WIN_COUNTS, "WISDM": WISDM_WIN_COUNTS,
    "PAMAP2": PAMAP2_WIN_COUNTS, "MHEALTH": mhealth_wins, "MotionSense": motionsense_wins,
}

print("\n[LOAD] All datasets assembled.")

# ============================================================================
# 7. BUILD UNIFIED LONG-FORMAT TABLES
# ============================================================================
def build_performance_table():
    rows = []
    for ds in DATASETS:
        for m in CANONICAL_MODELS:
            vals = PERFORMANCE[ds].get(m, {})
            rows.append({
                "Dataset": ds, "Model": m,
                "Accuracy": vals.get("accuracy", np.nan),
                "Macro_F1": vals.get("f1_macro", np.nan),
                "CV_Accuracy_Mean": vals.get("cv_accuracy_mean", np.nan),
                "Training_Time_sec": vals.get("train_time_sec", np.nan),
            })
    return pd.DataFrame(rows)

def build_tinyml_table():
    rows = []
    for ds in DATASETS:
        for m in CANONICAL_MODELS:
            vals = TINYML[ds].get(m, {})
            rows.append({
                "Dataset": ds, "Model": m,
                "Model_Size_KB": vals.get("model_size_kb", np.nan),
                "RAM_Estimate_KB": vals.get("ram_kb", np.nan),
                "Flash_Estimate_KB": vals.get("flash_kb", np.nan),
                "Inference_Latency_ms": vals.get("latency_ms", np.nan),
                "Deployment_Friendliness_Score": vals.get("score", np.nan),
                "Deployment_Tier_As_Reported": vals.get("tier_raw", "N/A"),
            })
    return pd.DataFrame(rows)

def build_robustness_table():
    rows = []
    for ds in DATASETS:
        for m in CANONICAL_MODELS:
            vals = ROBUSTNESS[ds].get(m, {})
            rows.append({
                "Dataset": ds, "Model": m,
                "Overall_Robustness_Score": vals.get("overall_score", np.nan),
                "Noise_Retention_%": vals.get("noise_retention", np.nan),
                "Feature_Removal_Retention_%": vals.get("feature_removal_retention", np.nan),
                "Sensor_Dropout_Retention_%": vals.get("sensor_retention", np.nan),
            })
    return pd.DataFrame(rows)

def build_statistics_table():
    rows = []
    for ds in DATASETS:
        for m in CANONICAL_MODELS:
            wvals = STATISTICS_WILSON[ds].get(m, {})
            wins = STATISTICS_WINS[ds].get(m, np.nan)
            rows.append({
                "Dataset": ds, "Model": m,
                "Accuracy": wvals.get("accuracy", np.nan),
                "Wilson_CI_Lower": wvals.get("ci_lower", np.nan),
                "Wilson_CI_Upper": wvals.get("ci_upper", np.nan),
                "CI_Width": (wvals.get("ci_upper", np.nan) - wvals.get("ci_lower", np.nan))
                            if pd.notna(wvals.get("ci_upper")) and pd.notna(wvals.get("ci_lower")) else np.nan,
                "Significant_Pairwise_Wins": wins,
            })
    return pd.DataFrame(rows)

performance_df = build_performance_table()
tinyml_df = build_tinyml_table()
robustness_df = build_robustness_table()
statistics_df = build_statistics_table()

cochran_summary_df = pd.DataFrame([
    {"Dataset": ds, **STATISTICS_COCHRAN[ds]} for ds in DATASETS
])

print("\n[BUILD] Unified long-format tables constructed for all 4 categories.")

# ============================================================================
# 8. RANKINGS
# ============================================================================
def rank_within_dataset(df: pd.DataFrame, value_col: str, dataset_col="Dataset", model_col="Model", ascending=False):
    df = df.copy()
    df[f"{value_col}_Rank"] = df.groupby(dataset_col)[value_col].rank(ascending=ascending, method="min")
    return df

performance_df = rank_within_dataset(performance_df, "Accuracy")
tinyml_df = rank_within_dataset(tinyml_df, "Deployment_Friendliness_Score")
robustness_df = rank_within_dataset(robustness_df, "Overall_Robustness_Score")

# Statistical rank: primary = Significant_Pairwise_Wins desc, secondary = Accuracy desc
statistics_df["_sort_key_wins"] = -statistics_df["Significant_Pairwise_Wins"].fillna(-999)
statistics_df["_sort_key_acc"] = -statistics_df["Accuracy"].fillna(-999)
statistics_df["Statistical_Rank"] = statistics_df.groupby("Dataset")[["_sort_key_wins", "_sort_key_acc"]] \
    .apply(lambda g: g.apply(tuple, axis=1).rank(method="min").astype(int)) \
    .reset_index(level=0, drop=True)
statistics_df = statistics_df.drop(columns=["_sort_key_wins", "_sort_key_acc"])

# Average rank per model per category (NaN-safe mean across datasets)
avg_perf_rank = performance_df.groupby("Model")["Accuracy_Rank"].mean()
avg_tinyml_rank = tinyml_df.groupby("Model")["Deployment_Friendliness_Score_Rank"].mean()
avg_robust_rank = robustness_df.groupby("Model")["Overall_Robustness_Score_Rank"].mean()
avg_stat_rank = statistics_df.groupby("Model")["Statistical_Rank"].mean()

# Diagnostic: how many datasets actually contributed a non-NaN value to each
# category's average, per model — makes coverage explicit rather than implicit.
n_perf = performance_df.groupby("Model")["Accuracy"].apply(lambda s: s.notna().sum())
n_tinyml = tinyml_df.groupby("Model")["Deployment_Friendliness_Score"].apply(lambda s: s.notna().sum())
n_robust = robustness_df.groupby("Model")["Overall_Robustness_Score"].apply(lambda s: s.notna().sum())
n_stat = statistics_df.groupby("Model")["Significant_Pairwise_Wins"].apply(lambda s: s.notna().sum())

overall_ranking = pd.DataFrame({
    "Avg_Performance_Rank": avg_perf_rank,
    "N_Datasets_Performance": n_perf,
    "Avg_TinyML_Rank": avg_tinyml_rank,
    "N_Datasets_TinyML": n_tinyml,
    "Avg_Robustness_Rank": avg_robust_rank,
    "N_Datasets_Robustness": n_robust,
    "Avg_Statistical_Rank": avg_stat_rank,
    "N_Datasets_Statistical": n_stat,
}).reindex(CANONICAL_MODELS)

overall_ranking["Overall_Combined_Rank_Score"] = overall_ranking[
    ["Avg_Performance_Rank", "Avg_TinyML_Rank", "Avg_Robustness_Rank", "Avg_Statistical_Rank"]
].mean(axis=1)

overall_ranking = overall_ranking.sort_values("Overall_Combined_Rank_Score")
overall_ranking.insert(0, "Overall_Rank", range(1, len(overall_ranking) + 1))

print("\n[RANKING] Overall cross-dataset classifier ranking (lower combined rank score = better):")
print(overall_ranking.round(2))

# ============================================================================
# 9. SAVE TABLES
# ============================================================================
performance_df.to_csv(RESULTS_DIR / "cross_dataset_performance.csv", index=False, encoding=ENCODING)
tinyml_df.to_csv(RESULTS_DIR / "cross_dataset_tinyml.csv", index=False, encoding=ENCODING)
robustness_df.to_csv(RESULTS_DIR / "cross_dataset_robustness.csv", index=False, encoding=ENCODING)
statistics_df.to_csv(RESULTS_DIR / "cross_dataset_statistics.csv", index=False, encoding=ENCODING)
cochran_summary_df.to_csv(RESULTS_DIR / "cross_dataset_cochran_q.csv", index=False, encoding=ENCODING)
overall_ranking.to_csv(RESULTS_DIR / "cross_dataset_overall_ranking.csv", encoding=ENCODING)

print(f"\n[SAVE] Tables written to {RESULTS_DIR}")

# ============================================================================
# 10. FIGURES (300 dpi)
# ============================================================================
def pivot_heatmap(df, value_col, title, filename, fmt=".3f", cmap="viridis"):
    pivot = df.pivot(index="Model", columns="Dataset", values=value_col).reindex(
        index=CANONICAL_MODELS, columns=DATASETS
    )
    plt.figure(figsize=(11, 7))
    sns.heatmap(pivot.astype(float), annot=True, fmt=fmt, cmap=cmap,
                cbar_kws={"label": value_col}, linewidths=0.5, linecolor="white")
    plt.title(title)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=FIGURE_DPI)
    plt.close()

pivot_heatmap(performance_df, "Accuracy",
              "Cross-Dataset Accuracy Heatmap (all 5 datasets, 7 classifiers)",
              "cross_dataset_accuracy_heatmap.png", fmt=".3f", cmap="YlGnBu")

pivot_heatmap(tinyml_df, "Deployment_Friendliness_Score",
              "Cross-Dataset Deployment Friendliness Score Heatmap",
              "cross_dataset_tinyml_score_heatmap.png", fmt=".1f", cmap="YlOrRd_r")

pivot_heatmap(robustness_df, "Overall_Robustness_Score",
              "Cross-Dataset Overall Robustness Score Heatmap\n"
              "(PAMAP2 = worst-case retention; all others = mean retention — see report note)",
              "cross_dataset_robustness_heatmap.png", fmt=".1f", cmap="RdYlGn")

# Average rank per category, grouped bar chart
plt.figure(figsize=(13, 7))
plot_order = overall_ranking.index.tolist()
categories = ["Avg_Performance_Rank", "Avg_TinyML_Rank", "Avg_Robustness_Rank", "Avg_Statistical_Rank"]
cat_labels = ["Performance", "TinyML", "Robustness", "Statistical"]
x = np.arange(len(plot_order))
width = 0.2
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
for i, (cat, label, color) in enumerate(zip(categories, cat_labels, colors)):
    plt.bar(x + (i - 1.5) * width, overall_ranking.loc[plot_order, cat], width, label=label, color=color)
plt.xticks(x, plot_order, rotation=45, ha="right")
plt.ylabel("Average Rank (1 = best)")
plt.title("Average Classifier Rank per Category (across all 5 datasets)")
plt.gca().invert_yaxis()
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_dataset_avg_rank_per_category.png", dpi=FIGURE_DPI)
plt.close()

# Overall combined rank bar chart
plt.figure(figsize=(11, 6))
values = overall_ranking.loc[plot_order, "Overall_Combined_Rank_Score"]
bars = plt.bar(plot_order, values, color="#8172B2")
plt.ylabel("Overall Combined Rank Score (lower = better)")
plt.title("Overall Cross-Dataset Classifier Ranking (all 4 categories combined)")
plt.xticks(rotation=45, ha="right")
plt.gca().invert_yaxis()
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
              f"{val:.2f}", ha="center", va="top", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_dataset_overall_ranking.png", dpi=FIGURE_DPI)
plt.close()

# Accuracy vs Deployment Score scatter, colored by dataset
plt.figure(figsize=(10, 8))
merged = performance_df.merge(tinyml_df, on=["Dataset", "Model"])
for ds in DATASETS:
    sub = merged[merged["Dataset"] == ds]
    plt.scatter(sub["Accuracy"], sub["Deployment_Friendliness_Score"], label=ds, s=70, alpha=0.8)
plt.xlabel("Accuracy")
plt.ylabel("Deployment Friendliness Score")
plt.title("Accuracy vs Deployment Friendliness Score — All Datasets & Classifiers")
plt.legend()
plt.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_dataset_accuracy_vs_deployment.png", dpi=FIGURE_DPI)
plt.close()

print(f"[FIGURES] Saved 6 figures (300 dpi) to {FIGURES_DIR}")

# ============================================================================
# 11. MARKDOWN REPORT
# ============================================================================
report_lines = [
    "# Module 5 — Cross-Dataset Comparative Analysis",
    "## Standardized HAR Framework: UCI HAR, WISDM, PAMAP2, MHEALTH, MotionSense\n",
    "### Data Provenance",
    "- **UCI HAR, WISDM**: values transcribed directly from console outputs "
    "provided earlier in this conversation (original experiments run in Google "
    "Colab; the underlying CSV/JSON output files were not available for this "
    "module). Any metric not present in that console output is marked **N/A** "
    "rather than estimated.",
    "- **PAMAP2**: Performance metrics (accuracy, F1, CV accuracy, training "
    "time) and Statistical Validation (Wilson CI, Cochran's Q, and the full "
    "pairwise McNemar/Holm table) are transcribed from console output and the "
    "uploaded `wilson_confidence_intervals.csv`, `cochran_q_results.csv`, and "
    "`holm_adjusted_results.csv` files — all real, uploaded data.",
    "- **PAMAP2 TinyML Deployment Friendliness Score/Tier**: these were NOT "
    "printed in PAMAP2's console log, but all five raw ingredients (model "
    "size, RAM estimate, flash estimate, latency, accuracy) WERE printed. The "
    "score/tier shown here are COMPUTED from those real, console-reported "
    "numbers using this project's exact standard formula (25% RAM, 20% Flash, "
    "25% Latency, 15% Size, 15% Accuracy, min-max normalized across PAMAP2's "
    "own 7 models) — this is arithmetic on real data, not fabrication.",
    "- **PAMAP2 Overall Robustness Score — IMPORTANT METRIC DIFFERENCE**: "
    "PAMAP2's console log reported baseline accuracy and a single 'worst-case "
    "corrupted accuracy' per model (the floor under the worst perturbation "
    "tested), NOT the mean retention across 9-10 perturbation conditions used "
    "for every other dataset's Overall Robustness Score. The value shown here "
    "is computed as `100 x worst_case_accuracy / baseline_accuracy` — a real "
    "number from real data, but measuring worst-case degradation rather than "
    "average degradation. This makes PAMAP2's robustness scores read "
    "considerably harsher than the other four datasets, and the two should "
    "not be read as directly equivalent measurements.",
    "- **MHEALTH, MotionSense**: read directly from their saved output folders "
    "on disk (read-only, no retraining, no preprocessing, no re-evaluation).",
    "- **Important caveat on TinyML Tier labels**: UCI HAR/WISDM were profiled "
    "under an earlier tier-naming convention (Class 0/1/2 / Not TinyML-"
    "suitable) while PAMAP2/MHEALTH/MotionSense use the later Tier 1-4 "
    "convention. This module does not force-convert one naming scheme into "
    "the other. Only the **numeric** Deployment Friendliness Score is used "
    "for cross-dataset ranking/comparison; the tier label is shown per-dataset "
    "as originally reported or computed.\n",
    "### Ranking Methodology",
    "- Each of the 4 categories ranks the 7 classifiers 1 (best) to 7 (worst) "
    "per dataset on a single primary metric: Accuracy (Performance), "
    "Deployment Friendliness Score (TinyML), Overall Robustness Score "
    "(Robustness), and Significant Pairwise Wins with Accuracy as tie-break "
    "(Statistical).",
    "- Average rank per model per category = mean rank across datasets where "
    "that metric is available (NaN-safe). The `N_Datasets_*` columns in the "
    "ranking table make this coverage explicit — e.g. a value of 5 means all "
    "five datasets contributed to that model's average rank in that category, "
    "while a lower number means one or more datasets had no data for that "
    "model in that category.",
    "- Overall Combined Rank Score = mean of the 4 category-average ranks "
    "(equal weighting across categories, no additional weighting applied).\n",
    "## 1. Performance Comparison\n",
    performance_df.round(4).to_markdown(index=False),
    "\n## 2. TinyML Deployment Comparison\n",
    tinyml_df.round(3).to_markdown(index=False),
    "\n## 3. Robustness Comparison\n",
    "Note: Noise/Feature-Removal/Sensor-Dropout retention breakdowns are N/A "
    "for UCI HAR, WISDM, and PAMAP2 (only aggregate/worst-case retention was "
    "available for these three). PAMAP2's Overall_Robustness_Score reflects "
    "worst-case retention, not mean retention across conditions like the "
    "other four datasets — see Data Provenance note above.\n",
    robustness_df.round(2).to_markdown(index=False),
    "\n## 4. Statistical Validation Comparison\n",
    statistics_df.round(4).to_markdown(index=False),
    "\n### Cochran's Q Test Summary (all 5 datasets)\n",
    cochran_summary_df.round(6).to_markdown(index=False),
    "\n## 5. Average Classifier Rank per Category (across all 5 datasets)\n",
    overall_ranking.round(2).to_markdown(),
    "\n## 6. Overall Cross-Dataset Classifier Ranking\n",
    "Ranked by Overall Combined Rank Score (lower = better, averaged equally "
    "across Performance, TinyML, Robustness, and Statistical Validation):\n",
    overall_ranking[["Overall_Rank", "Overall_Combined_Rank_Score"]].to_markdown(),
    "\n## Top-Ranked Classifier Overall\n",
    f"**{overall_ranking.index[0]}** — Overall Combined Rank Score: "
    f"{overall_ranking.iloc[0]['Overall_Combined_Rank_Score']:.2f} "
    f"(Performance avg rank {overall_ranking.iloc[0]['Avg_Performance_Rank']:.2f}, "
    f"TinyML avg rank {overall_ranking.iloc[0]['Avg_TinyML_Rank']:.2f}, "
    f"Robustness avg rank {overall_ranking.iloc[0]['Avg_Robustness_Rank']:.2f}, "
    f"Statistical avg rank {overall_ranking.iloc[0]['Avg_Statistical_Rank']:.2f})",
]

with open(REPORTS_DIR / "cross_dataset_module5_report.md", "w", encoding=ENCODING) as f:
    f.write("\n".join(str(x) for x in report_lines))

print(f"[REPORT] Markdown report saved to {REPORTS_DIR / 'cross_dataset_module5_report.md'}")

# ============================================================================
# 12. FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 50)
print("MODULE 5 COMPLETE")
print("=" * 50)

[CONFIG] Module 5 — Cross-Dataset Comparative Analysis
[CONFIG] Writing all outputs to: C:\Users\Rohan Jaiswal\cross_dataset_comparison
[CONFIG] UCI_HAR, WISDM, and PAMAP2 use hardcoded values transcribed from console output/uploaded CSVs.
[CONFIG] MHEALTH and MotionSense are read from their saved output folders (read-only).

[HARDCODED] UCI_HAR significant pairwise wins (derived from transcribed McNemar table):
  {'Logistic_Regression': 4, 'Decision_Tree': 1, 'Random_Forest': 3, 'KNN': 2, 'Gaussian_NB': 0, 'Linear_SVM': 6, 'XGBoost': 4}
[HARDCODED] WISDM significant pairwise wins (derived from transcribed McNemar table):
  {'Logistic_Regression': 2, 'Decision_Tree': 3, 'Random_Forest': 4, 'KNN': 6, 'Gaussian_NB': 0, 'Linear_SVM': 1, 'XGBoost': 5}

[HARDCODED] PAMAP2 data loaded from console-log transcription and uploaded CSVs (Deployment Score/Tier computed from raw metrics; Robustness computed as worst-case retention; McNemar win-counts derived from uploaded holm_adjusted_results.csv